# Video Studio on Kaggle: any link → narrated video, as landscape and reel

Paste a link (a **news article in Nepali or English**, any web page, a **GitHub repo**, a **YouTube** video, a **PDF**) or describe an idea, and get back a narrated video with music, rendered both as **16:9 landscape** and **9:16 reel**. It uses the link's **own photos**, speaks the link's **language** (Nepali and Hindi included), and picks a look from the content: broadcast-style graphics for news, documentary, tech and more. Everything runs on the free Kaggle GPU; no paid APIs.

| Step | Tool | Runs on |
|---|---|---|
| Read the link: article text, date, photos and captions | [trafilatura](https://github.com/adbar/trafilatura) + headless Chromium fallback, GitHub API, yt-dlp, pymupdf | CPU |
| Pick and edit the photos; describe them so each scene gets the right one | Pillow, Gemma 3 vision | CPU / GPU 0 |
| Write the script in the video's language (checked and retried if it drifts) | Ollama `gemma3:12b` (140+ languages) | GPU 0 |
| Narration | Indic Parler-TTS, Svara-TTS (male and female Nepali / Hindi), Piper, Kokoro; optional online Microsoft voices | GPU 1 / CPU |
| Motion on real photos: 2.5D depth parallax, subject lift-off; map fly-in to the story's place | Depth Anything V2 Small, BiRefNet, MapLibre + OpenStreetMap | GPU 1 / CPU |
| Illustrations and AI video clips only for non-news topics; background music matched to the tone | SDXL, LTX-Video / Wan 2.2 (opt-in), MusicGen or ACE-Step | GPU 1 |
| Animate (broadcast lower-thirds, kinetic type, light sweeps, film grain, wipes, transition sounds) and render both formats in parallel | headless Chromium (WebGL) + ffmpeg | CPU |

Inspired by [nexu-io/html-video](https://github.com/nexu-io/html-video) and [HyperFrames](https://github.com/heygen-com/hyperframes) (Apache 2.0). A local model only fills in a script; ready-made templates do the design, and every frame is rendered exactly.

| Path | What |
|---|---|
| `/` | The studio: paste a link, pick 15/30/60/90 s and formats, watch progress, play and download both videos, edit the script |
| `/api/*` | JSON API |
| `/mcp` | MCP server for agents (tools: `make_video`, `get_job`, `get_storyboard`, `update_storyboard`, `add_asset`, `render`, `list_jobs`, `delete_job`, `list_options`) |

### Before you run
1. **Accelerator:** GPU T4 x2 (one GPU works, more slowly). **Internet:** on.
2. **A Cloudflare Tunnel** with a public hostname pointing to `http://localhost:7860`.
3. **Kaggle secrets** (Add-ons → Secrets, then tick them for this notebook):
   - `VIDEO_TUNNEL_TOKEN`: the tunnel token. Without it everything still runs, only without a public URL.
   - `VIDEO_UI_PASSWORD` (recommended): the password for the studio's sign-in page; agents send `Authorization: Bearer <password>`. Without it a random password is printed below.
   - `GITHUB_TOKEN` (optional): any GitHub token, to lift the API limit of 60 repository lookups an hour.
4. **Run All.** First run: about 12–15 minutes (installs + ~20 GB of models). Then a 60-second video takes roughly 5–10 minutes on T4 x2; 720p renders about twice as fast.

### Licences and responsibility
Gemma 3: Gemma Terms of Use. Indic Parler-TTS, Svara-TTS, Kokoro, Depth Anything V2 Small, ACE-Step, Wan 2.2: Apache 2.0. BiRefNet, Piper: MIT. LTX-Video: Lightricks' open-weights licence (check it before commercial use). Map data © OpenStreetMap contributors (ODbL), tiles by OpenFreeMap. SDXL: CreativeML OpenRAIL++. **MusicGen weights are CC-BY-NC 4.0 (non-commercial)**: for monetised videos set `MUSIC_MODEL = "ace-step"` (Apache 2.0) or `""`. The online voices (`ONLINE_VOICES = True`) send the narration text to Microsoft and are not open source. **Photos and text from a link belong to their publisher**: the video credits the site, but make sure you may reuse them (for example your own site, or with permission).

Videos are saved in `/kaggle/working/video_studio/jobs/` and disappear when the session ends: download what you want to keep. Logs: `/kaggle/working/logs/`.

In [ ]:
# 1. Settings
LLM_MODEL = "gemma3:12b"                                    # writes Nepali well and can look at photos; "gemma3:4b" is faster
IMAGE_MODEL = "stabilityai/stable-diffusion-xl-base-1.0"    # only for non-news topics without photos; "" = never draw
IMAGE_STEPS = 25
MUSIC_MODEL = "facebook/musicgen-small"                     # "ace-step" = Apache 2.0 music (commercial OK, slower, own virtualenv);
                                                            # "" = no music (MusicGen weights are non-commercial)
NARRATION = True
INDIC_VOICES = True                                         # Indic Parler-TTS for Nepali / Hindi (~4 GB, own virtualenv)
SVARA_VOICES = True                                         # Svara-TTS: male + female Nepali / Hindi voices (~7 GB)
ONLINE_VOICES = False                                       # Microsoft Edge neural voices (Nepali Hemkala / Sagar...): online,
                                                            # very natural, but text goes to Microsoft; not open source
DEPTH_PARALLAX = True                                       # 2.5D motion of real photos (Depth Anything V2 Small, ~100 MB)
SUBJECT_CUTOUT = True                                       # headline subject lifts off the photo (BiRefNet lite, ~200 MB)
LOCATION_MAPS = True                                        # map fly-in to the story's place (OpenStreetMap, needs internet)
I2V_MODEL = ""                                              # AI video clips for NON-news topics, picked per video as Motion "AI":
                                                            # "ltx" (LTX-Video, faster) or "wan" (Wan 2.2 5B, slow); "" = off
RENDER_QUALITY = "1080p"                                    # or "720p" (about twice as fast); the page can override it
PORT = 7860
WORK_DIR = "/kaggle/working/video_studio"
LOG_DIR = "/kaggle/working/logs"
PARLER_ENV = "/tmp/parler_env"                             # outside /kaggle/working (kept out of the notebook's output)
PIPER_DIR = "/kaggle/working/piper"
ACE_ENV = "/tmp/ace_env"
ACE_CKPT = "/tmp/ace_checkpoints"
APP_DIR = '/kaggle/working/video_app'
print("Settings saved.")

### 2. App files
These cells only write files into `APP_DIR`.

In [ ]:
# Shared web kit: password, MCP endpoint, proxy, process helpers
# Writes studio_http.py into APP_DIR. Safe to re-run.
import os
APP_DIR = '/kaggle/working/video_app'
os.makedirs(APP_DIR, exist_ok=True)
with open(os.path.join(APP_DIR, 'studio_http.py'), "w", encoding="utf-8") as f:
    f.write(r'''"""studio_http: the small web kit behind the newer agentic-notebooks apps.

build_notebook.py copies this file into each notebook's app folder, so every notebook stays
self-contained: nothing is imported from GitHub at run time. Standard library only.

An app built on it gets:
- routes with regex paths, JSON in and out, file (with byte ranges), zip and chunked responses;
- one password for everything: browsers get a login page and a signed session cookie (no browser pop-up);
  agents send ``Authorization: Bearer <password>``, ``X-Access-Token: <password>`` or HTTP Basic.
  ``GET /health`` stays open;
- an MCP server at ``POST /mcp`` (Streamable HTTP transport, plain JSON responses, no sessions),
  and the same tools as REST: ``GET /mcp/tools`` and ``POST /mcp/tools/<name>`` with the arguments as JSON;
- an optional reverse proxy (HTTP and WebSocket) for every path no route matches;
- IPv4 and IPv6 listeners on one port (cloudflared may dial [::1] for "localhost").

The notebook cells use the helpers at the bottom: kaggle_secret(), spawn() / stop_process() with pid
files (so re-running a cell, or the whole notebook after a kernel restart, never leaves a second copy
running), wait_http(), tail(), start_tunnel() and keep_alive(). Logs go to files, never to the notebook.
"""
import base64
import hashlib
import hmac
import html as _html
import http.client
import json
import logging
import mimetypes
import os
import re
import select
import signal
import socket
import subprocess
import sys
import threading
import time
import traceback
import urllib.error
import urllib.request
import zipfile
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from urllib.parse import parse_qs, quote, urlparse

MCP_VERSIONS = ("2025-06-18", "2025-03-26", "2024-11-05")
HOP = {"connection", "keep-alive", "proxy-authenticate", "proxy-authorization", "te", "trailer", "trailers",
       "transfer-encoding", "upgrade"}
SECRET_HEADERS = {"authorization", "x-access-token"}
SESSION_COOKIE = "studio_session"
SESSION_DAYS = 30
PID_DIR = "/kaggle/working/.pids" if os.path.isdir("/kaggle/working") else "/tmp/agentic-notebooks-pids"


# ====================================================================== responses
class HTTPError(Exception):
    def __init__(self, status, message):
        super().__init__(message)
        self.status = status


class Response:
    def __init__(self, body=b"", status=200, ctype="text/plain; charset=utf-8", headers=None):
        self.body = body.encode("utf-8") if isinstance(body, str) else body
        self.status, self.ctype, self.headers = status, ctype, dict(headers or {})


class FileResponse:
    """A file from disk. ``name`` makes it a download. Single byte ranges work (audio/video seeking)."""

    def __init__(self, path, name=None, ctype=None, cache="no-cache"):
        if not os.path.isfile(path):
            raise HTTPError(404, "file not found")
        self.path, self.name, self.cache = path, name, cache
        self.ctype = ctype or mimetypes.guess_type(path)[0] or "application/octet-stream"


class StreamResponse:
    """Chunked response from an iterable of bytes or str."""

    def __init__(self, chunks, ctype="text/plain; charset=utf-8", headers=None):
        self.chunks, self.ctype, self.headers = chunks, ctype, dict(headers or {})


class ZipResponse:
    """Streams ``folder`` (or a list of (path, arcname) pairs) as a zip download."""

    def __init__(self, files, name):
        if isinstance(files, str):
            root = os.path.dirname(os.path.abspath(files))
            pairs = []
            for dp, dn, fn in os.walk(files):
                dn.sort()
                for f in sorted(fn):
                    if not f.endswith((".tmp", ".part")):
                        pairs.append((os.path.join(dp, f), os.path.relpath(os.path.join(dp, f), root)))
            files = pairs
        self.files, self.name = files, name


class ToolContent(list):
    """Return this from an MCP tool to send raw content blocks, for example an image."""


def text_block(text):
    return {"type": "text", "text": text}


def image_block(data, mime="image/png"):
    return {"type": "image", "data": base64.b64encode(data).decode("ascii"), "mimeType": mime}


def file_logger(path, name="app"):
    """A logger that writes to ``path`` only (nothing reaches the notebook output)."""
    log = logging.getLogger(name)
    path = os.path.abspath(path)
    if not any(getattr(h, "baseFilename", None) == path for h in log.handlers):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        h = logging.FileHandler(path, encoding="utf-8")
        h.setFormatter(logging.Formatter("%(asctime)s %(levelname)s %(message)s", "%Y-%m-%d %H:%M:%S"))
        log.addHandler(h)
    log.setLevel(logging.INFO)
    log.propagate = False
    return log


def _disposition(name):
    ascii_name = re.sub(r'[^A-Za-z0-9_.() -]+', "_", name) or "download"
    return "attachment; filename=\"%s\"; filename*=UTF-8''%s" % (ascii_name, quote(name))


# ====================================================================== requests
class Request:
    def __init__(self, handler, params, max_body):
        self.handler = handler
        u = urlparse(handler.path)
        self.method = handler.command
        self.path = u.path
        self.query = {k: v[-1] for k, v in parse_qs(u.query, keep_blank_values=True).items()}
        self.params = params
        self.headers = handler.headers
        self.max_body = max_body
        self.consumed = False

    @property
    def length(self):
        try:
            return int(self.headers.get("Content-Length") or 0)
        except ValueError:
            raise HTTPError(400, "bad Content-Length")

    @property
    def chunked(self):
        return "chunked" in (self.headers.get("Transfer-Encoding") or "").lower()

    def has_body(self):
        return self.chunked or self.length > 0

    def iter_body(self, limit=None):
        """Yields the request body in pieces (Content-Length or chunked), enforcing ``limit`` bytes."""
        limit = self.max_body if limit is None else limit
        too_big = HTTPError(413, "request body too large (limit %d MB)" % (limit // 2 ** 20))
        rf = self.handler.rfile
        self.consumed = True
        if self.chunked:
            total = 0
            while True:
                line = rf.readline(1024)
                try:
                    size = int(line.split(b";")[0].strip() or b"0", 16)
                except ValueError:
                    raise HTTPError(400, "bad chunked body")
                if size == 0:
                    while rf.readline(1024) not in (b"\r\n", b"\n", b""):
                        pass
                    return
                total += size
                if total > limit:
                    raise too_big
                left = size
                while left:
                    data = rf.read(min(left, 1 << 20))
                    if not data:
                        raise HTTPError(400, "request body was cut off")
                    left -= len(data)
                    yield data
                rf.readline(8)
        else:
            left = self.length
            if left > limit:
                raise too_big
            while left:
                data = rf.read(min(left, 1 << 20))
                if not data:
                    raise HTTPError(400, "request body was cut off")
                left -= len(data)
                yield data

    def body(self):
        return b"".join(self.iter_body())

    def json(self):
        raw = self.body()
        if not raw.strip():
            return {}
        try:
            return json.loads(raw.decode("utf-8"))
        except ValueError:
            raise HTTPError(400, "body is not valid JSON")

    def save_body(self, path, limit):
        """Writes the raw body to ``path`` (atomically) and returns its size."""
        if not self.has_body():
            raise HTTPError(411, "send the file as the raw request body")
        tmp = path + ".part"
        size = 0
        try:
            with open(tmp, "wb") as f:
                for data in self.iter_body(limit):
                    f.write(data)
                    size += len(data)
        except BaseException:
            if os.path.exists(tmp):
                os.remove(tmp)
            raise
        os.replace(tmp, path)
        return size

    def arg(self, name, default=None, cast=str):
        v = self.query.get(name)
        if v in (None, ""):
            return default
        try:
            return cast(v)
        except (TypeError, ValueError):
            raise HTTPError(400, "bad value for %s" % name)


# ====================================================================== app
def _strip_session(cookie):
    """Never forward our session cookie to a proxied backend."""
    return "; ".join(p.strip() for p in (cookie or "").split(";")
                     if p.strip() and p.strip().partition("=")[0] != SESSION_COOKIE)


LOGIN_PAGE = """<!doctype html><html lang="en"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1"><title>Sign in · {{NAME}}</title>
<style>
:root{--bg:#f4f5f8;--card:#fff;--text:#14171c;--muted:#636b77;--line:#dfe2e8;--accent:#2f6fec;--bad:#c4372f}
@media (prefers-color-scheme:dark){:root{--bg:#0d0f13;--card:#161a20;--text:#e8eaee;--muted:#98a0ab;--line:#2a2f37;--accent:#5b8ff5;--bad:#ef6a60}}
*{box-sizing:border-box}html,body{height:100%;margin:0}
body{background:radial-gradient(1200px 600px at 10% -10%,color-mix(in srgb,var(--accent) 18%,transparent),transparent),var(--bg);
color:var(--text);font:15px/1.5 system-ui,-apple-system,"Segoe UI",Roboto,"Noto Sans","Noto Sans Devanagari",sans-serif;
display:flex;align-items:center;justify-content:center;padding:16px}
.card{width:100%;max-width:380px;background:var(--card);border:1px solid var(--line);border-radius:18px;padding:30px 28px;
box-shadow:0 20px 60px rgba(0,0,0,.12)}
.logo{width:44px;height:44px;border-radius:12px;background:var(--accent);display:flex;align-items:center;justify-content:center;
color:#fff;font-weight:800;font-size:20px;margin-bottom:18px}
h1{font-size:20px;margin:0 0 4px}p{color:var(--muted);margin:0 0 22px;font-size:14px}
label{display:block;font-size:13px;color:var(--muted);margin-bottom:6px}
.field{position:relative}
input{width:100%;font:inherit;color:var(--text);background:transparent;border:1px solid var(--line);border-radius:10px;padding:11px 44px 11px 12px}
input:focus{outline:2px solid color-mix(in srgb,var(--accent) 45%,transparent);border-color:var(--accent)}
.eye{position:absolute;right:6px;top:50%;transform:translateY(-50%);border:0;background:none;color:var(--muted);cursor:pointer;padding:6px;font-size:13px}
button.go{width:100%;margin-top:16px;border:0;border-radius:10px;padding:12px;background:var(--accent);color:#fff;font:600 15px system-ui;cursor:pointer}
button.go:disabled{opacity:.6}.err{color:var(--bad);font-size:13px;min-height:20px;margin-top:10px}
.hint{font-size:12px;color:var(--muted);margin-top:18px;border-top:1px solid var(--line);padding-top:14px}
</style></head><body>
<form class="card" id="f" method="post" action="/login">
<div class="logo">&#9656;</div><h1>{{NAME}}</h1><p>Sign in with the password from your notebook.</p>
<label for="pw">Password</label><div class="field"><input id="pw" name="password" type="password" autocomplete="current-password" autofocus required>
<button class="eye" type="button" id="eye">Show</button></div>
<button class="go" id="go">Sign in</button><div class="err" id="err"></div>
<div class="hint">It is the <b>*_UI_PASSWORD</b> Kaggle secret, or the password the notebook printed.</div>
</form>
<script>
const f=document.getElementById('f'),pw=document.getElementById('pw'),err=document.getElementById('err'),go=document.getElementById('go');
document.getElementById('eye').onclick=e=>{pw.type=pw.type==='password'?'text':'password';e.target.textContent=pw.type==='password'?'Show':'Hide';};
f.onsubmit=async e=>{e.preventDefault();go.disabled=true;err.textContent='';
 try{const r=await fetch('/login',{method:'POST',headers:{'Content-Type':'application/json'},body:JSON.stringify({password:pw.value})});
  if(r.ok){location.reload();return;} const d=await r.json().catch(()=>({}));err.textContent=d.error||('Sign-in failed ('+r.status+')');}
 catch(x){err.textContent='Network error: '+x;} go.disabled=false;pw.select();};
</script></body></html>"""


def _rpc_error(mid, code, message):
    return {"jsonrpc": "2.0", "id": mid, "error": {"code": code, "message": message}}


class App:
    def __init__(self, name, password=None, version="1.0", instructions="", proxy=None, log=None,
                 max_body=64 << 20):
        self.name, self.version, self.instructions = name, version, instructions
        self.password = password or None
        self.proxy = proxy.rstrip("/") if proxy else None
        self.log = log or logging.getLogger(name)
        self.max_body = max_body
        self.routes, self.tools, self.servers = [], {}, []
        self._fails = {}

        self.route("GET", "/health", auth=False)(lambda req: {"ok": True})
        self.route("POST", "/login", auth=False)(self._login)
        self.route("GET", "/logout", auth=False)(self._logout)
        self.route("POST", "/logout", auth=False)(self._logout)
        self.route("POST", "/mcp")(self._mcp_http)
        self.route("GET", "/mcp")(lambda req: Response(
            "This MCP server answers POST requests (Streamable HTTP transport, JSON responses).", 405,
            headers={"Allow": "POST"}))
        self.route("DELETE", "/mcp")(lambda req: Response(b"", 405, headers={"Allow": "POST"}))
        self.route("GET", "/mcp/tools")(lambda req: {"tools": self.tool_list()})
        self.route("POST", r"/mcp/tools/(?P<name>[A-Za-z0-9_\-]+)")(self._tool_http)

    # ------------------------------------------------------------------ registration
    def route(self, method, pattern, auth=True):
        rx = re.compile(pattern)

        def deco(fn):
            self.routes.append((method.upper(), rx, fn, auth))
            return fn
        return deco

    def page(self, path, file):
        self.route("GET", re.escape(path))(lambda req: FileResponse(file, ctype="text/html; charset=utf-8"))

    def static(self, prefix, folder):
        root = os.path.realpath(folder)

        def serve(req):
            full = os.path.realpath(os.path.join(root, req.params["rel"]))
            if not full.startswith(root + os.sep):
                raise HTTPError(404, "not found")
            return FileResponse(full)
        self.route("GET", re.escape(prefix) + r"(?P<rel>.+)")(serve)

    def tool(self, name, description, properties=None, required=()):
        """Registers an MCP tool. ``properties`` is a JSON Schema properties object."""
        def deco(fn):
            self.tools[name] = {"fn": fn, "schema": {
                "name": name, "description": description,
                "inputSchema": {"type": "object", "properties": properties or {}, "required": list(required)}}}
            return fn
        return deco

    def tool_list(self):
        return [t["schema"] for t in self.tools.values()]

    def call_tool(self, name, args):
        """Runs a tool. Returns (content_blocks, is_error)."""
        t = self.tools[name]
        if not isinstance(args, dict):
            return [text_block("arguments must be a JSON object")], True
        try:
            out = t["fn"](**args)
        except TypeError as e:
            if "argument" in str(e):
                return [text_block("bad arguments for %s: %s" % (name, e))], True
            self.log.error("tool %s failed\n%s", name, traceback.format_exc())
            return [text_block("TypeError: %s" % e)], True
        except HTTPError as e:
            return [text_block(str(e))], True
        except (ValueError, KeyError) as e:
            return [text_block(str(e).strip("'\""))], True
        except Exception as e:
            self.log.error("tool %s failed\n%s", name, traceback.format_exc())
            return [text_block("%s: %s" % (type(e).__name__, e))], True
        if isinstance(out, ToolContent):
            return list(out), False
        if isinstance(out, str):
            return [text_block(out)], False
        return [text_block(json.dumps(out, indent=1, ensure_ascii=False, default=str))], False

    # ------------------------------------------------------------------ MCP
    def _mcp_http(self, req):
        try:
            msg = json.loads(req.body().decode("utf-8") or "null")
        except ValueError:
            return _rpc_error(None, -32700, "parse error")
        if isinstance(msg, list):
            out = [r for r in map(self._rpc, msg) if r is not None]
            return out if out else Response(b"", 202)
        r = self._rpc(msg)
        return r if r is not None else Response(b"", 202)

    def _rpc(self, m):
        if not isinstance(m, dict) or m.get("jsonrpc") != "2.0":
            return _rpc_error(m.get("id") if isinstance(m, dict) else None, -32600, "invalid request")
        if "method" not in m or "id" not in m:
            return None            # a notification, or a response to a request we never send
        mid, method, params = m["id"], m["method"], m.get("params") or {}
        try:
            if method == "initialize":
                v = params.get("protocolVersion")
                result = {"protocolVersion": v if v in MCP_VERSIONS else MCP_VERSIONS[0],
                          "capabilities": {"tools": {"listChanged": False}},
                          "serverInfo": {"name": self.name, "version": self.version}}
                if self.instructions:
                    result["instructions"] = self.instructions
            elif method == "ping":
                result = {}
            elif method == "tools/list":
                result = {"tools": self.tool_list()}
            elif method == "tools/call":
                name = params.get("name")
                if name not in self.tools:
                    return _rpc_error(mid, -32602, "unknown tool: %s" % name)
                content, err = self.call_tool(name, params.get("arguments") or {})
                result = {"content": content, "isError": err}
            elif method == "resources/list":
                result = {"resources": []}
            elif method == "resources/templates/list":
                result = {"resourceTemplates": []}
            elif method == "prompts/list":
                result = {"prompts": []}
            else:
                return _rpc_error(mid, -32601, "method not found: %s" % method)
        except Exception as e:
            self.log.error("mcp %s failed\n%s", method, traceback.format_exc())
            return _rpc_error(mid, -32603, str(e))
        return {"jsonrpc": "2.0", "id": mid, "result": result}

    def _tool_http(self, req):
        name = req.params["name"]
        if name not in self.tools:
            raise HTTPError(404, "unknown tool: %s" % name)
        content, err = self.call_tool(name, req.json())
        if len(content) == 1 and content[0]["type"] == "text":
            try:
                data = json.loads(content[0]["text"])
            except ValueError:
                data = content[0]["text"]
            body = {"error": data} if err else data
        else:
            body = {"content": content, "isError": err}
        return Response(json.dumps(body, ensure_ascii=False, default=str), 400 if err else 200,
                        "application/json; charset=utf-8")

    # ------------------------------------------------------------------ auth
    def _key(self):
        return hashlib.sha256(("studio-session:" + self.password).encode("utf-8")).digest()

    def make_session(self, days=SESSION_DAYS):
        exp = str(int(time.time() + days * 86400))
        return "%s.%s" % (exp, hmac.new(self._key(), exp.encode(), hashlib.sha256).hexdigest()[:40])

    def _session_ok(self, headers):
        for part in (headers.get("Cookie") or "").split(";"):
            k, _, v = part.strip().partition("=")
            if k == SESSION_COOKIE:
                exp, _, sig = v.partition(".")
                if exp.isdigit() and int(exp) > time.time():
                    want = hmac.new(self._key(), exp.encode(), hashlib.sha256).hexdigest()[:40]
                    if hmac.compare_digest(sig.encode(), want.encode()):
                        return True
        return False

    def _cookie(self, headers, value, max_age):
        secure = (headers.get("X-Forwarded-Proto") or "").lower() == "https" or '"https"' in (headers.get("Cf-Visitor") or "")
        return "%s=%s; Path=/; Max-Age=%d; HttpOnly; SameSite=Lax%s" % (
            SESSION_COOKIE, value, max_age, "; Secure" if secure else "")

    def _login(self, req):
        if not self.password:
            return {"ok": True}
        ip = req.headers.get("Cf-Connecting-Ip") or req.handler.client_address[0]
        now = time.time()
        fails = [t for t in self._fails.get(ip, []) if now - t < 300]
        if len(fails) >= 8:
            raise HTTPError(429, "too many attempts; wait a few minutes")
        if "json" in (req.headers.get("Content-Type") or ""):
            given = str(req.json().get("password") or "")
        else:
            given = parse_qs(req.body().decode("utf-8", "replace")).get("password", [""])[0]
        if not hmac.compare_digest(given.encode("utf-8"), self.password.encode("utf-8")):
            fails.append(now)
            self._fails[ip] = fails
            time.sleep(0.5)
            raise HTTPError(401, "wrong password")
        self._fails.pop(ip, None)
        return Response(json.dumps({"ok": True}), 200, "application/json; charset=utf-8",
                        {"Set-Cookie": self._cookie(req.headers, self.make_session(), SESSION_DAYS * 86400)})

    def _logout(self, req):
        return Response(b"", 303, "text/plain", {"Location": "/", "Set-Cookie": self._cookie(req.headers, "", 0)})

    def authorized(self, headers):
        pw = self.password
        if not pw:
            return True
        if self._session_ok(headers):
            return True

        def ok(given):
            return bool(given) and hmac.compare_digest(given.encode("utf-8"), pw.encode("utf-8"))
        if ok(headers.get("X-Access-Token", "")):
            return True
        h = headers.get("Authorization", "")
        if h[:6].lower() == "basic ":
            try:
                given = base64.b64decode(h[6:].strip()).decode("utf-8").partition(":")[2]
            except Exception:
                given = ""
            return ok(given)
        if h[:7].lower() == "bearer ":
            return ok(h[7:].strip())
        return False

    def _deny(self, h):
        """Pages get the login form; everything else a JSON 401 (no browser pop-up)."""
        if h.command in ("GET", "HEAD") and "text/html" in (h.headers.get("Accept") or ""):
            return self._respond(h, Response(LOGIN_PAGE.replace("{{NAME}}", _html.escape(self.name)), 401,
                                             "text/html; charset=utf-8"))
        self._respond(h, Response(json.dumps({"error": "login required: sign in on the page, or send "
                                              "'Authorization: Bearer <password>'"}), 401,
                                  "application/json; charset=utf-8"))

    # ------------------------------------------------------------------ dispatch
    def dispatch(self, h):
        h._sent = False
        req = None
        try:
            path = urlparse(h.path).path
            method = "GET" if h.command == "HEAD" else h.command
            found, path_known = None, False
            for m, rx, fn, auth in self.routes:
                mt = rx.fullmatch(path)
                if mt:
                    path_known = True
                    if m == method:
                        found = (fn, auth, mt.groupdict())
                        break
            if found is None:
                if path_known:
                    raise HTTPError(405, "method not allowed")
                if not self.proxy:
                    raise HTTPError(404, "not found")
                if not self.authorized(h.headers):
                    return self._deny(h)
                return self._proxy(h)
            fn, auth, params = found
            if auth and not self.authorized(h.headers):
                return self._deny(h)
            req = Request(h, params, self.max_body)
            self._respond(h, fn(req))
        except (BrokenPipeError, ConnectionResetError):
            h.close_connection = True
        except HTTPError as e:
            self._fail(h, e.status, str(e))
        except (ValueError, KeyError) as e:
            msg = ("missing field %s" % e) if isinstance(e, KeyError) else str(e)
            self._fail(h, 400, msg)
        except Exception as e:
            self.log.error("%s %s failed\n%s", h.command, h.path, traceback.format_exc())
            self._fail(h, 500, "%s: %s" % (type(e).__name__, e))
        finally:
            body_left = (req is None or not req.consumed) and (
                "chunked" in (h.headers.get("Transfer-Encoding") or "").lower()
                or (h.headers.get("Content-Length") or "0").strip() not in ("", "0"))
            if body_left:
                h.close_connection = True   # an unread body would corrupt the next request on this connection

    def _fail(self, h, status, message):
        if h._sent:
            h.close_connection = True
            return
        try:
            self._respond(h, Response(json.dumps({"error": message}), status, "application/json; charset=utf-8"))
        except OSError:
            h.close_connection = True

    def _start(self, h, status, headers):
        h.send_response(status)
        h._sent = True
        for k, v in headers:
            h.send_header(k, v)
        h.end_headers()

    def _respond(self, h, out):
        head = h.command == "HEAD"
        if out is None:
            out = Response(b"", 204)
        elif isinstance(out, (dict, list)):
            out = Response(json.dumps(out, ensure_ascii=False, default=str), 200, "application/json; charset=utf-8")
        elif isinstance(out, str):
            out = Response(out)
        elif isinstance(out, bytes):
            out = Response(out, 200, "application/octet-stream")

        if isinstance(out, Response):
            hdrs = {"Content-Type": out.ctype, "Cache-Control": "no-store"}
            hdrs.update(out.headers)
            if out.status not in (204, 304):
                hdrs["Content-Length"] = str(len(out.body))
            self._start(h, out.status, hdrs.items())
            if not head and out.status not in (204, 304) and out.body:
                h.wfile.write(out.body)
        elif isinstance(out, FileResponse):
            self._send_file(h, out, head)
        elif isinstance(out, StreamResponse):
            self._start(h, 200, list({"Content-Type": out.ctype, "Cache-Control": "no-cache",
                                      "X-Accel-Buffering": "no", **out.headers}.items())
                        + [("Transfer-Encoding", "chunked")])
            if head:
                h.wfile.write(b"0\r\n\r\n")
                return
            for chunk in out.chunks:
                if isinstance(chunk, str):
                    chunk = chunk.encode("utf-8")
                if chunk:
                    h.wfile.write(b"%x\r\n" % len(chunk) + chunk + b"\r\n")
                    h.wfile.flush()
            h.wfile.write(b"0\r\n\r\n")
        elif isinstance(out, ZipResponse):
            self._send_zip(h, out, head)
        else:
            raise TypeError("handler returned %r" % type(out))

    def _send_file(self, h, f, head):
        size = os.path.getsize(f.path)
        start, end, status = 0, size - 1, 200
        m = re.fullmatch(r"bytes=(\d*)-(\d*)", (h.headers.get("Range") or "").strip())
        if m and size and (m.group(1) or m.group(2)):
            a, b = m.groups()
            if a == "":
                start = max(0, size - int(b))
            else:
                start, end = int(a), min(int(b), size - 1) if b else size - 1
            if start > end or start >= size:
                return self._start(h, 416, [("Content-Range", "bytes */%d" % size), ("Content-Length", "0")])
            status = 206
        hdrs = [("Content-Type", f.ctype), ("Content-Length", str(end - start + 1)), ("Accept-Ranges", "bytes"),
                ("Cache-Control", f.cache)]
        if status == 206:
            hdrs.append(("Content-Range", "bytes %d-%d/%d" % (start, end, size)))
        if f.name:
            hdrs.append(("Content-Disposition", _disposition(f.name)))
        self._start(h, status, hdrs)
        if head:
            return
        with open(f.path, "rb") as fh:
            fh.seek(start)
            left = end - start + 1
            while left > 0:
                data = fh.read(min(left, 1 << 20))
                if not data:
                    break
                h.wfile.write(data)
                left -= len(data)

    def _send_zip(self, h, z, head):
        self._start(h, 200, [("Content-Type", "application/zip"), ("Content-Disposition", _disposition(z.name)),
                             ("Cache-Control", "no-store"), ("Transfer-Encoding", "chunked")])
        w = h.wfile

        class Chunks:
            def write(self, data):
                if data:
                    w.write(b"%x\r\n" % len(data) + bytes(data) + b"\r\n")
                return len(data)

            def flush(self):
                w.flush()

        if not head:
            with zipfile.ZipFile(Chunks(), "w", zipfile.ZIP_DEFLATED, allowZip64=True) as zf:
                for path, arc in z.files:
                    if isinstance(path, bytes):
                        zf.writestr(arc, path)
                    elif os.path.isfile(path):
                        zf.write(path, arc, compress_type=zipfile.ZIP_STORED if path.endswith(
                            (".safetensors", ".gguf", ".bin", ".npy", ".png", ".jpg", ".mp4", ".zip", ".wav"))
                            else zipfile.ZIP_DEFLATED)
        w.write(b"0\r\n\r\n")
        w.flush()

    # ------------------------------------------------------------------ reverse proxy
    def _proxy(self, h):
        pu = urlparse(self.proxy)
        host, port = pu.hostname, pu.port or 80
        if ("upgrade" in (h.headers.get("Connection") or "").lower()
                and (h.headers.get("Upgrade") or "").lower() == "websocket"):
            return self._proxy_ws(h, host, port)
        req = Request(h, {}, self.max_body)
        headers = [(k, _strip_session(v) if k.lower() == "cookie" else v) for k, v in h.headers.items()
                   if k.lower() not in HOP and k.lower() not in SECRET_HEADERS and k.lower() != "content-length"]
        headers = [(k, v) for k, v in headers if v]
        headers.append(("X-Forwarded-For", h.client_address[0]))
        body = req.body() if req.has_body() else None
        if body is not None:
            headers.append(("Content-Length", str(len(body))))
        conn = http.client.HTTPConnection(host, port, timeout=900)
        try:
            # the original Host header is kept: some backends (ComfyUI) compare it with Origin
            conn.putrequest(h.command, h.path, skip_host="host" in {k.lower() for k, v in headers},
                            skip_accept_encoding=True)
            for k, v in headers:
                conn.putheader(k, v)
            conn.endheaders(body)
            r = conn.getresponse()
        except OSError as e:
            conn.close()
            raise HTTPError(502, "backend not reachable (%s)" % type(e).__name__)
        try:
            hdrs = [(k, v) for k, v in r.getheaders() if k.lower() not in HOP and k.lower() != "content-length"]
            cl = r.getheader("Content-Length")
            chunked = "chunked" in (r.getheader("Transfer-Encoding") or "").lower()
            if h.command == "HEAD" or r.status in (204, 304) or r.status < 200:
                return self._start(h, r.status, hdrs + ([("Content-Length", cl)] if cl else []))
            if cl is not None and not chunked:
                self._start(h, r.status, hdrs + [("Content-Length", cl)])
                while True:
                    data = r.read(1 << 16)
                    if not data:
                        break
                    h.wfile.write(data)
            else:
                self._start(h, r.status, hdrs + [("Transfer-Encoding", "chunked")])
                while True:
                    data = r.read1(1 << 16)
                    if not data:
                        break
                    h.wfile.write(b"%x\r\n" % len(data) + data + b"\r\n")
                    h.wfile.flush()
                h.wfile.write(b"0\r\n\r\n")
        finally:
            conn.close()

    def _proxy_ws(self, h, host, port):
        try:
            up = socket.create_connection((host, port), timeout=10)
        except OSError as e:
            raise HTTPError(502, "backend not reachable (%s)" % type(e).__name__)
        lines = ["%s %s HTTP/1.1" % (h.command, h.path)]
        lines += ["%s: %s" % (k, _strip_session(v) if k.lower() == "cookie" else v) for k, v in h.headers.items()
                  if k.lower() not in SECRET_HEADERS and (k.lower() != "cookie" or _strip_session(v))]
        h._sent = True
        h.close_connection = True
        client = h.connection
        try:
            up.sendall(("\r\n".join(lines) + "\r\n\r\n").encode("latin-1"))
            up.settimeout(None)
            client.settimeout(None)
            while True:
                ready, _, broken = select.select([client, up], [], [client, up], 600)
                if broken:
                    break
                for s in ready:
                    data = s.recv(1 << 16)
                    if not data:
                        return
                    (up if s is client else client).sendall(data)
        except OSError:
            pass
        finally:
            up.close()

    # ------------------------------------------------------------------ serving
    def handler_class(self):
        app = self

        class Handler(BaseHTTPRequestHandler):
            protocol_version = "HTTP/1.1"
            server_version = re.sub(r"\W+", "", app.name) or "studio"

            def log_message(self, fmt, *args):
                pass

            def do_GET(self):
                app.dispatch(self)

            do_POST = do_PUT = do_PATCH = do_DELETE = do_HEAD = do_OPTIONS = do_GET
        return Handler

    def start(self, port):
        """Listens on 0.0.0.0 and [::] in background threads. Calling it again restarts cleanly."""
        self.stop()
        handler = self.handler_class()
        for cls, host in ((_V4Server, "0.0.0.0"), (_V6Server, "::")):
            try:
                srv = cls((host, port), handler)
            except OSError as e:
                self.log.warning("could not listen on [%s]:%d: %s", host, port, e)
                continue
            threading.Thread(target=srv.serve_forever, daemon=True).start()
            self.servers.append(srv)
        if not self.servers:
            raise RuntimeError("nothing is listening on port %d" % port)
        self.log.info("%s listening on port %d", self.name, port)

    def stop(self):
        while self.servers:
            srv = self.servers.pop()
            try:
                srv.shutdown()
                srv.server_close()
            except Exception:
                pass

    def serve_forever(self, port):
        """For app processes started with spawn(): serve until SIGTERM."""
        done = threading.Event()
        signal.signal(signal.SIGTERM, lambda *a: done.set())
        signal.signal(signal.SIGINT, lambda *a: done.set())
        self.start(port)
        done.wait()
        self.log.info("stopping")
        self.stop()


class _V4Server(ThreadingHTTPServer):
    allow_reuse_address = True
    daemon_threads = True
    request_queue_size = 64


class _V6Server(_V4Server):
    address_family = socket.AF_INET6

    def server_bind(self):
        self.socket.setsockopt(socket.IPPROTO_IPV6, socket.IPV6_V6ONLY, 1)
        super().server_bind()


# ====================================================================== JSON files and jobs
def read_json(path, default=None):
    try:
        with open(path, encoding="utf-8") as f:
            return json.load(f)
    except Exception:
        return default


def write_json(path, data):
    """Atomic write, so readers in other threads or processes never see half a file."""
    tmp = "%s.%d.%d.tmp" % (path, os.getpid(), threading.get_ident())
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=1, ensure_ascii=False, default=str)
    os.replace(tmp, path)


def safe_name(s, n=60):
    return re.sub(r"[^A-Za-z0-9_.-]+", "-", str(s or "")).strip("-.")[:n]


def gpu_info():
    try:
        out = subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.used,memory.total,utilization.gpu",
                              "--format=csv,noheader,nounits"], capture_output=True, text=True, timeout=5).stdout
        gpus = []
        for line in out.strip().splitlines():
            i, n, u, t, g = [x.strip() for x in line.split(",")]
            gpus.append({"index": int(i), "name": n, "mem_used_mb": int(u), "mem_total_mb": int(t), "util": int(g)})
        return gpus
    except Exception:
        return []


# ====================================================================== notebook helpers
def kaggle_secret(name):
    """Value of a Kaggle secret, or None when it is missing or not attached to this notebook.
    Outside Kaggle (local runs, Docker) it reads the environment variable of the same name."""
    try:
        from kaggle_secrets import UserSecretsClient
    except ImportError:
        return os.environ.get(name) or None
    try:
        return UserSecretsClient().get_secret(name) or None
    except Exception:
        return None


def ui_password(secret_name):
    pw = kaggle_secret(secret_name)
    if pw:
        print("UI password: from the %s secret. Browser login: any username + that password." % secret_name)
        return pw
    import secrets
    pw = secrets.token_urlsafe(9)
    print("No %s secret, so this session's password is: %s" % (secret_name, pw))
    print("Browser login: any username + that password. Add the secret to keep a fixed password.")
    return pw


def _pidfile(name):
    return os.path.join(PID_DIR, safe_name(name) + ".json")


def _alive(pid, exe=None):
    try:
        with open("/proc/%d/stat" % pid) as f:
            if f.read().rsplit(")", 1)[1].split()[0] == "Z":
                return False
        if exe:
            with open("/proc/%d/cmdline" % pid, "rb") as f:
                return exe.encode() in f.read()
        return True
    except (OSError, IndexError):
        return False


def process_alive(name):
    info = read_json(_pidfile(name))
    return bool(info) and _alive(info["pid"], info.get("exe"))


def stop_process(name, timeout=20):
    """Stops a process spawn() started under ``name`` (and its children). Returns True if one was running."""
    info = read_json(_pidfile(name))
    if not info:
        return False
    pid, exe = info["pid"], info.get("exe")
    running = _alive(pid, exe)
    if running:
        for sig, wait in ((signal.SIGTERM, timeout), (signal.SIGKILL, 5)):
            try:
                os.killpg(pid, sig)
            except OSError:
                break
            t = time.time()
            while time.time() - t < wait and _alive(pid, exe):
                time.sleep(0.2)
            if not _alive(pid, exe):
                break
    try:
        os.remove(_pidfile(name))
    except OSError:
        pass
    return running


def spawn(name, cmd, log_path, env=None, cwd=None):
    """Starts ``cmd`` in the background with its output appended to ``log_path``.
    A process started earlier under the same name is stopped first, so re-running a cell is safe."""
    stop_process(name)
    os.makedirs(PID_DIR, exist_ok=True)
    os.makedirs(os.path.dirname(os.path.abspath(log_path)), exist_ok=True)
    with open(log_path, "ab") as log:
        log.write(("\n===== %s: starting %s =====\n" % (time.strftime("%Y-%m-%d %H:%M:%S"), name)).encode())
        log.flush()
        proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL,
                                env=env, cwd=cwd, start_new_session=True)
    write_json(_pidfile(name), {"pid": proc.pid, "exe": os.path.basename(cmd[0]), "log": log_path,
                                "started": time.time()})
    return proc


def wait_http(url, timeout=120, name=None, headers=None):
    """Waits until ``url`` answers (any HTTP status). Returns False on timeout or if process ``name`` died."""
    t = time.time()
    while time.time() - t < timeout:
        try:
            urllib.request.urlopen(urllib.request.Request(url, headers=headers or {}), timeout=3)
            return True
        except urllib.error.HTTPError:
            return True
        except Exception:
            pass
        if name and not process_alive(name):
            return False
        time.sleep(1)
    return False


def tail(path, n=30):
    try:
        with open(path, "rb") as f:
            f.seek(0, 2)
            f.seek(max(0, f.tell() - 64 * 1024))
            return "\n".join(f.read().decode("utf-8", "replace").splitlines()[-n:])
    except OSError:
        return "(no log yet at %s)" % path


def ensure_cloudflared(path="/kaggle/working/cloudflared"):
    if not os.path.exists(path):
        url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
        urllib.request.urlretrieve(url, path + ".part")
        os.replace(path + ".part", path)
        os.chmod(path, 0o755)
    return path


def start_tunnel(secret_name, port=7860, log_path="/kaggle/working/logs/cloudflared.log",
                 binary="/kaggle/working/cloudflared"):
    """Starts cloudflared with the token from ``secret_name`` (passed as TUNNEL_TOKEN, never printed).
    A missing secret is not an error: the app keeps running inside the session, without a public URL."""
    stop_process("cloudflared")
    token = kaggle_secret(secret_name)
    if not token:
        print("No public URL: there is no Kaggle secret named %s attached to this notebook.\n"
              "The app is running inside this session anyway. To publish it:\n"
              "  1. Cloudflare Zero Trust -> Networks -> Tunnels: create a tunnel and add a public hostname\n"
              "     pointing to http://localhost:%d\n"
              "  2. Kaggle: Add-ons -> Secrets -> add %s with the tunnel token, and tick it for this notebook\n"
              "  3. Re-run this cell." % (secret_name, port, secret_name))
        return None
    ensure_cloudflared(binary)
    offset = os.path.getsize(log_path) if os.path.exists(log_path) else 0
    proc = spawn("cloudflared", [binary, "tunnel", "--no-autoupdate", "run"], log_path,
                 env=dict(os.environ, TUNNEL_TOKEN=token))

    def new_log():
        with open(log_path, "rb") as f:
            f.seek(offset)
            return f.read().decode("utf-8", "replace")
    for _ in range(45):
        time.sleep(1)
        if "Registered tunnel connection" in new_log():
            print("Tunnel connected. Open the public hostname you routed to http://localhost:%d" % port)
            return proc
        if proc.poll() is not None:
            print("cloudflared stopped (exit code %s). Last log lines:" % proc.returncode)
            print("\n".join(new_log().splitlines()[-15:]))
            print("Check that %s holds the token of a tunnel that still exists." % secret_name)
            return None
    print("Tunnel not registered yet; it keeps trying in the background. Last log lines:")
    print("\n".join(new_log().splitlines()[-15:]))
    return proc


def keep_alive(port, names, extra=None, interval=60):
    """The last cell: prints one status line a minute so the Kaggle session stays active.
    Interrupting it only stops the printing; the app keeps running."""
    try:
        while True:
            try:
                urllib.request.urlopen("http://127.0.0.1:%d/health" % port, timeout=5)
                parts = ["ui=up"]
            except Exception:
                parts = ["ui=DOWN"]
            for n in names:
                if n == "cloudflared" and not os.path.exists(_pidfile(n)):
                    parts.append("tunnel=off")
                else:
                    parts.append("%s=%s" % (n, "up" if process_alive(n) else "DOWN"))
            gpus = gpu_info()
            if gpus:
                parts.append("GPU " + ", ".join("%d: %.1f/%.0f GB %d%%" % (g["index"], g["mem_used_mb"] / 1024,
                                                                           g["mem_total_mb"] / 1024, g["util"])
                                                for g in gpus))
            if extra:
                try:
                    s = extra()
                    if s:
                        parts.append(s)
                except Exception as e:
                    parts.append("status error: %s" % e)
            print(time.strftime("%H:%M:%S"), " ".join(parts), flush=True)
            time.sleep(interval)
    except KeyboardInterrupt:
        print("Stopped watching. Everything keeps running in the background; re-run this cell to watch again.")


def api_get(port, path, password, timeout=10):
    """GET a JSON endpoint of a local app (used by keep-alive status lines)."""
    req = urllib.request.Request("http://127.0.0.1:%d%s" % (port, path),
                                 headers={"Authorization": "Bearer %s" % password} if password else {})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.loads(r.read().decode("utf-8"))
''')
print("wrote", os.path.join(APP_DIR, 'studio_http.py'))

In [ ]:
# Shared page styles
# Writes kit.css into APP_DIR. Safe to re-run.
import os
APP_DIR = '/kaggle/working/video_app'
os.makedirs(APP_DIR, exist_ok=True)
with open(os.path.join(APP_DIR, 'kit.css'), "w", encoding="utf-8") as f:
    f.write(r'''/* Shared look for the agentic-notebooks app pages (served at /static/kit.css). */
:root{--bg:#f5f6f8;--panel:#fff;--panel2:#f0f2f5;--text:#1b1f24;--muted:#626b76;--line:#dde1e6;
  --accent:#2f6fec;--accent-ink:#fff;--ok:#1a7f4b;--warn:#a96500;--bad:#c4372f;--radius:10px;
  --mono:ui-monospace,SFMono-Regular,Menlo,Consolas,monospace}
@media (prefers-color-scheme:dark){:root{--bg:#0f1114;--panel:#171a1f;--panel2:#1e2228;--text:#e7e9ec;
  --muted:#9aa3ad;--line:#2b3038;--accent:#5b8ff5;--ok:#3fbf7f;--warn:#e0a03a;--bad:#ef6a60}}
*{box-sizing:border-box}
html,body{margin:0;background:var(--bg);color:var(--text);font:14px/1.5 system-ui,-apple-system,"Segoe UI",Roboto,sans-serif}
a{color:var(--accent)}
header.top{display:flex;align-items:center;gap:16px;flex-wrap:wrap;padding:12px 20px;border-bottom:1px solid var(--line);background:var(--panel)}
header.top h1{font-size:17px;margin:0;font-weight:650}
header.top .sub{color:var(--muted);font-size:13px}
header.top .right{margin-left:auto;display:flex;gap:10px;align-items:center;flex-wrap:wrap}
nav.tabs{display:flex;gap:4px;padding:0 20px;border-bottom:1px solid var(--line);background:var(--panel);overflow-x:auto}
nav.tabs button{border:0;background:none;color:var(--muted);padding:10px 12px;font:inherit;cursor:pointer;border-bottom:2px solid transparent;white-space:nowrap}
nav.tabs button.on{color:var(--text);border-bottom-color:var(--accent);font-weight:600}
main{max-width:1180px;margin:0 auto;padding:20px}
[data-pane]{display:none}[data-pane].on{display:block}
.card{background:var(--panel);border:1px solid var(--line);border-radius:var(--radius);padding:16px;margin-bottom:16px}
.card h2{font-size:15px;margin:0 0 12px}
.card h3{font-size:14px;margin:14px 0 8px}
.grid{display:grid;gap:16px;grid-template-columns:repeat(auto-fit,minmax(280px,1fr))}
.row{display:flex;gap:10px;align-items:center;flex-wrap:wrap}
.col{display:flex;flex-direction:column;gap:6px}
.muted{color:var(--muted)}.small{font-size:12px}.mono{font-family:var(--mono);font-size:12.5px}
label{display:flex;flex-direction:column;gap:4px;font-size:12.5px;color:var(--muted)}
label.inline{flex-direction:row;align-items:center;gap:6px;color:var(--text);font-size:13px}
input,select,textarea{font:inherit;color:var(--text);background:var(--panel2);border:1px solid var(--line);border-radius:8px;padding:7px 9px;min-width:0}
input[type=checkbox]{width:auto}
textarea{width:100%;resize:vertical;min-height:90px}
input:focus,select:focus,textarea:focus{outline:2px solid color-mix(in srgb,var(--accent) 40%,transparent);border-color:var(--accent)}
button,.btn{font:inherit;border:1px solid var(--line);background:var(--panel2);color:var(--text);border-radius:8px;padding:7px 13px;cursor:pointer;text-decoration:none;display:inline-flex;align-items:center;gap:6px}
button.primary,.btn.primary{background:var(--accent);border-color:var(--accent);color:var(--accent-ink);font-weight:600}
button.danger{color:var(--bad)}
button:disabled{opacity:.5;cursor:default}
table{width:100%;border-collapse:collapse}
th,td{text-align:left;padding:7px 8px;border-bottom:1px solid var(--line);vertical-align:top}
th{font-size:12px;color:var(--muted);font-weight:600}
.badge{display:inline-block;padding:1px 8px;border-radius:999px;font-size:12px;background:var(--panel2);border:1px solid var(--line)}
.badge.ok{color:var(--ok)}.badge.warn{color:var(--warn)}.badge.bad{color:var(--bad)}
.bar{height:6px;background:var(--panel2);border-radius:99px;overflow:hidden}
.bar>i{display:block;height:100%;background:var(--accent);transition:width .3s}
pre.log{background:var(--panel2);border:1px solid var(--line);border-radius:8px;padding:10px;max-height:320px;overflow:auto;font:12px/1.45 var(--mono);white-space:pre-wrap;margin:0}
.drop{border:2px dashed var(--line);border-radius:var(--radius);padding:22px;text-align:center;color:var(--muted)}
.drop.over{border-color:var(--accent);color:var(--text)}
#toasts{position:fixed;right:16px;bottom:16px;display:flex;flex-direction:column;gap:8px;z-index:50}
.toast{background:var(--panel);border:1px solid var(--line);border-left:4px solid var(--accent);padding:10px 14px;border-radius:8px;box-shadow:0 6px 20px rgba(0,0,0,.15);max-width:380px}
.toast.bad{border-left-color:var(--bad)}.toast.ok{border-left-color:var(--ok)}
.empty{color:var(--muted);padding:18px;text-align:center}
@media (max-width:640px){main{padding:12px}header.top{padding:10px 12px}nav.tabs{padding:0 8px}}
''')
print("wrote", os.path.join(APP_DIR, 'kit.css'))

In [ ]:
# Shared page scripts
# Writes kit.js into APP_DIR. Safe to re-run.
import os
APP_DIR = '/kaggle/working/video_app'
os.makedirs(APP_DIR, exist_ok=True)
with open(os.path.join(APP_DIR, 'kit.js'), "w", encoding="utf-8") as f:
    f.write(r'''// Shared helpers for the agentic-notebooks app pages (served at /static/kit.js).
// The browser already holds the HTTP Basic login, so fetch() calls need no token.
const $ = (sel, root = document) => root.querySelector(sel);
const $$ = (sel, root = document) => [...root.querySelectorAll(sel)];

function el(tag, attrs = {}, ...kids) {
  const e = document.createElement(tag);
  for (const [k, v] of Object.entries(attrs || {})) {
    if (v == null || v === false) continue;
    if (k.startsWith("on")) e.addEventListener(k.slice(2), v);
    else if (k === "html") e.innerHTML = v;
    else if (k === "style" && typeof v === "object") Object.assign(e.style, v);
    else e.setAttribute(k, v === true ? "" : v);
  }
  for (const kid of kids.flat()) if (kid != null && kid !== false) e.append(kid.nodeType ? kid : String(kid));
  return e;
}

async function api(path, opts = {}) {
  const init = { method: opts.method || (opts.json !== undefined || opts.body ? "POST" : "GET"), headers: {} };
  if (opts.json !== undefined) { init.body = JSON.stringify(opts.json); init.headers["Content-Type"] = "application/json"; }
  else if (opts.body) init.body = opts.body;
  const r = await fetch(path, init);
  if (r.status === 401) { location.reload(); throw new Error("Signed out; please sign in again"); }
  const ct = r.headers.get("Content-Type") || "";
  const data = ct.includes("json") ? await r.json() : await r.text();
  if (!r.ok) throw new Error((data && data.error) || r.status + " " + r.statusText);
  return data;
}

function toast(msg, kind = "") {
  let box = $("#toasts");
  if (!box) { box = el("div", { id: "toasts" }); document.body.append(box); }
  const t = el("div", { class: "toast " + kind }, msg);
  box.append(t);
  setTimeout(() => t.remove(), kind === "bad" ? 9000 : 4500);
}

async function guard(fn, okMsg) {
  try { const r = await fn(); if (okMsg) toast(okMsg, "ok"); return r; }
  catch (e) { toast(e.message || String(e), "bad"); throw e; }
}

function setupTabs(onChange) {
  const btns = $$("nav.tabs button[data-tab]");
  const show = (name) => {
    btns.forEach(b => b.classList.toggle("on", b.dataset.tab === name));
    $$("[data-pane]").forEach(p => p.classList.toggle("on", p.dataset.pane === name));
    try { history.replaceState(null, "", "#" + name); } catch (e) {}
    if (onChange) onChange(name);
  };
  btns.forEach(b => b.addEventListener("click", () => show(b.dataset.tab)));
  const first = location.hash.slice(1);
  show(btns.some(b => b.dataset.tab === first) ? first : btns[0].dataset.tab);
  return show;
}

function fmtBytes(n) {
  if (n == null) return "";
  const u = ["B", "KB", "MB", "GB", "TB"]; let i = 0;
  while (n >= 1024 && i < u.length - 1) { n /= 1024; i++; }
  return (i ? n.toFixed(1) : n) + " " + u[i];
}

function fmtDur(s) {
  if (s == null || isNaN(s)) return "";
  s = Math.max(0, Math.round(s));
  const h = Math.floor(s / 3600), m = Math.floor(s % 3600 / 60), sec = s % 60;
  return (h ? h + ":" + String(m).padStart(2, "0") : m) + ":" + String(sec).padStart(2, "0");
}

function fmtAgo(ts) {
  if (!ts) return "";
  const d = Date.now() / 1000 - ts;
  if (d < 60) return "just now";
  if (d < 3600) return Math.floor(d / 60) + " min ago";
  if (d < 86400) return Math.floor(d / 3600) + " h ago";
  return new Date(ts * 1000).toLocaleString();
}

function badge(state) {
  const kind = { done: "ok", ready: "ok", running: "warn", queued: "", loading: "warn", error: "bad", stopped: "bad" }[state] || "";
  return el("span", { class: "badge " + kind }, state || "?");
}

function progressBar(frac) {
  return el("div", { class: "bar" }, el("i", { style: { width: Math.round(Math.max(0, Math.min(1, frac || 0)) * 100) + "%" } }));
}

function every(ms, fn) {
  let stopped = false;
  const tick = async () => { if (stopped) return; try { await fn(); } catch (e) {} if (!stopped) setTimeout(tick, ms); };
  tick();
  return () => { stopped = true; };
}

function dropZone(zone, input, onFiles) {
  zone.addEventListener("click", () => input.click());
  input.addEventListener("change", () => { if (input.files.length) onFiles([...input.files]); input.value = ""; });
  zone.addEventListener("dragover", e => { e.preventDefault(); zone.classList.add("over"); });
  zone.addEventListener("dragleave", () => zone.classList.remove("over"));
  zone.addEventListener("drop", e => { e.preventDefault(); zone.classList.remove("over"); if (e.dataTransfer.files.length) onFiles([...e.dataTransfer.files]); });
}

function mcpHelp(name) {
  const url = location.origin + "/mcp";
  return "claude mcp add --transport http " + name + " " + url + " --header \"Authorization: Bearer <password>\"";
}
''')
print("wrote", os.path.join(APP_DIR, 'kit.js'))

In [ ]:
# Reads any link: articles (trafilatura), GitHub, YouTube, PDF; photos and language
# Writes vs_source.py into APP_DIR. Safe to re-run.
import os
APP_DIR = '/kaggle/working/video_app'
os.makedirs(APP_DIR, exist_ok=True)
with open(os.path.join(APP_DIR, 'vs_source.py'), "w", encoding="utf-8") as f:
    f.write(r'''"""Turns any input into source material (text, facts, photos, language) for the storyboard writer.

fetch_source(text) accepts:
- a GitHub repository URL      -> description, stats, topics, languages, file tree, README, README screenshots
- a YouTube URL                -> title, channel, description and subtitles (yt-dlp, no video download), thumbnail
- a PDF link                   -> text and embedded images (pymupdf)
- any other web page / article -> main text (trafilatura when installed), title, date, photos with captions;
                                  falls back to a headless-browser fetch for sites that block plain downloads
- anything else                -> treated as a plain prompt
Returns {"kind", "url", "title", "description", "text", "facts", "images", "language", "instructions"}.
images: [{"url", "alt", "caption", "score"}] best first (downloading and checking them is vs_server's job).
"""
import html
import html.parser
import json
import os
import re
import shutil
import subprocess
import tempfile
import urllib.error
import urllib.parse
import urllib.request

UA = "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126 Safari/537.36"
MAX_TEXT = 14000
TIMEOUT = 30
NE_MARKERS = ("छ", "छन्", "गरेको", "भएको", "हुन्छ", "गर्न", "पनि", "थियो", "भने", "रहेको", "गरी", "लागि", "जना", "सम्म", "बाट")
HI_MARKERS = ("है", "हैं", "में", "का", "की", "के", "और", "था", "थी", "गया", "किया", "रहा", "लिए", "साथ", "बाद")
BAD_IMG = re.compile(r"logo|icon|avatar|sprite|placeholder|blank|pixel|spacer|tracking|gravatar|emoji|badge|banner|"
                     r"/ads?[/_-]|advert|sponsor|facebook\.com|twitter\.com|\.svg(\?|$)|\.gif(\?|$)|data:image|doubleclick|"
                     r"googlesyndication|adservice|adnxs|taboola|outbrain|mgid\.com|/banners?/|/promo", re.I)
# Blocks that hold ads, sponsored posts, "related news", share buttons or sidebars: their pictures are not the story's
AD_HINT = re.compile(r"(^|[\s_-])(ads?|adv|advert\w*|sponsor\w*|promo\w*|banner\w*|dfp|gpt|adsense|outbrain|taboola|mgid|"
                     r"related|recommend\w*|trending|popular|most|also|more|sidebar|widget|newsletter|share|social|"
                     r"subscribe|footer)($|[\s_-])", re.I)
VOID = {"area", "base", "br", "col", "embed", "hr", "img", "input", "link", "meta", "source", "track", "wbr"}


def _get(url, headers=None, limit=25 << 20):
    req = urllib.request.Request(url, headers=dict({"User-Agent": UA, "Accept-Language": "ne,en;q=0.8,hi;q=0.6",
                                                    "Accept": "text/html,application/xhtml+xml,*/*;q=0.8"}, **(headers or {})))
    with urllib.request.urlopen(req, timeout=TIMEOUT) as r:
        data = r.read(limit + 1)
        if len(data) > limit:
            raise ValueError("the page is larger than %d MB" % (limit >> 20))
        return data, r.headers.get("Content-Type", ""), r.geturl()


def browser_fetch(url):
    """Renders the page in headless Chromium (for sites that block scripts or need JavaScript)."""
    from playwright.sync_api import sync_playwright
    with sync_playwright() as pw:
        b = pw.chromium.launch(executable_path=os.environ.get("CHROMIUM_PATH") or None, args=["--no-sandbox"])
        page = b.new_page(user_agent=UA, locale="ne-NP", viewport={"width": 1366, "height": 900})
        page.goto(url, wait_until="domcontentloaded", timeout=45000)
        try:
            page.wait_for_load_state("networkidle", timeout=8000)
        except Exception:
            pass
        page.mouse.wheel(0, 6000)                       # trigger lazy-loaded images
        page.wait_for_timeout(1200)
        raw, final = page.content(), page.url
        b.close()
    return raw, final


def _clip(text, n=MAX_TEXT):
    text = re.sub(r"\n{3,}", "\n\n", re.sub(r"[ \t]+", " ", text or "")).strip()
    return text if len(text) <= n else text[:n].rsplit(" ", 1)[0] + " ..."


def _human(n):
    n = int(n or 0)
    return "%.1fk" % (n / 1000) if n >= 1000 else str(n)


def detect_language(text, hint=None):
    """Two-letter code from the script (Devanagari / Latin / ...) plus Nepali-vs-Hindi word markers."""
    sample = (text or "")[:6000]
    letters = [c for c in sample if c.isalpha()]
    if not letters:
        return (hint or "en")[:2]
    deva = sum(1 for c in letters if "ऀ" <= c <= "ॿ") / len(letters)
    if deva > 0.3:
        words = re.findall(r"[ऀ-ॿ]+", sample)
        ne = sum(1 for w in words if w in NE_MARKERS)
        hi = sum(1 for w in words if w in HI_MARKERS)
        return "hi" if hi > ne * 1.3 else "ne"
    for lo, hi_, code in (("ঀ", "৿", "bn"), ("஀", "௿", "ta"), ("؀", "ۿ", "ur"),
                          ("一", "鿿", "zh"), ("぀", "ヿ", "ja")):
        if sum(1 for c in letters if lo <= c <= hi_) / len(letters) > 0.3:
            return code
    h = (hint or "").lower()[:2]
    return h if h and h.isalpha() and h not in ("ne", "hi") else "en"


# ---------------------------------------------------------------------- GitHub
GH = re.compile(r"^https?://(?:www\.)?github\.com/([\w.-]+)/([\w.-]+?)(?:\.git)?(?:[/?#].*)?$")


def github(url):
    m = GH.match(url)
    owner, repo = m.group(1), m.group(2)
    hdr = {"Accept": "application/vnd.github+json"}
    token = os.environ.get("GITHUB_TOKEN")
    if token:
        hdr["Authorization"] = "Bearer " + token
    api = "https://api.github.com/repos/%s/%s" % (owner, repo)
    try:
        info = json.loads(_get(api, hdr)[0])
    except urllib.error.HTTPError as e:
        if e.code == 404:
            raise ValueError("GitHub repository %s/%s not found (or private)" % (owner, repo))
        if e.code == 403:
            raise ValueError("GitHub API rate limit reached; add a GITHUB_TOKEN secret or try again later")
        raise
    langs = {}
    try:
        langs = json.loads(_get(api + "/languages", hdr)[0])
    except Exception:
        pass
    readme = ""
    try:
        readme = _get(api + "/readme", dict(hdr, Accept="application/vnd.github.raw"))[0].decode("utf-8", "replace")
    except Exception:
        pass
    tree = []
    try:
        tree = [("%s/" % x["name"]) if x["type"] == "dir" else x["name"]
                for x in json.loads(_get(api + "/contents", hdr)[0])][:40]
    except Exception:
        pass
    branch = info.get("default_branch") or "HEAD"
    images = [{"url": "https://opengraph.githubassets.com/1/%s/%s" % (owner, repo), "alt": info.get("full_name"),
               "caption": "", "score": 5}]
    for m2 in re.finditer(r"!\[([^\]]*)\]\(([^)\s]+)|<img[^>]+src=[\"']([^\"']+)", readme):
        src = m2.group(2) or m2.group(3)
        if not src or BAD_IMG.search(src) or "shields.io" in src or "badge" in src:
            continue
        if not re.match(r"^https?://", src):
            src = "https://raw.githubusercontent.com/%s/%s/%s/%s" % (owner, repo, branch, src.lstrip("./"))
        images.append({"url": src, "alt": (m2.group(1) or "")[:120], "caption": "", "score": 3})
    readme = re.sub(r"<!--.*?-->", "", readme, flags=re.S)
    readme = re.sub(r"\[!\[[^\]]*\]\([^)]*\)\]\([^)]*\)", "", readme)
    readme = re.sub(r"!\[[^\]]*\]\([^)]*\)", "", readme)
    readme = re.sub(r"<img[^>]*>", "", readme, flags=re.I)
    total = sum(langs.values()) or 1
    top_langs = ["%s %d%%" % (k, round(100 * v / total)) for k, v in sorted(langs.items(), key=lambda kv: -kv[1])[:4]]
    facts = {"name": info.get("full_name"), "site": "GitHub", "stars": _human(info.get("stargazers_count")),
             "forks": _human(info.get("forks_count")), "language": info.get("language"), "languages": top_langs,
             "license": (info.get("license") or {}).get("spdx_id"), "topics": info.get("topics", [])[:8],
             "homepage": info.get("homepage") or None, "created": (info.get("created_at") or "")[:10],
             "updated": (info.get("pushed_at") or "")[:10], "open_issues": info.get("open_issues_count")}
    text = "Repository: %s\nDescription: %s\nStars: %s, forks: %s\nLanguages: %s\nTopics: %s\nLicense: %s\n" \
           "Top-level files: %s\n\nREADME:\n%s" % (
               info.get("full_name"), info.get("description") or "", facts["stars"], facts["forks"],
               ", ".join(top_langs), ", ".join(facts["topics"]), facts["license"], ", ".join(tree), readme)
    return {"kind": "github", "url": info.get("html_url", url), "title": info.get("full_name") or repo,
            "description": info.get("description") or "", "text": _clip(text), "facts": facts, "images": images[:10],
            "language": detect_language(readme, "en")}


# ---------------------------------------------------------------------- YouTube
YT = re.compile(r"^https?://(?:www\.|m\.)?(?:youtube\.com/(?:watch|shorts/|live/)|youtu\.be/)", re.I)


def _vtt_text(vtt):
    lines, last = [], None
    for line in vtt.splitlines():
        line = line.strip()
        if not line or "-->" in line or line.startswith(("WEBVTT", "Kind:", "Language:", "NOTE")) or line.isdigit():
            continue
        line = re.sub(r"<[^>]+>", "", html.unescape(line))
        if line != last:
            lines.append(line)
            last = line
    return " ".join(lines)


def youtube(url):
    if not shutil.which("yt-dlp"):
        raise ValueError("YouTube links need yt-dlp (installed by the notebook's install cell)")
    with tempfile.TemporaryDirectory() as d:
        p = subprocess.run(["yt-dlp", "--skip-download", "--write-info-json", "--write-auto-subs", "--write-subs",
                            "--sub-langs", "ne.*,ne,hi.*,en.*,en", "--sub-format", "vtt", "--no-playlist", "-o",
                            os.path.join(d, "v.%(ext)s"), url], capture_output=True, text=True, timeout=180)
        info_path = os.path.join(d, "v.info.json")
        if not os.path.exists(info_path):
            raise ValueError("could not read that YouTube video: %s" % (p.stderr.strip().splitlines() or ["?"])[-1])
        info = json.load(open(info_path, encoding="utf-8"))
        subs = ""
        for f in sorted(os.listdir(d)):
            if f.endswith(".vtt"):
                subs = _vtt_text(open(os.path.join(d, f), encoding="utf-8", errors="replace").read())
                break
    facts = {"channel": info.get("channel") or info.get("uploader"), "site": "YouTube", "views": _human(info.get("view_count")),
             "duration": "%d:%02d" % divmod(int(info.get("duration") or 0), 60), "date": info.get("upload_date")}
    text = "Video: %s\nChannel: %s\n\nDescription:\n%s\n\nTranscript:\n%s" % (
        info.get("title"), facts["channel"], info.get("description") or "", subs or "(no subtitles)")
    thumbs = sorted((t for t in info.get("thumbnails") or [] if t.get("width")), key=lambda t: -t["width"])
    images = [{"url": (thumbs[0]["url"] if thumbs else info.get("thumbnail")), "alt": info.get("title"), "caption": "", "score": 4}]
    return {"kind": "youtube", "url": url, "title": info.get("title") or "YouTube video",
            "description": (info.get("description") or "")[:300], "text": _clip(text), "facts": facts,
            "images": [i for i in images if i["url"]], "language": detect_language(text, info.get("language"))}


# ---------------------------------------------------------------------- web pages
class _Page(html.parser.HTMLParser):
    """One pass over the page: metadata, JSON-LD, readable text (fallback) and candidate photos."""
    SKIP = {"script", "style", "noscript", "svg", "nav", "footer", "header", "aside", "form", "button", "iframe",
            "template", "select"}
    BLOCK = {"p", "div", "section", "article", "li", "br", "h1", "h2", "h3", "h4", "h5", "h6", "pre", "blockquote",
             "tr", "td", "main", "figcaption", "dd", "dt"}
    ARTICLE_HINT = re.compile(r"(^|[\s_-])(content|entry|post|story|article|news|detail|single|body|main)", re.I)

    def __init__(self, base):
        super().__init__(convert_charrefs=True)
        self.base = base
        self.out, self.skip, self.meta, self.title, self.in_title, self.lang = [], 0, {}, "", False, ""
        self.ld, self.in_ld, self.imgs, self.depth_article, self.in_cap, self.cap = [], False, [], 0, False, ""
        self.stack, self.depth_ad, self.links = [], 0, []
        self.host = _site(base)

    def handle_starttag(self, tag, attrs):
        a = {k: (v or "") for k, v in attrs}
        if tag == "html":
            self.lang = a.get("lang", "")
        if tag == "meta":
            key = (a.get("property") or a.get("name") or a.get("itemprop") or "").lower()
            if key and a.get("content"):
                self.meta.setdefault(key, a["content"])
        elif tag == "title":
            self.in_title = True
        elif tag == "script" and "ld+json" in a.get("type", ""):
            self.in_ld = True
            self.ld.append("")
            return
        names = a.get("class", "") + " " + a.get("id", "")
        art = tag in ("article", "main", "figure") or bool(self.ARTICLE_HINT.search(names))
        ad = tag in ("aside", "nav", "footer", "ins") or bool(AD_HINT.search(names))
        if tag not in VOID:                             # void tags never close: keep them off the stack
            self.depth_article += art
            self.depth_ad += ad
            self.stack.append((tag, art, ad))
        if tag == "a":
            href = urllib.parse.urljoin(self.base, a.get("href", ""))
            self.links.append(bool(re.match(r"^https?://", href)) and _site(href) != self.host)
        if tag == "figcaption":
            self.in_cap, self.cap = True, ""
        if tag in ("img", "source"):
            src = a.get("data-src") or a.get("data-lazy-src") or a.get("data-original") or a.get("data-full-url") or ""
            srcset = a.get("data-srcset") or a.get("srcset") or ""
            if srcset:
                best = max((p.strip().split(" ") for p in srcset.split(",") if p.strip()),
                           key=lambda p: int(re.sub(r"\D", "", p[1]) or 0) if len(p) > 1 else 0, default=None)
                if best and (not src or len(best) > 1):
                    src = best[0]
            src = src or a.get("src", "")
            if src and not src.startswith("data:"):
                w = int(re.sub(r"\D", "", a.get("width", "")) or 0)
                self.imgs.append({"url": urllib.parse.urljoin(self.base, src), "alt": a.get("alt", "")[:200], "caption": "",
                                  "score": (2 if self.depth_article else 0) + (1 if len(a.get("alt", "")) > 15 else 0)
                                  - (3 if 0 < w < 300 else 0) - (10 if self.depth_ad else 0)
                                  - (6 if any(self.links) else 0)})   # a picture that links to another site is an ad
        if tag in self.SKIP:
            self.skip += 1
        elif tag in self.BLOCK:
            self.out.append("\n")
            if tag in ("h1", "h2", "h3"):
                self.out.append("## ")

    def handle_endtag(self, tag):
        if tag == "title":
            self.in_title = False
        elif tag == "script" and self.in_ld:
            self.in_ld = False
            return
        if tag == "figcaption":
            self.in_cap = False
            if self.imgs and not self.imgs[-1]["caption"]:
                self.imgs[-1]["caption"] = " ".join(self.cap.split())[:240]
        if tag == "a" and self.links:
            self.links.pop()
        if tag not in VOID and any(t == tag for t, _, _ in self.stack):
            while self.stack:                           # also closes unclosed <p>, <li>... inside it
                t, art, ad = self.stack.pop()
                self.depth_article = max(0, self.depth_article - art)
                self.depth_ad = max(0, self.depth_ad - ad)
                if t == tag:
                    break
        if tag in self.SKIP:
            self.skip = max(0, self.skip - 1)
        elif tag in self.BLOCK:
            self.out.append("\n")

    def handle_data(self, data):
        if self.in_ld:
            self.ld[-1] += data
        elif self.in_title:
            self.title += data
        else:
            if self.in_cap:
                self.cap += data
            if not self.skip:
                self.out.append(data)


def _site(url):
    """example.com for www.example.com and img.example.com (two labels; three for .com.np-style domains)."""
    host = (urllib.parse.urlparse(url).hostname or "").lower()
    parts = host.split(".")
    n = 3 if len(parts) > 2 and parts[-2] in ("com", "co", "org", "gov", "edu", "net", "ac") and len(parts[-1]) == 2 else 2
    return ".".join(parts[-n:])


def _ld_items(blobs):
    for b in blobs:
        try:
            d = json.loads(b.strip())
        except ValueError:
            continue
        stack = [d]
        while stack:
            x = stack.pop()
            if isinstance(x, list):
                stack.extend(x)
            elif isinstance(x, dict):
                if "@graph" in x:
                    stack.extend(x["@graph"] if isinstance(x["@graph"], list) else [x["@graph"]])
                yield x


def _ld_images(item):
    im = item.get("image")
    for v in (im if isinstance(im, list) else [im]):
        if isinstance(v, str):
            yield v
        elif isinstance(v, dict) and v.get("url"):
            yield v["url"]


def wp_original(url):
    """WordPress thumbnails (photo-300x200.jpg) -> the full-size original (photo.jpg)."""
    return re.sub(r"-\d{2,4}x\d{2,4}(?=\.(jpe?g|png|webp)(\?|$))", "", url, flags=re.I)


def harvest_images(p, final, trafi_images=()):
    cands = []
    for key, score in (("og:image", 6), ("og:image:url", 6), ("twitter:image", 5), ("twitter:image:src", 5), ("image", 4)):
        if p.meta.get(key):
            cands.append({"url": urllib.parse.urljoin(final, p.meta[key]), "alt": p.meta.get("og:image:alt", ""), "caption": "", "score": score})
    for item in _ld_items(p.ld):
        if str(item.get("@type", "")).lower() in ("newsarticle", "article", "blogposting", "reportagenewsarticle", "webpage"):
            for u in _ld_images(item):
                cands.append({"url": urllib.parse.urljoin(final, u), "alt": item.get("headline", "")[:200], "caption": "", "score": 5})
    for u in trafi_images:
        cands.append({"url": urllib.parse.urljoin(final, u), "alt": "", "caption": "", "score": 4})
    cands += p.imgs
    seen, out = {}, []
    for c in cands:
        c["url"] = wp_original(html.unescape(c["url"]).strip())
        if not re.match(r"^https?://", c["url"]) or BAD_IMG.search(c["url"]):
            continue
        key = re.sub(r"[?#].*$", "", c["url"])
        if key in seen:
            prev = seen[key]
            prev["score"] = max(prev["score"], c["score"]) + 1
            prev["caption"] = prev["caption"] or c["caption"]
            prev["alt"] = prev["alt"] or c["alt"]
            continue
        seen[key] = c
        out.append(c)
    out = [c for c in out if c["score"] >= 0]          # tiny or off-article pictures (sidebars, related news)
    out.sort(key=lambda c: -c["score"])
    return out[:14]


def _trafilatura(raw, url):
    try:
        import trafilatura
    except ImportError:
        return None
    try:
        doc = trafilatura.bare_extraction(raw, url=url, with_metadata=True, include_images=True, include_comments=False,
                                          include_tables=True, favor_precision=True)
    except TypeError:
        doc = trafilatura.bare_extraction(raw, url=url, include_images=True, include_comments=False)
    if doc is None:
        return None
    if not isinstance(doc, dict):
        doc = doc.as_dict() if hasattr(doc, "as_dict") else vars(doc)
    return doc


def webpage(url):
    raw = final = ctype = None
    try:
        data, ctype, final = _get(url)
        if "pdf" in ctype.lower() or final.lower().split("?")[0].endswith(".pdf") or data[:5] == b"%PDF-":
            return pdf(final, data)
        if ctype and not any(t in ctype.lower() for t in ("html", "xml", "text")):
            raise ValueError("unsupported content type %r at that link" % ctype)
        enc = re.search(r"charset=([\w-]+)", ctype or "")
        raw = data.decode(enc.group(1) if enc else "utf-8", "replace")
    except (urllib.error.HTTPError, urllib.error.URLError, OSError) as e:
        blocked = e
    else:
        blocked = None
    text, doc, p = "", None, None
    if raw:
        doc = _trafilatura(raw, final)
        p = _Page(final)
        p.feed(raw)
        p.close()
        text = (doc or {}).get("text") or ""
    if len(text.split()) < 60:                         # blocked, JavaScript-only, or a poor extraction: render it
        try:
            raw2, final2 = browser_fetch(url)
            doc2 = _trafilatura(raw2, final2)
            p2 = _Page(final2)
            p2.feed(raw2)
            p2.close()
            t2 = (doc2 or {}).get("text") or ""
            if len(t2.split()) > len(text.split()):
                raw, final, doc, p, text = raw2, final2, doc2, p2, t2
        except Exception as e:
            if raw is None:
                raise ValueError("could not open that link (%s; the browser fallback also failed: %s)" % (
                    blocked or "no content", str(e)[:200]))
    if not text and p is not None:                     # no trafilatura: our own readable-text pass
        lines = [re.sub(r"\s+", " ", ln).strip() for ln in "".join(p.out).splitlines()]
        text = "\n".join(ln for ln in lines if ln.startswith("## ") or len(ln.split()) >= 6)
    if len(text.split()) < 30:
        raise ValueError("that page has almost no readable text (it may need a login)")
    doc = doc or {}
    meta = p.meta if p else {}
    title = doc.get("title") or meta.get("og:title") or " ".join((p.title if p else "").split()) or urllib.parse.urlparse(final).netloc
    desc = doc.get("description") or meta.get("og:description") or meta.get("description") or ""
    site = doc.get("sitename") or meta.get("og:site_name") or urllib.parse.urlparse(final).netloc.replace("www.", "")
    date = doc.get("date") or meta.get("article:published_time", "")[:10]
    trafi_imgs = [u for u in re.findall(r"!\[[^\]]*\]\(([^)\s]+)", text)] + ([doc["image"]] if doc.get("image") else [])
    text = re.sub(r"!\[[^\]]*\]\([^)]*\)", "", text)
    lang = detect_language(title + "\n" + text, (p.lang if p else "") or doc.get("language"))
    return {"kind": "article", "url": final, "title": html.unescape(title)[:200], "description": html.unescape(desc)[:400],
            "text": _clip("Title: %s\nSite: %s\nDate: %s\nAuthor: %s\nSummary: %s\n\n%s" % (
                title, site, date, doc.get("author") or "", desc, text)),
            "facts": {"site": site, "date": date, "author": doc.get("author"), "words": len(text.split())},
            "images": harvest_images(p, final, trafi_imgs) if p else [], "language": lang}


def pdf(url, data):
    try:
        import pymupdf
    except ImportError:
        raise ValueError("PDF links need pymupdf (installed by the notebook's install cell)")
    doc = pymupdf.open(stream=data, filetype="pdf")
    text = "\n".join(page.get_text("text") for page in doc)
    title = (doc.metadata or {}).get("title") or os.path.basename(urllib.parse.urlparse(url).path) or "Document"
    images = []
    for page in doc:
        for im in page.get_images(full=True)[:3]:
            try:
                info = doc.extract_image(im[0])
            except Exception:
                continue
            if info and info.get("width", 0) >= 500 and info.get("height", 0) >= 300:
                images.append({"bytes": info["image"], "ext": info.get("ext", "png"), "alt": "", "caption": "", "score": 3})
        if len(images) >= 6:
            break
    pages = len(doc)
    doc.close()
    if len(text.split()) < 40:
        raise ValueError("that PDF has no text layer (scanned?)")
    return {"kind": "pdf", "url": url, "title": title[:200], "description": "", "text": _clip(text),
            "facts": {"pages": pages, "site": urllib.parse.urlparse(url).netloc.replace("www.", "")},
            "images": images, "language": detect_language(text)}


# ---------------------------------------------------------------------- entry point
def fetch_source(text):
    text = (text or "").strip()
    if not text:
        raise ValueError("give a link or describe the video")
    first = text.split()[0]
    if re.match(r"^https?://", first):
        url, extra = first, text[len(first):].strip()
        if GH.match(url):
            src = github(url)
        elif YT.match(url):
            src = youtube(url)
        else:
            src = webpage(url)
        src["instructions"] = extra
        return src
    return {"kind": "prompt", "url": None, "title": text[:80], "description": "", "text": _clip(text),
            "facts": {}, "images": [], "language": detect_language(text), "instructions": ""}
''')
print("wrote", os.path.join(APP_DIR, 'vs_source.py'))

In [ ]:
# Script and storyboard in the video's language, validation and fallback
# Writes vs_story.py into APP_DIR. Safe to re-run.
import os
APP_DIR = '/kaggle/working/video_app'
os.makedirs(APP_DIR, exist_ok=True)
with open(os.path.join(APP_DIR, 'vs_story.py'), "w", encoding="utf-8") as f:
    f.write(r'''"""Storyboard: the JSON the local LLM writes, its validation (including the language and script), and a
fallback built from the source's own sentences when the model fails.

storyboard = {
  "title", "tagline", "language": "ne", "category": "news|tech|education|story|promo|other",
  "tone": "tragic|serious|neutral|upbeat|inspiring", "music_prompt": "...",
  "scenes": [{"layout": "headline|photo|bullets|stats|code|quote|outro", "heading", "kicker", "bullets": [...],
              "narration", "photo": <index into the source photos or -1>, "image_prompt", "code", "quote",
              "stats": [{"value", "label"}]}]
}
The model never writes HTML: vs_scenes.py animates whatever it puts in these fields.
"""
import json
import re
import urllib.request

LAYOUTS = ("headline", "photo", "bullets", "stats", "code", "quote", "outro")
ALIASES = {"title": "headline", "image": "photo", "intro": "headline", "end": "outro"}
LENGTHS = {15: (2, 3), 30: (4, 5), 60: (6, 8), 90: (8, 11)}
WPS = {"en": 2.4, "ne": 1.55, "hi": 1.8}           # spoken words per second (Nepali TTS speaks long words slowly)
LANGS = {"en": ("English", "English", ""), "ne": ("Nepali", "नेपाली", ", in Devanagari script"),
         "hi": ("Hindi", "हिन्दी", ", in Devanagari script"), "es": ("Spanish", "español", ""),
         "fr": ("French", "français", ""), "it": ("Italian", "italiano", ""), "pt": ("Portuguese", "português", ""),
         "bn": ("Bengali", "বাংলা", ", in Bengali script"), "ur": ("Urdu", "اردو", ", in Urdu script")}
DEVANAGARI = {"ne", "hi", "mr", "sa", "mai"}
NO_PHOTO = -2                                      # the user chose "no picture" for a scene (-1 = pick one for me)
CATEGORIES = ("news", "tech", "education", "story", "promo", "other")
TONES = ("tragic", "serious", "neutral", "upbeat", "inspiring")

PROMPT = """You are an experienced {role}. Turn the SOURCE below into a {length}-second video storyboard
(it will be rendered both as a vertical reel and as a landscape video).

LANGUAGE: write every "title", "tagline", "heading", "kicker", "bullets", "quote" and "narration" in {lang_name}
({lang_native}){script}. Never use any other language or script for those fields. Only "image_prompt" and
"music_prompt" are written in English.

Return ONLY a JSON object:
{{
  "title": "video title, max 9 words",
  "tagline": "one-line summary, max 16 words",
  "category": one of "news", "tech", "education", "story", "promo", "other",
  "tone": one of "tragic", "serious", "neutral", "upbeat", "inspiring",
  "music_prompt": "background music in English: genre, mood, tempo, instruments, no vocals",
  "place": "the main real place where the story happens, in English with district and country (e.g. 'Bardaghat, Nawalparasi, Nepal'), or empty if there is none",
  "place_local": "the same place name in {lang_name}, short (e.g. a town and district), or empty",
  "scenes": [
    {{
      "layout": one of "headline", "photo", "bullets", "stats", "quote", "code", "outro",
      "heading": "on-screen headline for this scene, max 8 words",
      "kicker": "tiny label above it, e.g. a place and date, max 4 words (may be empty)",
      "bullets": ["for bullets: 2-4 points, max 9 words each; for photo: one optional sub-line"],
      "narration": "what the voice says during this scene: 1-3 natural spoken sentences, max {max_words} words",
      "photo": index of the PHOTOS entry that fits this scene best, or -1,
      "image_prompt": "only if no photo fits and the topic is not a real news event: an illustration idea, else empty",
      "code": "code layout only: a short real snippet from the source",
      "quote": "quote layout only: a sentence that appears in the source"
    }}
  ]
}}

Rules:
- {n_min} to {n_max} scenes. The first scene uses layout "headline", the last uses layout "outro".
- Narration in total about {words} words, so it fits {length} seconds. Spoken, clear and concrete.
- {photo_rule}
- Every fact must come from the SOURCE: never invent numbers, names, places, quotes or causes.{news_rule}{stats_rule}{extra}

PHOTOS:
{photos}

SOURCE ({kind}):
{text}
"""


def lang_info(lang):
    return LANGS.get(lang, (lang, lang, ""))


def build_prompt(src, length, lang, photos):
    n_min, n_max = LENGTHS.get(length, LENGTHS[60])
    name, native, script = lang_info(lang)
    words = int(length * WPS.get(lang, 2.3))
    role = {"github": "tech video producer", "youtube": "video editor", "pdf": "explainer video producer"}.get(
        src["kind"], "news and explainer video producer")
    photo_rule = ("Prefer \"photo\" scenes built on the PHOTOS, and give each scene its own photo when there are "
                  "enough. Pick the photo whose description matches the scene." if photos else
                  "There are no photos; use \"bullets\" and \"quote\" scenes, and image prompts only for non-news topics.")
    news_rule = ("\n- If this is news about real people: be factual, neutral and respectful; attribute claims (\"police said\"); "
                 "no speculation, no graphic detail, no dramatic adjectives; image_prompt must be empty.")
    stats_rule = ("\n- Do not add a \"stats\" scene: one with the real numbers is added automatically."
                  if src["kind"] in ("github", "youtube") else "\n- Use \"stats\" only for numbers stated in the SOURCE.")
    extra = ("\n- The user asked: " + src["instructions"]) if src.get("instructions") else ""
    plist = "\n".join("[%d] %s" % (i, p.get("description") or p.get("caption") or p.get("alt") or "(no description)")
                      for i, p in enumerate(photos)) or "(none)"
    return PROMPT.format(role=role, length=length, lang_name=name, lang_native=native, script=script,
                         max_words=45 if length > 15 else 30, n_min=n_min, n_max=n_max, words=words, photo_rule=photo_rule,
                         news_rule=news_rule, stats_rule=stats_rule, extra=extra, photos=plist, kind=src["kind"],
                         text=src["text"])


def ollama_json(url, model, prompt, keep_alive="10m", timeout=900):
    body = json.dumps({"model": model, "stream": False, "format": "json", "keep_alive": keep_alive,
                       "messages": [{"role": "user", "content": prompt}],
                       "options": {"temperature": 0.5, "num_ctx": 16384}}).encode()
    req = urllib.request.Request(url.rstrip("/") + "/api/chat", data=body, headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        content = json.loads(r.read())["message"]["content"]
    m = re.search(r"\{.*\}", content, re.S)
    if not m:
        raise ValueError("the model did not return JSON")
    return json.loads(m.group(0))


# ---------------------------------------------------------------------- validation
def _s(v, n):
    v = re.sub(r"\s+", " ", str(v or "")).strip()
    return v if len(v) <= n else v[:n].rsplit(" ", 1)[0].rstrip(",;:") + "…"


def _words(v, n):
    w = str(v or "").split()
    return " ".join(w[:n]) + ("…" if len(w) > n else "")


def script_check(sb, lang):
    """Raises ValueError when the on-screen text and narration are not in the expected script."""
    text = " ".join([sb.get("title") or ""] + [" ".join([sc.get("heading") or "", sc.get("narration") or ""] + list(sc.get("bullets") or []))
                                               for sc in sb.get("scenes", []) if isinstance(sc, dict)])
    letters = [c for c in text if c.isalpha()]
    if not letters:
        raise ValueError("the storyboard has no text")
    cjk = sum(1 for c in letters if "぀" <= c <= "鿿") / len(letters)
    deva = sum(1 for c in letters if "ऀ" <= c <= "ॿ") / len(letters)
    latin = sum(1 for c in letters if c.isascii() or "À" <= c <= "ɏ") / len(letters)
    name = lang_info(lang)[0]
    if lang not in ("zh", "ja") and cjk > 0.02:
        raise ValueError("the text drifted into Chinese/Japanese instead of %s" % name)
    if lang in DEVANAGARI and deva < 0.6:
        raise ValueError("the text is not in %s (Devanagari); only %d%% Devanagari" % (name, deva * 100))
    if lang in ("en", "es", "fr", "it", "pt") and latin < 0.8:
        raise ValueError("the text is not in %s" % name)


def stats_for(src):
    f = src.get("facts") or {}
    if src["kind"] == "github":
        stats = [{"value": f.get("stars"), "label": "GitHub stars"}, {"value": f.get("forks"), "label": "forks"}]
        if f.get("language"):
            stats.append({"value": f["language"], "label": "main language"})
        elif f.get("license"):
            stats.append({"value": f["license"], "label": "license"})
        return [s for s in stats if s["value"]]
    if src["kind"] == "youtube":
        return [s for s in ({"value": f.get("views"), "label": "views"}, {"value": f.get("duration"), "label": "long"},
                            {"value": f.get("channel"), "label": "channel"}) if s["value"]]
    return []


def stats_narration(src):
    f = src.get("facts") or {}
    if src["kind"] == "github":
        return "%s has %s stars and %s forks on GitHub%s." % (
            (f.get("name") or "It").split("/")[-1], f.get("stars"), f.get("forks"),
            ", written mostly in %s" % f["language"] if f.get("language") else "")
    if src["kind"] == "youtube":
        return "The original video by %s has %s views." % (f.get("channel") or "its creator", f.get("views"))
    return ""


def _scene(sc, n_photos):
    layout = str(sc.get("layout") or "bullets").lower().strip()
    layout = ALIASES.get(layout, layout)
    layout = layout if layout in LAYOUTS else "bullets"
    bullets = [_s(b, 90) for b in (sc.get("bullets") or []) if isinstance(b, (str, int, float)) and str(b).strip()][:4]
    try:
        photo = int(sc.get("photo", -1))
    except (TypeError, ValueError):
        photo = -1
    out = {"layout": layout, "heading": _s(sc.get("heading"), 80), "kicker": _s(sc.get("kicker"), 40), "bullets": bullets,
           "narration": _words(sc.get("narration"), 60), "image_prompt": _s(sc.get("image_prompt"), 300),
           "photo": photo if 0 <= photo < n_photos or photo == NO_PHOTO else -1,
           "code": "\n".join(str(sc.get("code") or "").splitlines()[:10])[:600], "quote": _s(sc.get("quote"), 220), "stats": []}
    if layout == "stats":
        out["stats"] = [{"value": _s(x.get("value"), 20), "label": _s(x.get("label"), 30)} for x in (sc.get("stats") or [])
                        if isinstance(x, dict) and x.get("value")][:3]
    if layout == "bullets" and not bullets:
        out["layout"] = "photo" if out["photo"] >= 0 else "quote"
        out["quote"] = out["quote"] or out["narration"]
    if layout == "code" and not out["code"].strip():
        out["layout"] = "bullets" if bullets else "photo"
    if layout == "quote" and not out["quote"]:
        out["layout"] = "bullets" if bullets else "photo"
    return out


def clean(sb, src, length, lang, n_photos=0, keep_stats=False, fit=True):
    """Validates and repairs a storyboard; raises ValueError if nothing usable is left."""
    if not isinstance(sb, dict) or not isinstance(sb.get("scenes"), list):
        raise ValueError("storyboard must be an object with a scenes list")
    n_min, n_max = LENGTHS.get(length, LENGTHS[60])
    scenes = []
    for sc in sb["scenes"]:
        if not isinstance(sc, dict):
            continue
        out = _scene(sc, n_photos)
        if out["layout"] == "stats" and not (keep_stats and out["stats"]):
            continue                                   # only real numbers: added below from the source facts
        if not out["heading"] and not out["narration"]:
            continue
        scenes.append(out)
    if len(scenes) < (1 if length <= 15 else 2):
        raise ValueError("storyboard has too few usable scenes")
    blank = {"heading": "", "kicker": "", "bullets": [], "narration": "", "image_prompt": "", "photo": -1, "code": "", "quote": "", "stats": []}
    if scenes[0]["layout"] != "headline":
        scenes.insert(0, dict(blank, layout="headline", photo=scenes[0]["photo"], image_prompt=scenes[0]["image_prompt"]))
    if scenes[-1]["layout"] != "outro":
        scenes.append(dict(blank, layout="outro"))
    for sc in scenes[1:-1]:
        if sc["layout"] in ("headline", "outro"):
            sc["layout"] = "bullets" if sc["bullets"] else "photo"
    stats = stats_for(src)
    if stats and not any(s["layout"] == "stats" for s in scenes) and length > 15:
        scenes.insert(min(2, len(scenes) - 1), dict(blank, layout="stats", heading="", narration=stats_narration(src), stats=stats))
    while len(scenes) > n_max:                         # keep headline/outro, drop from the middle
        scenes.pop(-2)
    cat = str(sb.get("category") or "").lower()
    tone = str(sb.get("tone") or "").lower()
    title = _s(sb.get("title") or src["title"], 90)
    out = {"title": title, "tagline": _s(sb.get("tagline") or src.get("description"), 140), "language": lang,
           "category": cat if cat in CATEGORIES else ("news" if src["kind"] == "article" else "other"),
           "tone": tone if tone in TONES else "neutral",
           "music_prompt": _s(sb.get("music_prompt") or "light background music, no vocals", 200),
           "scenes": scenes, "source": {k: src.get(k) for k in ("kind", "url", "title", "facts")},
           "place": _s(sb.get("place"), 120), "place_local": _s(sb.get("place_local"), 60)}
    if out["category"] == "news" or out["tone"] in ("tragic", "serious"):
        for sc in scenes:
            sc["image_prompt"] = ""                    # never invent pictures of real events
        if out["tone"] in ("tragic", "serious"):
            out["music_prompt"] = "soft slow ambient pad, minimal, somber, warm, no drums, no vocals"
    h, o = scenes[0], scenes[-1]
    h["heading"] = h["heading"] or title
    h["narration"] = h["narration"] or out["tagline"] or title
    o["heading"] = o["heading"] or ""
    o["narration"] = o["narration"] or ""
    if fit:                                            # user-edited scripts are kept as written
        fit_words(scenes, int(length * WPS.get(lang, 2.3) * 0.85), lang)
    return out


def _fit(text, n, lang):
    """Whole sentences up to n words; a single long sentence is cut and closed."""
    out = []
    for snt in re.split(r"(?<=[.!?।॥])\s+", text.strip()):
        if out and len(" ".join(out + [snt]).split()) > n:
            break
        out.append(snt)
    w = " ".join(out).split()
    if len(w) <= n:
        return " ".join(w)
    return " ".join(w[:n]).rstrip(",;:—-") + ("।" if lang in DEVANAGARI else ".")


def fit_words(scenes, budget, lang):
    """Shortens narration so the whole video fits its length (the model often writes too much)."""
    total = sum(len((sc.get("narration") or "").split()) for sc in scenes)
    if total <= budget:
        return
    for sc in scenes:
        n = len((sc.get("narration") or "").split())
        if n:
            sc["narration"] = _fit(sc["narration"], max(5, int(n * budget / total)), lang)


def sentences(text):
    return [s.strip() for s in re.split(r"(?<=[.!?।॥])\s+", text) if len(s.split()) >= 4]


def fallback(src, length, lang, n_photos=0):
    """A plain storyboard straight from the source's own sentences (so it is always in the right language)."""
    body = re.sub(r"^(Title|Site|Date|Author|Summary):.*$", "", src["text"], flags=re.M)
    sents = sentences(body)
    n = LENGTHS.get(length, LENGTHS[60])[1] - 2
    per = max(1, min(3, len(sents) // max(1, n)))
    scenes = [{"layout": "headline", "heading": src["title"], "narration": src.get("description") or (sents[0] if sents else src["title"]),
               "photo": 0 if n_photos else -1}]
    for i in range(max(1, n)):
        chunk = sents[1 + i * per: 1 + (i + 1) * per]
        if not chunk:
            break
        first = chunk[0].split()
        scenes.append({"layout": "photo" if n_photos else "bullets", "heading": " ".join(first[:8]) + ("…" if len(first) > 8 else ""),
                       "bullets": [_s(s, 90) for s in chunk[1:3]] or [_s(chunk[0], 90)], "narration": _words(" ".join(chunk), 40),
                       "photo": (i + 1) % n_photos if n_photos else -1})
    scenes.append({"layout": "outro", "heading": src["title"], "narration": ""})
    return clean({"title": src["title"], "tagline": src.get("description"), "category": "news" if src["kind"] == "article" else "other",
                  "scenes": scenes}, src, length, lang, n_photos)


def write_storyboard(src, length, lang, ollama_url, model, keep_alive="10m", photos=(), log=None):
    prompt = build_prompt(src, length, lang, photos)
    last, nudge = None, ""
    for attempt in range(3):
        try:
            sb = clean(ollama_json(ollama_url, model, prompt + nudge, keep_alive), src, length, lang, len(photos))
            script_check(sb, lang)
            return sb, None
        except Exception as e:
            last = e
            if log:
                log.warning("storyboard attempt %d failed: %s", attempt + 1, e)
            name, native, script = lang_info(lang)
            nudge = ("\n\nYOUR PREVIOUS ANSWER WAS REJECTED: %s. Answer again with valid JSON, every text field in %s (%s)%s."
                     % (str(e)[:200], name, native, script))
    return fallback(src, length, lang, len(photos)), "the model's script was unusable (%s); used the article's own sentences" % last


def validate_user_storyboard(sb, src, length, lang, n_photos):
    """For storyboards edited in the page or sent by an agent: same repairs, keeping their stats scenes."""
    out = clean(sb, dict(src, kind="prompt"), length, sb.get("language") or lang, n_photos, keep_stats=True, fit=False)
    out["source"] = sb.get("source") or out["source"]
    for k in ("category", "tone", "music_prompt"):
        if sb.get(k):
            out[k] = sb[k] if k == "music_prompt" else out[k]
    return out
''')
print("wrote", os.path.join(APP_DIR, 'vs_story.py'))

In [ ]:
# Animated scene templates: broadcast, documentary and more; landscape and reel
# Writes vs_scenes.py into APP_DIR. Safe to re-run.
import os
APP_DIR = '/kaggle/working/video_app'
os.makedirs(APP_DIR, exist_ok=True)
with open(os.path.join(APP_DIR, 'vs_scenes.py'), "w", encoding="utf-8") as f:
    f.write(r'''"""Animated HTML for each storyboard scene, in landscape (16:9) and reel (9:16).

Every animation is a CSS animation, so the renderer can seek it exactly: window.__seek(t) pauses all
animations at t seconds and updates captions, count-ups and code typing. Frames are identical however slow
the machine is. Sizes use vmin, so one template serves both orientations.

Photos (from the source page, or generated) get two layers: a blurred, darkened copy that fills the frame
and the photo itself, cropped when its shape suits the frame and letterboxed over the blur when it does not,
then colour-graded, vignetted and slowly pushed in. Styles: broadcast (news), documentary (film grain, light
leaks), midnight, paper, neon, swiss. Labels follow the video's language.
"""
import html
import json
import os
import re
from string import Template

DEVA = "'Noto Sans Devanagari','Mukta','Noto Sans',sans-serif"
STYLES = {
    "broadcast": {"bg1": "#0a0d14", "bg2": "#18233b", "fg": "#ffffff", "muted": "#c4cad6", "accent": "#e1261c",
                  "accent2": "#ffcc00", "card": "rgba(255,255,255,.08)", "panel": "rgba(8,12,22,.86)",
                  "font": "'Inter','Noto Sans'," + DEVA, "head": "'Inter','Noto Sans'," + DEVA, "weight": "800"},
    "documentary": {"bg1": "#0d0b09", "bg2": "#2a2219", "fg": "#f5efe6", "muted": "#cbbfae", "accent": "#e9b872",
                    "accent2": "#c4572e", "card": "rgba(255,240,220,.07)", "panel": "rgba(14,11,8,.72)",
                    "font": "'Noto Serif','DejaVu Serif','Noto Serif Devanagari'," + DEVA,
                    "head": "'Noto Serif','DejaVu Serif','Noto Serif Devanagari'," + DEVA, "weight": "700"},
    "midnight": {"bg1": "#0b1020", "bg2": "#1b2550", "fg": "#f3f5ff", "muted": "#a9b3d6", "accent": "#7c9cff",
                 "accent2": "#ff7ac6", "card": "rgba(255,255,255,.07)", "panel": "rgba(11,16,32,.78)",
                 "font": "'Inter','Noto Sans'," + DEVA, "head": "'Inter','Noto Sans'," + DEVA, "weight": "800"},
    "paper": {"bg1": "#f6f1e7", "bg2": "#e9dfcc", "fg": "#1f1a14", "muted": "#6b5f50", "accent": "#d2552d",
              "accent2": "#2d6cd2", "card": "rgba(0,0,0,.05)", "panel": "rgba(246,241,231,.9)",
              "font": "'Noto Serif','DejaVu Serif','Noto Serif Devanagari'," + DEVA,
              "head": "'Noto Serif','DejaVu Serif','Noto Serif Devanagari'," + DEVA, "weight": "700"},
    "neon": {"bg1": "#07020f", "bg2": "#1a0638", "fg": "#f6f0ff", "muted": "#b9a6e0", "accent": "#00f0ff",
             "accent2": "#ff2bd6", "card": "rgba(0,240,255,.07)", "panel": "rgba(7,2,15,.8)",
             "font": "'Inter','Noto Sans'," + DEVA, "head": "'Noto Sans Mono','DejaVu Sans Mono'," + DEVA, "weight": "800"},
    "swiss": {"bg1": "#ffffff", "bg2": "#f1f1f1", "fg": "#111111", "muted": "#555555", "accent": "#e3001b",
              "accent2": "#111111", "card": "#f4f4f4", "panel": "rgba(255,255,255,.92)",
              "font": "'Helvetica Neue','Liberation Sans',Arial," + DEVA, "head": "'Helvetica Neue','Liberation Sans',Arial," + DEVA,
              "weight": "900"},
}
LIGHT = {"paper", "swiss"}
SIZES = {"landscape": (1920, 1080), "reel": (1080, 1920)}
LABELS = {
    "en": {"latest": "LATEST", "numbers": "By the numbers", "source": "Source", "photo": "Photo", "thanks": "Thanks for watching", "where": "Where"},
    "ne": {"latest": "ताजा समाचार", "numbers": "तथ्यांकमा", "source": "स्रोत", "photo": "तस्बिर", "thanks": "हेर्नुभएकोमा धन्यवाद", "where": "घटनास्थल"},
    "hi": {"latest": "ताज़ा ख़बर", "numbers": "आँकड़ों में", "source": "स्रोत", "photo": "फ़ोटो", "thanks": "देखने के लिए धन्यवाद", "where": "स्थान"},
    "es": {"latest": "ÚLTIMA HORA", "numbers": "En cifras", "source": "Fuente", "photo": "Foto", "thanks": "Gracias por ver", "where": "Dónde"},
    "fr": {"latest": "DERNIÈRE MINUTE", "numbers": "En chiffres", "source": "Source", "photo": "Photo", "thanks": "Merci d'avoir regardé", "where": "Où"},
}
DEVANAGARI_LANGS = {"ne", "hi", "mr", "sa", "mai", "bho", "new"}
GRAIN = ("data:image/svg+xml;utf8,<svg xmlns='http://www.w3.org/2000/svg' width='220' height='220'><filter id='n'>"
         "<feTurbulence type='fractalNoise' baseFrequency='.85' numOctaves='3' stitchTiles='stitch'/></filter>"
         "<rect width='100%25' height='100%25' filter='url(%23n)' opacity='.55'/></svg>")

CSS = Template("""
:root{--bg1:$bg1;--bg2:$bg2;--fg:$fg;--muted:$muted;--accent:$accent;--accent2:$accent2;--card:$card;--panel:$panel;
 --font:$font;--head:$head;--dur:${dur}s}
*{box-sizing:border-box}
html,body{margin:0;width:100%;height:100%;overflow:hidden;background:var(--bg1);color:var(--fg);font-family:var(--font)}
.stage{position:absolute;inset:0;overflow:hidden;background:radial-gradient(120% 90% at 20% 10%,var(--bg2),var(--bg1) 70%)}
.blob{position:absolute;width:70vmin;height:70vmin;border-radius:50%;filter:blur(9vmin);opacity:.42;animation:drift var(--dur) ease-in-out both}
.blob.a{background:var(--accent);left:-15vmin;top:-20vmin}.blob.b{background:var(--accent2);right:-20vmin;bottom:-25vmin;animation-direction:reverse}
@keyframes drift{from{transform:translate(0,0) scale(1)}to{transform:translate(12vmin,8vmin) scale(1.15)}}
/* photo layers */
.ph{position:absolute;inset:0;overflow:hidden}
.ph .fill{position:absolute;inset:-6%;background-size:cover;background-position:center;filter:blur(5vmin) brightness(.55) saturate(1.2)}
.ph .img{position:absolute;inset:0;background-repeat:no-repeat;background-position:center;filter:contrast(1.06) saturate(1.06);
 animation:$kb var(--dur) cubic-bezier(.3,.1,.3,1) both}
.ph .img{background-size:cover}
.ph .img.band{box-shadow:0 2vmin 7vmin rgba(0,0,0,.55);animation-name:kbband!important}
.reel .ph .img.band{inset:auto 0 auto 0;top:13vmin}
.landscape .ph .img.band{inset:0 0 0 auto}
@keyframes kbband{from{transform:scale(1)}to{transform:scale(1.05)}}
.ph .img.gl.live{animation-name:none!important}
.ph canvas.depth,.ph canvas.seq{position:absolute;inset:0;width:100%;height:100%;display:block;opacity:0}
.ph .img.gl.live canvas{opacity:1}
.ph .img.cut{animation-name:cutbg!important}
@keyframes cutbg{from{transform:scale(1.03)}to{transform:scale(1.07)}}
.ph .dimmer{position:absolute;inset:0;background:rgba(0,0,0,.42);animation:dimin 1.1s ease-out .3s both}
@keyframes dimin{from{opacity:0}to{opacity:1}}
.ph .cutfg{position:absolute;inset:0;background-size:inherit;background-position:center;background-repeat:no-repeat;
 filter:drop-shadow(0 1.2vmin 2.6vmin rgba(0,0,0,.65));animation:cutpop var(--dur) cubic-bezier(.2,.7,.2,1) both}
@keyframes cutpop{0%{transform:scale(1) translateY(0)}12%{transform:scale(1.035) translateY(-.6%)}100%{transform:scale(1.085) translateY(-1.2%)}}
@keyframes kbin{from{transform:scale(1.04)}to{transform:scale(1.16)}}
@keyframes kbout{from{transform:scale(1.16)}to{transform:scale(1.04)}}
@keyframes kbleft{from{transform:scale(1.14) translateX(2.5%)}to{transform:scale(1.14) translateX(-2.5%)}}
@keyframes kbright{from{transform:scale(1.14) translateX(-2.5%)}to{transform:scale(1.14) translateX(2.5%)}}
.ph .vig{position:absolute;inset:0;background:radial-gradient(120% 95% at 50% 45%,transparent 55%,rgba(0,0,0,.55))}
.ph .shade{position:absolute;inset:0;background:linear-gradient(180deg,rgba(0,0,0,.05) 35%,rgba(0,0,0,.78))}
.light .ph .shade{background:linear-gradient(180deg,rgba(255,255,255,0) 40%,rgba(255,255,255,.9))}
.dim .ph .shade{background:linear-gradient(180deg,rgba(0,0,0,.45),rgba(0,0,0,.75))}
.light.dim .ph .shade{background:linear-gradient(180deg,rgba(255,255,255,.65),rgba(255,255,255,.9))}
/* text over a photo, in every style: a dark scrim under the text, white type with a soft shadow, and a
   frosted plate behind centred headlines. Light styles switch to this too: dark type on photos is unreadable. */
html.onphoto{--fg:#fff;--muted:rgba(255,255,255,.88)}
html.onphoto .ph .shade{background:linear-gradient(180deg,rgba(0,0,0,.35) 0%,rgba(0,0,0,0) 22%,rgba(0,0,0,.25) 48%,rgba(0,0,0,.82) 78%,rgba(0,0,0,.92))}
html.onphoto.dim .ph .shade{background:linear-gradient(180deg,rgba(0,0,0,.5),rgba(0,0,0,.38) 45%,rgba(0,0,0,.8))}
html.onphoto h1,html.onphoto h2,html.onphoto .tag,html.onphoto .quote,html.onphoto li{text-shadow:0 .2vmin .4vmin rgba(0,0,0,.6),0 .6vmin 3vmin rgba(0,0,0,.5)}
html.onphoto .kicker{color:#fff;background:var(--accent);display:inline-block;align-self:flex-start;padding:.7vmin 1.6vmin;border-radius:.5vmin;text-shadow:none}
html.onphoto .center .kicker{align-self:center}
html.onphoto .chip{background:rgba(0,0,0,.5);color:#fff}
html.onphoto .url{color:#fff;text-shadow:0 .2vmin 1vmin rgba(0,0,0,.7)}
.plate{background:rgba(10,12,18,.55);backdrop-filter:blur(1.8vmin) saturate(1.15);-webkit-backdrop-filter:blur(1.8vmin);
 border:.15vmin solid rgba(255,255,255,.12);border-radius:2.6vmin;padding:4.5vmin 5.5vmin;max-width:100%;
 box-shadow:0 3vmin 8vmin rgba(0,0,0,.35)}
.center .plate{display:flex;flex-direction:column;align-items:center}
/* band photos: the text goes beside (landscape) or below (reel) the picture, on the darkened blur */
html.onphoto.pc.reel .ph .shade{background:linear-gradient(180deg,rgba(0,0,0,.5),rgba(0,0,0,0) 13vmin,rgba(0,0,0,0) calc(var(--below) - 8vmin),rgba(0,0,0,.72) var(--below),rgba(0,0,0,.85))}
html.onphoto.pc.landscape .ph .shade{background:linear-gradient(90deg,rgba(0,0,0,.82),rgba(0,0,0,.62) 45%,rgba(0,0,0,.08) 62%)}
.pc.reel .content.lower,.pc.reel .content.center,.pc.reel .content.qt{justify-content:center;padding-top:var(--below,82vmin);padding-bottom:44vmin}
.pc.reel h1{font-size:8.2vmin}.pc.reel h2,.pc.reel .lower h2{font-size:6.8vmin;margin-bottom:2vmin}.pc.reel .tag{font-size:3.7vmin;margin-top:2.4vmin}
.pc.reel .plate{padding:3.6vmin 4.4vmin}
.pc.reel .l3{bottom:auto;top:var(--below,82vmin)}
.pc.landscape .content{padding-right:var(--beside,92vmin);padding-bottom:12vmin}
.pc.landscape .center{align-items:flex-start;text-align:left}.pc.landscape .center .plate{align-items:flex-start}
.pc.landscape .center .tag{margin-left:0}.pc.landscape .lower h2{max-width:100%}
.pc.landscape .l3{right:var(--beside,92vmin)}
.credit{position:absolute;right:3vmin;bottom:3vmin;z-index:25;font:500 1.7vmin var(--font);color:rgba(255,255,255,.75);
 text-shadow:0 .2vmin .6vmin rgba(0,0,0,.6)}
.reel .credit,.broadcast .credit{bottom:auto;top:9vmin}
.broadcast:not(.reel) .credit{top:5vmin}
/* motion */
.up{opacity:0;animation:up .9s cubic-bezier(.2,.75,.2,1) both}
@keyframes up{from{opacity:0;transform:translateY(5vmin)}to{opacity:1;transform:none}}
.pop{opacity:0;animation:pop .8s cubic-bezier(.2,.9,.25,1.2) both}
@keyframes pop{from{opacity:0;transform:scale(.85)}to{opacity:1;transform:none}}
.w{display:inline-block;opacity:0;animation:wd .7s cubic-bezier(.2,.75,.2,1) both;white-space:pre}
@keyframes wd{from{opacity:0;transform:translateY(60%);filter:blur(.6vmin)}to{opacity:1;transform:none;filter:none}}
.slide{opacity:0;animation:sl .7s cubic-bezier(.2,.8,.2,1) both}
@keyframes sl{from{opacity:0;transform:translateX(-6vmin)}to{opacity:1;transform:none}}
/* transitions */
.veil{position:absolute;inset:0;background:var(--bg1);pointer-events:none;z-index:60;
 animation:veilin .45s ease-out both,veilout .45s ease-in calc(var(--dur) - .45s) both}
@keyframes veilin{from{opacity:1}to{opacity:0}}@keyframes veilout{from{opacity:0}to{opacity:1}}
.first .veil{animation:veilout .45s ease-in calc(var(--dur) - .45s) both;opacity:0}
.last .veil{animation:veilin .45s ease-out both}
.wipe{position:absolute;inset:0;z-index:61;pointer-events:none;background:var(--accent);
 animation:wipein .5s cubic-bezier(.7,0,.3,1) both,wipeout .4s cubic-bezier(.7,0,.3,1) calc(var(--dur) - .4s) both}
@keyframes wipein{from{transform:translateX(0)}to{transform:translateX(101%)}}
@keyframes wipeout{from{transform:translateX(-101%)}to{transform:translateX(0)}}
.first .wipe{animation:wipeout .4s cubic-bezier(.7,0,.3,1) calc(var(--dur) - .4s) both;transform:translateX(-101%)}
.last .wipe{animation:wipein .5s cubic-bezier(.7,0,.3,1) both}
.broadcast .veil{display:none}
.progress{position:absolute;left:0;bottom:0;height:.7vmin;background:var(--accent);z-index:40;animation:prog var(--dur) linear both}
@keyframes prog{from{width:${p0}%}to{width:${p1}%}}
/* texture */
.grain{position:absolute;inset:-50%;z-index:45;pointer-events:none;opacity:.16;background-image:url("$grain");
 animation:grain .5s steps(5) infinite;mix-blend-mode:overlay}
@keyframes grain{0%{transform:translate(0,0)}20%{transform:translate(-3%,2%)}40%{transform:translate(2%,-3%)}60%{transform:translate(-2%,-1%)}80%{transform:translate(3%,3%)}}
.leak{position:absolute;inset:0;z-index:44;pointer-events:none;mix-blend-mode:screen;
 background:radial-gradient(40% 55% at 0% 30%,rgba(255,150,60,.55),transparent 70%),radial-gradient(35% 45% at 100% 80%,rgba(255,80,120,.35),transparent 70%);
 animation:leak var(--dur) ease-in-out both}
@keyframes leak{0%{opacity:0}25%{opacity:.85}60%{opacity:.25}100%{opacity:.6}}
.bars:before,.bars:after{content:"";position:absolute;left:0;right:0;height:7vmin;background:#000;z-index:46}
.bars:before{top:0}.bars:after{bottom:0}.reel .bars:before,.reel .bars:after{display:none}
/* type */
.content{position:absolute;inset:0;padding:8vmin 9vmin;display:flex;flex-direction:column;justify-content:center;z-index:5}
h1,h2{font-family:var(--head);font-weight:$weight;letter-spacing:-.015em;margin:0;line-height:1.08}
.deva h1,.deva h2{line-height:1.32;letter-spacing:0}
h1{font-size:8.6vmin}h2{font-size:6.4vmin;margin-bottom:4vmin}
.reel h1{font-size:9.4vmin}.reel h2{font-size:7vmin}
.kicker{font:700 2.6vmin var(--font);letter-spacing:.22em;text-transform:uppercase;color:var(--accent);margin-bottom:2.4vmin}
.deva .kicker{letter-spacing:.04em}
.chip{display:inline-block;align-self:flex-start;padding:1.1vmin 2.4vmin;border-radius:99vmin;background:var(--card);
 border:.25vmin solid color-mix(in srgb,var(--accent) 60%,transparent);color:var(--fg);font:600 3vmin var(--font);margin-bottom:3.5vmin}
.tag{font-size:3.9vmin;color:var(--muted);margin:3.5vmin 0 0;max-width:80%;line-height:1.35}
.reel .tag{max-width:100%}
ul{list-style:none;margin:0;padding:0}
li{font-size:4.1vmin;line-height:1.35;margin:0 0 2.6vmin;padding-left:5.5vmin;position:relative}
li::before{content:"";position:absolute;left:0;top:1.5vmin;width:2.4vmin;height:2.4vmin;border-radius:.6vmin;background:var(--accent)}
.split{flex-direction:row;align-items:center;gap:6vmin}
.split .text{flex:1.15;min-width:0}.split .pic{flex:1;height:74%;border-radius:2.4vmin;overflow:hidden;position:relative;
 box-shadow:0 3vmin 7vmin rgba(0,0,0,.35)}
.reel .split{flex-direction:column-reverse;justify-content:flex-end;padding-top:14vmin}
.reel .split .pic{flex:none;width:100%;height:44vmin;min-height:44vmin}.reel .split .text{flex:none;width:100%}
.lower{justify-content:flex-end;padding-bottom:24vmin}
.lower h2{font-size:7vmin;text-shadow:0 .5vmin 3vmin rgba(0,0,0,.5);max-width:88%}
.reel .lower{padding-bottom:58vmin}
.reel .content:not(.lower):not(.center){padding-bottom:46vmin;justify-content:flex-start;padding-top:16vmin}
.reel .stat{padding:3vmin 4vmin}.reel .stat b{font-size:9vmin}
.light .lower h2{text-shadow:none}
.stats{display:flex;gap:4vmin;flex-wrap:wrap}.reel .stats{flex-direction:column}
.stat{flex:1;min-width:30vmin;max-width:100%;overflow:hidden;background:var(--card);border-radius:3vmin;padding:4.5vmin 4vmin}
.stat b{display:block;font:$weight 11vmin var(--head);color:var(--accent);letter-spacing:-.03em;line-height:1.05}
.stat b.word{font-size:6.4vmin;line-height:1.2;overflow-wrap:anywhere}
.stat span{display:block;font-size:3.2vmin;color:var(--muted);margin-top:1.6vmin}
.win{background:#0d1117;color:#e6edf3;border-radius:2.4vmin;overflow:hidden;box-shadow:0 3vmin 7vmin rgba(0,0,0,.35)}
.win .bar{height:5vmin;background:#161b22;display:flex;align-items:center;gap:1.4vmin;padding:0 2.4vmin}
.win .bar i{width:1.8vmin;height:1.8vmin;border-radius:50%;background:#ff5f56}.win .bar i:nth-child(2){background:#ffbd2e}.win .bar i:nth-child(3){background:#27c93f}
pre{margin:0;padding:3.6vmin 4vmin;font:3.4vmin/1.45 'Noto Sans Mono','DejaVu Sans Mono',monospace;white-space:pre-wrap;word-break:break-word;min-height:24vmin}
pre .cur{display:inline-block;width:1.4vmin;height:3.2vmin;background:var(--accent);vertical-align:-.5vmin;margin-left:.3vmin}
.qmark{font:900 22vmin/0.7 Georgia,serif;color:var(--accent);height:12vmin}
.quote{font:500 5.4vmin/1.35 var(--head);max-width:92%}
.reel .quote{font-size:6vmin}
.src{color:var(--muted);font-size:3vmin;margin-top:4vmin}
.center{align-items:center;text-align:center}.center .chip{align-self:center}.center .tag{margin-left:auto;margin-right:auto}
.url{font:600 3.2vmin 'Noto Sans Mono','DejaVu Sans Mono',monospace;color:var(--accent);margin-top:4vmin;word-break:break-all}
.cap{position:absolute;left:50%;bottom:7vmin;transform:translateX(-50%);z-index:50;max-width:86%;text-align:center;
 font:700 3.6vmin/1.35 var(--font);color:#fff;background:rgba(0,0,0,.66);padding:1.3vmin 2.6vmin;border-radius:1.6vmin;opacity:0}
.reel .cap{bottom:30vmin;font-size:4.6vmin;max-width:84%}
.broadcast:not(.reel) .cap{bottom:11vmin}.broadcast.reel .cap{bottom:34vmin}
.num{position:absolute;right:5vmin;top:4.5vmin;font:700 2.4vmin var(--font);color:var(--muted);z-index:20;letter-spacing:.15em}
/* broadcast furniture */
.bug{position:absolute;left:5vmin;top:4.5vmin;z-index:30;display:flex;gap:1.2vmin;align-items:center}
.bug .live{background:var(--accent);color:#fff;font:800 2.5vmin var(--font);padding:.8vmin 1.8vmin;border-radius:.6vmin;letter-spacing:.06em}
.bug .site{background:rgba(0,0,0,.55);color:#fff;font:600 2.4vmin var(--font);padding:.8vmin 1.6vmin;border-radius:.6vmin}
.l3{position:absolute;left:5vmin;right:5vmin;bottom:25vmin;z-index:20}
.reel .l3{bottom:62vmin}
.l3 .k{display:inline-block;background:var(--accent);color:#fff;font:800 2.8vmin var(--font);padding:1vmin 2vmin;border-radius:.6vmin .6vmin 0 0}
.l3 .h{background:var(--panel);color:#fff;font:$weight 5.6vmin/1.3 var(--head);padding:2vmin 2.8vmin;border-left:1.1vmin solid var(--accent);
 border-radius:0 .8vmin .8vmin .8vmin}
.reel .l3 .h{font-size:6.2vmin}
.l3 .s{background:rgba(255,255,255,.92);color:#111;font:600 3vmin/1.35 var(--font);padding:1.2vmin 2.8vmin;display:inline-block;margin-top:.8vmin;border-radius:.6vmin}
/* motion-graphic accents: a light sweep over headline panels, a drawn-on underline, drifting dust */
.l3 .h,.plate{position:relative;overflow:hidden}
.l3 .h::after,.plate::after{content:"";position:absolute;top:0;bottom:0;left:-40%;width:30%;pointer-events:none;
 background:linear-gradient(100deg,transparent,rgba(255,255,255,.22),transparent);transform:skewX(-18deg);animation:sweep 1.4s ease-in-out .9s both}
@keyframes sweep{from{left:-40%}to{left:130%}}
.content h1::after,.content h2::after{content:"";display:block;height:.7vmin;width:12vmin;margin-top:2vmin;background:var(--accent);
 border-radius:.4vmin;transform-origin:left;animation:draw .8s cubic-bezier(.6,0,.2,1) .7s both}
.center h1::after{margin-left:auto;margin-right:auto;transform-origin:center}
.pc.landscape .center h1::after{margin-left:0}
@keyframes draw{from{transform:scaleX(0)}to{transform:scaleX(1)}}
.dust{position:absolute;inset:0;z-index:43;pointer-events:none;opacity:.55;
 background-image:radial-gradient(circle,rgba(255,240,210,.9) 0 .18vmin,transparent .3vmin),radial-gradient(circle,rgba(255,240,210,.6) 0 .12vmin,transparent .25vmin);
 background-size:23vmin 19vmin,31vmin 27vmin;background-position:0 0,11vmin 7vmin;animation:dust var(--dur) linear both}
@keyframes dust{from{transform:translate(0,0)}to{transform:translate(-4vmin,-7vmin)}}
/* map scene: three snapshots, each 3x closer, cross-faded while zooming = one continuous fly-in */
.map{position:absolute;inset:0;overflow:hidden;background:#0b0f17}
.map .lv{position:absolute;inset:0;background-size:cover;background-position:center;opacity:0}
.map .lv.wide{animation:mz1 var(--dur) linear both}.map .lv.mid{animation:mz2 var(--dur) linear both}.map .lv.close{animation:mz3 var(--dur) linear both}
@keyframes mz1{0%{opacity:1;transform:scale(1)}26%{opacity:1;transform:scale(2.6)}32%{opacity:0;transform:scale(3.03)}100%{opacity:0;transform:scale(3.03)}}
@keyframes mz2{0%,24%{opacity:0;transform:scale(.86)}32%{opacity:1;transform:scale(1)}56%{opacity:1;transform:scale(2.6)}62%{opacity:0;transform:scale(3.03)}100%{opacity:0;transform:scale(3.03)}}
@keyframes mz3{0%,54%{opacity:0;transform:scale(.86)}62%{opacity:1;transform:scale(1)}100%{opacity:1;transform:scale(1.18)}}
.map .tint{position:absolute;inset:0;background:radial-gradient(60% 60% at 50% 50%,transparent 40%,rgba(0,0,0,.55))}
.pin{position:absolute;left:50%;top:50%;z-index:6;width:0;height:0}
.pin .dot{position:absolute;left:-2.2vmin;top:-2.2vmin;width:4.4vmin;height:4.4vmin;border-radius:50%;background:var(--accent);
 border:.6vmin solid #fff;box-shadow:0 .6vmin 2vmin rgba(0,0,0,.5);animation:pindrop .7s cubic-bezier(.3,1.6,.5,1) calc(var(--dur) * .6) both}
@keyframes pindrop{from{opacity:0;transform:translateY(-12vmin) scale(.6)}to{opacity:1;transform:none}}
.pin .ring{position:absolute;left:-2.2vmin;top:-2.2vmin;width:4.4vmin;height:4.4vmin;border-radius:50%;border:.5vmin solid var(--accent);
 opacity:0;animation:ring 1.6s ease-out calc(var(--dur) * .6 + .5s) 4 both}
.pin .ring.r2{animation-delay:calc(var(--dur) * .6 + 1.3s)}
@keyframes ring{0%{opacity:.9;transform:scale(1)}100%{opacity:0;transform:scale(5)}}
.pin .lab{position:absolute;left:4.5vmin;top:-3.6vmin;white-space:nowrap;background:var(--panel);color:#fff;font:700 3.6vmin/1.2 var(--head);
 padding:1.2vmin 2.2vmin;border-radius:1vmin;border-left:.8vmin solid var(--accent);animation:sl .6s cubic-bezier(.2,.8,.2,1) calc(var(--dur) * .6 + .4s) both}
.osm{position:absolute;right:2vmin;bottom:2vmin;z-index:30;font:500 1.6vmin var(--font);color:rgba(255,255,255,.8);background:rgba(0,0,0,.45);padding:.4vmin 1vmin;border-radius:.6vmin}
.reel .osm{bottom:auto;top:13vmin}
.ticker{position:absolute;left:0;right:0;bottom:0;height:8vmin;z-index:35;display:flex;align-items:center;overflow:hidden;
 background:var(--accent2);color:#111;font:700 3vmin var(--font)}
.reel .ticker{bottom:20vmin}
.ticker .lab{flex:none;height:100%;display:flex;align-items:center;padding:0 2.4vmin;background:#111;color:#fff;z-index:2}
.ticker .run{white-space:nowrap;padding-left:4vmin;animation:tick var(--dur) linear both}
@keyframes tick{from{transform:translateX(30%)}to{transform:translateX(-60%)}}
""")

JS = """
const DUR = __DUR__, CAPS = __CAPS__, COUNTS = [...document.querySelectorAll('[data-count]')], TYPE = document.querySelector('[data-type]');
const capEl = document.querySelector('.cap');
function fmtNum(v, dec){ return dec ? v.toFixed(dec) : Math.round(v).toLocaleString('en-US'); }
window.__seek = function(t){
  document.getAnimations().forEach(a => { a.pause(); a.currentTime = t * 1000; });
  GL.forEach(draw => draw(t));
  if (capEl) { const c = CAPS.find(c => t >= c[0] && t < c[1]); capEl.textContent = c ? c[2] : ''; capEl.style.opacity = c ? 1 : 0; }
  COUNTS.forEach((el, i) => {
    const target = parseFloat(el.dataset.count), dec = +el.dataset.dec, f = Math.min(1, Math.max(0, (t - 0.6 - i * 0.2) / 1.4));
    el.textContent = fmtNum(target * (1 - Math.pow(1 - f, 3)), dec) + el.dataset.suffix;
  });
  if (TYPE) { const full = TYPE.dataset.type, n = Math.floor(Math.max(0, t - 0.9) * __CPS__);
    TYPE.textContent = full.slice(0, n); const cur = document.createElement('span'); cur.className = 'cur'; TYPE.appendChild(cur); }
};
const GL = [];
const VS = 'attribute vec2 p;varying vec2 v;void main(){v=p*.5+.5;gl_Position=vec4(p,0.,1.);}';
const FS = 'precision highp float;varying vec2 v;uniform sampler2D img,dep;uniform vec2 sc,dir;uniform float t,zoom,amt;' +
  'void main(){float e=t*t*(3.-2.*t);vec2 c=vec2(.5);vec2 uv=(v-c)*sc/(1.07+zoom*e)+c;vec2 s=dir*amt*(e*2.-1.);' +
  'vec2 q=uv;for(int i=0;i<10;i++){q=uv+s*(texture2D(dep,q).r-.45);}gl_FragColor=texture2D(img,clamp(q,0.,1.));}';
const MOVES = [[[.9, .25], .10], [[-1, .1], .05], [[.15, -.9], .07], [[-.6, -.4], -.04]];
function glLayer(cv, im, dp) {
  const box = cv.parentElement, w = Math.max(2, Math.round(box.clientWidth)), h = Math.max(2, Math.round(box.clientHeight));
  cv.width = w; cv.height = h;
  const g = cv.getContext('webgl', {preserveDrawingBuffer: true, antialias: false, premultipliedAlpha: false});
  if (!g) return null;
  const sh = (type, src) => { const s = g.createShader(type); g.shaderSource(s, src); g.compileShader(s); return s; };
  const pr = g.createProgram(); g.attachShader(pr, sh(g.VERTEX_SHADER, VS)); g.attachShader(pr, sh(g.FRAGMENT_SHADER, FS));
  g.linkProgram(pr); if (!g.getProgramParameter(pr, g.LINK_STATUS)) return null; g.useProgram(pr);
  const b = g.createBuffer(); g.bindBuffer(g.ARRAY_BUFFER, b);
  g.bufferData(g.ARRAY_BUFFER, new Float32Array([-1, -1, 1, -1, -1, 1, 1, 1]), g.STATIC_DRAW);
  const loc = g.getAttribLocation(pr, 'p'); g.enableVertexAttribArray(loc); g.vertexAttribPointer(loc, 2, g.FLOAT, false, 0, 0);
  g.pixelStorei(g.UNPACK_FLIP_Y_WEBGL, true);
  [[im, 0, 'img'], [dp, 1, 'dep']].forEach(([src, unit, name]) => {
    const tx = g.createTexture(); g.activeTexture(g.TEXTURE0 + unit); g.bindTexture(g.TEXTURE_2D, tx);
    g.texImage2D(g.TEXTURE_2D, 0, g.RGBA, g.RGBA, g.UNSIGNED_BYTE, src);
    g.texParameteri(g.TEXTURE_2D, g.TEXTURE_MIN_FILTER, g.LINEAR); g.texParameteri(g.TEXTURE_2D, g.TEXTURE_MAG_FILTER, g.LINEAR);
    g.texParameteri(g.TEXTURE_2D, g.TEXTURE_WRAP_S, g.CLAMP_TO_EDGE); g.texParameteri(g.TEXTURE_2D, g.TEXTURE_WRAP_T, g.CLAMP_TO_EDGE);
    g.uniform1i(g.getUniformLocation(pr, name), unit);
  });
  const ia = im.naturalWidth / im.naturalHeight, ca = w / h;
  g.uniform2f(g.getUniformLocation(pr, 'sc'), ca > ia ? 1 : ca / ia, ca > ia ? ia / ca : 1);
  const [d, z] = MOVES[+cv.dataset.move % 4];
  g.uniform2f(g.getUniformLocation(pr, 'dir'), d[0], d[1]); g.uniform1f(g.getUniformLocation(pr, 'zoom'), z);
  g.uniform1f(g.getUniformLocation(pr, 'amt'), 0.035);
  const ut = g.getUniformLocation(pr, 't');
  g.viewport(0, 0, w, h);
  if (g.getError() !== g.NO_ERROR) return null;
  return t => { g.uniform1f(ut, Math.min(1, Math.max(0, t / DUR))); g.drawArrays(g.TRIANGLE_STRIP, 0, 4); };
}
const loadImg = u => new Promise((ok, bad) => { const im = new Image(); im.onload = () => ok(im); im.onerror = bad; im.src = u; });
function seqLayer(cv, ims) {
  const box = cv.parentElement, w = Math.round(box.clientWidth), h = Math.round(box.clientHeight);
  cv.width = w; cv.height = h;
  const c = cv.getContext('2d'), fps = +cv.dataset.fps || 24, n = ims.length;
  return t => {                                     /* ping-pong so a 3 s clip fills a longer scene smoothly */
    let k = Math.floor(t * fps * 0.8) % Math.max(1, 2 * n - 2); if (k >= n) k = 2 * n - 2 - k;
    const im = ims[Math.max(0, Math.min(n - 1, k))], ia = im.naturalWidth / im.naturalHeight, ca = w / h;
    const sw = ca > ia ? im.naturalWidth : im.naturalHeight * ca, sh = ca > ia ? im.naturalWidth / ca : im.naturalHeight;
    c.drawImage(im, (im.naturalWidth - sw) / 2, (im.naturalHeight - sh) / 2, sw, sh, 0, 0, w, h);
  };
}
async function initGL() {
  for (const cv of document.querySelectorAll('canvas.seq')) {
    try {
      const ims = await Promise.all(JSON.parse(cv.dataset.frames).map(loadImg));
      if (ims.length) { GL.push(seqLayer(cv, ims)); cv.parentElement.classList.add('live'); }
    } catch (e) {}
  }
  for (const cv of document.querySelectorAll('canvas.depth')) {
    try {
      const [im, dp] = await Promise.all([loadImg(cv.dataset.img), loadImg(cv.dataset.depth)]);
      const draw = glLayer(cv, im, dp);
      if (draw) { GL.push(draw); cv.parentElement.classList.add('live'); }
    } catch (e) {}                                  /* no WebGL or no depth map: the CSS camera move stays */
  }
}
window.__ready = false;
const urls = [...document.querySelectorAll('[data-bg]')].map(e => e.dataset.bg);
Promise.all([document.fonts.ready, ...urls.map(u => new Promise(r => { const im = new Image(); im.onload = im.onerror = r; im.src = u; }))])
  .then(initGL).then(() => { window.__seek(0); window.__ready = true; });
"""


def esc(s):
    return html.escape(str(s or ""), quote=True)


def caption_chunks(text, start, dur, max_words):
    """[[start, end, text], ...] spread over the narration's duration, weighted by length."""
    words = (text or "").split()
    if not words or dur <= 0:
        return []
    chunks, cur = [], []
    for w in words:
        cur.append(w)
        if len(cur) >= max_words or re.search(r"[.!?;:।॥]$", w) and len(cur) >= 3:
            chunks.append(" ".join(cur))
            cur = []
    if cur:
        chunks.append(" ".join(cur))
    total = sum(len(c) for c in chunks)
    out, t = [], start
    for c in chunks:
        d = dur * len(c) / total
        out.append([round(t, 3), round(t + d, 3), c])
        t += d
    return out


def _num(value):
    """'12.3k' -> (12.3, 1, 'k'); 'Python' -> None. Devanagari digits are kept as words."""
    m = re.fullmatch(r"\s*([\d,]+(?:\.\d+)?)\s*([kKmMbB%+]*)\s*", str(value or ""))
    if not m:
        return None
    raw = m.group(1).replace(",", "")
    return float(raw), (len(raw.split(".")[1]) if "." in raw else 0), m.group(2)


def _url(path):
    return "file://" + os.path.abspath(path)


def photo_layer(ph, idx, fmt="landscape"):
    """ph: {"path", "fit": "cover"|"contain", "w", "h", "credit"}; idx picks the camera move. A photo whose shape does
    not suit the frame becomes a band: across the top of a reel (text goes below it) or down the right of a landscape
    frame (text goes left of it), so text never sits on the busy part of the picture."""
    if not ph or not ph.get("path"):
        return ""
    u = esc(_url(ph["path"]))
    cls, size = "img", ""
    if ph.get("fit") == "contain":
        pa = ph.get("w", 1) / float(ph.get("h", 1) or 1)
        cls = "img band"
        size = ("height:%.1fvmin;" % max(46, min(70, 100 / pa)) if fmt == "reel"
                else "width:%.1fvmin;" % max(56, min(100, 100 * pa)))
    inner = ""
    if ph.get("frames"):                                # an AI clip (non-news only): frames drawn at exact times
        cls += " gl"
        inner = '<canvas class="seq" data-fps="%s" data-frames="%s"></canvas>' % (
            esc(ph.get("fps", 24)), esc(json.dumps([_url(f) for f in ph["frames"]])))
    elif ph.get("cut"):                                   # the subject lifts off a dimming background
        cls += " cut"
        cu = esc(_url(ph["cut"]))
        inner = '<div class="dimmer"></div><div class="cutfg" data-bg="%s" style="background-image:url(\'%s\')"></div>' % (cu, cu)
    elif ph.get("depth"):                               # 2.5D parallax of the real photo (WebGL); CSS motion if WebGL fails
        cls += " gl"
        inner = '<canvas class="depth" data-img="%s" data-depth="%s" data-move="%d"></canvas>' % (
            u, esc(_url(ph["depth"])), idx % 4)
    return ('<div class="ph"><div class="fill" style="background-image:url(\'%s\')"></div>'
            '<div class="%s" data-bg="%s" style="%sbackground-image:url(\'%s\');animation-name:%s">%s</div>'
            '<div class="vig"></div><div class="shade"></div></div>') % (
        u, cls, u, size, u, ("kbin", "kbleft", "kbout", "kbright")[idx % 4], inner)


def words(text, start=0.25, step=0.07, limit=1.6):
    """Heading split into words that rise in one after another (kinetic type)."""
    ws = str(text or "").split()
    step = min(step, limit / max(1, len(ws)))
    return "".join('<span class="w" style="animation-delay:%.2fs">%s</span>%s' % (start + i * step, esc(w), " " if i < len(ws) - 1 else "")
                   for i, w in enumerate(ws))


def scene_html(scene, idx, n, story, style, fmt, dur, photo, captions, p0, p1):
    style = style if style in STYLES else "midnight"
    lang = (story.get("language") or "en").split("-")[0]
    L = dict(LABELS["en"], **LABELS.get(lang, {}))
    src = story.get("source") or {}
    facts = src.get("facts") or {}
    site = str(facts.get("site") or "")
    lay = {"image": "photo"}.get(scene["layout"], scene["layout"])
    h = scene.get("heading") or ""
    broadcast = style == "broadcast"
    bg = '<div class="blob a"></div><div class="blob b"></div>'
    credit = ""
    if photo and photo.get("credit"):
        credit = '<div class="credit">%s: %s</div>' % (esc(L["photo"]), esc(photo["credit"]))
    kicker = scene.get("kicker") or ""
    body = ""
    if lay in ("title", "headline"):
        bg += photo_layer(photo, idx, fmt)
        if broadcast:
            body = ('<div class="l3"><div class="k slide" style="animation-delay:.2s">%s</div><div class="h slide" style="animation-delay:.35s">%s</div>%s</div>') % (
                esc(kicker or L["latest"]), words(h, .5), '<div class="s up" style="animation-delay:1.1s">%s</div>' % esc(story.get("tagline")) if story.get("tagline") else "")
        else:
            label = {"github": "GitHub · " + str(src.get("title") or ""), "youtube": "YouTube", "pdf": "PDF"}.get(src.get("kind"), site)
            body = ('<div class="content center"><div class="plate up">%s<h1>%s</h1>%s</div></div>' if photo else
                    '<div class="content center">%s<h1>%s</h1>%s</div>') % (
                '<div class="chip up" style="animation-delay:.15s">%s</div>' % esc(kicker or label) if (kicker or label) else "", words(h, .3),
                '<p class="tag up" style="animation-delay:.9s">%s</p>' % esc(story.get("tagline")) if story.get("tagline") else "")
    elif lay == "bullets":
        lis = "".join('<li class="up" style="animation-delay:%.2fs">%s</li>' % (0.8 + i * 0.45, esc(b))
                      for i, b in enumerate(scene.get("bullets") or []))
        text = '<div class="text"><div class="kicker up" style="animation-delay:.1s">%s</div><h2>%s</h2><ul>%s</ul></div>' % (
            esc(kicker) or "%02d" % idx, words(h, .25), lis)
        pic = '<div class="pic pop" style="animation-delay:.35s">%s</div>' % photo_layer(dict(photo, fit="cover"), idx) if photo else ""
        body = '<div class="content %s">%s%s</div>' % ("split" if photo else "", text, pic)
    elif lay == "photo":
        bg += photo_layer(photo, idx, fmt)
        sub = (scene.get("bullets") or [""])[0]
        if broadcast:
            body = '<div class="l3"><div class="k slide" style="animation-delay:.2s">%s</div><div class="h slide" style="animation-delay:.3s">%s</div>%s</div>' % (
                esc(kicker or site or L["latest"]), words(h, .45), '<div class="s up" style="animation-delay:1s">%s</div>' % esc(sub) if sub else "")
        else:
            body = '<div class="content lower"><div class="kicker up" style="animation-delay:.2s">%s</div><h2>%s</h2>%s</div>' % (
                esc(kicker) or "%02d" % idx, words(h, .4), '<p class="tag up" style="animation-delay:1s">%s</p>' % esc(sub) if sub else "")
    elif lay == "map":
        m = scene.get("map") or {}
        bg = '<div class="map">%s<div class="tint"></div></div>' % "".join(
            '<div class="lv %s" data-bg="%s" style="background-image:url(\'%s\')"></div>' % (k, esc(_url(m[k])), esc(_url(m[k])))
            for k in ("wide", "mid", "close") if m.get(k))
        label = m.get("label") or ""
        body = ('<div class="pin"><div class="ring"></div><div class="ring r2"></div><div class="dot"></div>%s</div>'
                '<div class="osm">© OpenStreetMap</div>') % ('<div class="lab">%s</div>' % esc(label) if label else "")
        if broadcast:
            body += '<div class="l3"><div class="k slide" style="animation-delay:.2s">%s</div><div class="h slide" style="animation-delay:.3s">%s</div></div>' % (
                esc(L["where"]), words(h or label, .45))
        else:
            body += '<div class="content lower"><div class="kicker up" style="animation-delay:.2s">%s</div><h2>%s</h2></div>' % (
                esc(L["where"]), words(h or label, .4))
    elif lay == "stats":
        cards = ""
        for i, s in enumerate(scene.get("stats") or []):
            num = _num(s.get("value"))
            val = ('<b data-count="%s" data-dec="%d" data-suffix="%s">0</b>' % (num[0], num[1], esc(num[2])) if num
                   else '<b class="word">%s</b>' % esc(s.get("value")))
            cards += '<div class="stat pop" style="animation-delay:%.2fs">%s<span>%s</span></div>' % (0.4 + i * 0.2, val, esc(s.get("label")))
        body = '<div class="content"><h2>%s</h2><div class="stats">%s</div></div>' % (words(h or L["numbers"], .15), cards)
    elif lay == "code":
        body = ('<div class="content"><h2>%s</h2><div class="win pop" style="animation-delay:.4s"><div class="bar"><i></i><i></i><i></i></div>'
                '<pre data-type="%s"></pre></div></div>') % (words(h, .15), esc(scene.get("code")))
    elif lay == "quote":
        bg += photo_layer(photo, idx, fmt)
        body = ('<div class="content qt"><div class="qmark up">&ldquo;</div><div class="quote up" style="animation-delay:.3s">%s</div>'
                '<div class="src up" style="animation-delay:.9s">%s</div></div>') % (esc(scene.get("quote")), esc(scene.get("heading") or src.get("title")))
    else:  # outro
        bg += photo_layer(photo, idx, fmt)
        url = re.sub(r"^https?://(www\.)?", "", str(src.get("url") or "")).rstrip("/")
        if src.get("kind") != "github":
            url = url.split("/")[0]                       # articles: just the site
        tag = (scene.get("bullets") or [""])[0] if scene.get("bullets") else (story.get("tagline") or "")
        body = ('<div class="content center"><div class="plate up"><h1>%s</h1>%s%s</div></div>' if photo else
                '<div class="content center"><h1>%s</h1>%s%s</div>') % (
            words(h or L["thanks"], .2), '<p class="tag up" style="animation-delay:.8s">%s</p>' % esc(tag) if tag else "",
            '<div class="url up" style="animation-delay:1.1s">%s: %s</div>' % (esc(L["source"]), esc(url[:70])) if url else "")
    furniture = ""
    if broadcast:
        furniture = '<div class="bug"><span class="live">%s</span>%s</div>' % (esc(L["latest"]), '<span class="site">%s</span>' % esc(site) if site else "")
        if lay not in ("title", "headline", "outro"):
            furniture += '<div class="ticker"><div class="lab">%s</div><div class="run">%s</div></div>' % (
                esc(site or L["latest"]), esc(" • ".join(x for x in (story.get("title"), story.get("tagline")) if x)))
    texture = ""
    if style == "documentary":
        texture = '<div class="leak"></div><div class="grain"></div><div class="dust"></div>'
    elif style in ("broadcast", "midnight", "neon") and photo:
        texture = '<div class="grain" style="opacity:.07"></div>'
    onphoto = (bool(photo) and lay in ("title", "headline", "photo", "quote", "outro")) or lay == "map"
    pc = onphoto and bool(photo) and photo.get("fit") == "contain"
    var = ""
    if pc:                                              # where the text starts, beside or below the band photo
        pa = photo.get("w", 1) / float(photo.get("h", 1) or 1)
        var = ("--below:%.1fvmin" % (13 + max(46, min(70, 100 / pa)) + 6) if fmt == "reel"
               else "--beside:%.1fvmin" % (max(56, min(100, 100 * pa)) + 8))
    cls = " ".join(filter(None, [fmt, style, "onphoto" if onphoto else "", "pc" if pc else "", "light" if style in LIGHT else "", "deva" if lang in DEVANAGARI_LANGS else "",
                                 "dim" if lay in ("title", "outro", "quote") and not broadcast else "",
                                 "bars" if style == "documentary" else "", "first" if idx == 0 else "", "last" if idx == n - 1 else ""]))
    st = dict(STYLES[style], dur="%.3f" % dur, p0="%.3f" % p0, p1="%.3f" % p1, grain=GRAIN, kb="kbin")
    cps = max(18.0, len(scene.get("code") or "") / max(1.0, dur - 2.2))
    transition = '<div class="wipe"></div>' if broadcast else '<div class="veil"></div>'
    num = '<div class="num">%d / %d</div>' % (idx + 1, n) if not broadcast else ""
    return ('<!doctype html><html class="%s" style="%s" lang="%s"><head><meta charset="utf-8"><style>%s</style></head><body>'
            '<div class="stage">%s</div>%s%s%s%s%s%s<div class="progress"></div>%s'
            '<script>%s</script></body></html>') % (
        cls, var, esc(lang), CSS.substitute(st), bg, body, furniture, credit, texture, num,
        '<div class="cap"></div>' if captions else "", transition,
        JS.replace("__DUR__", "%.3f" % dur).replace("__CAPS__", json.dumps(captions or [], ensure_ascii=False)).replace("__CPS__", "%.2f" % cps))
''')
print("wrote", os.path.join(APP_DIR, 'vs_scenes.py'))

In [ ]:
# Photos, images (SDXL), music (MusicGen), Kokoro voices, audio mix
# Writes vs_media.py into APP_DIR. Safe to re-run.
import os
APP_DIR = '/kaggle/working/video_app'
os.makedirs(APP_DIR, exist_ok=True)
with open(os.path.join(APP_DIR, 'vs_media.py'), "w", encoding="utf-8") as f:
    f.write(r'''"""Photos, images, narration and music for Video Studio, plus the audio mix. Heavy libraries load on first use.

- Photos: downloaded from the source page, checked, de-duplicated and lightly edited (auto-contrast, sharpen);
  optionally described by a vision LLM (Gemma 3 via Ollama) so the script can match photos to scenes.
- Images: Stable Diffusion XL (diffusers), only for non-news topics without usable photos.
- Narration: Kokoro-82M here; Nepali / Hindi voices live in vs_tts.py.
- Music: MusicGen (transformers), ~30 s generated once, looped under the whole video.
- mix(): narration track + music ducked under the voice -> one 44.1 kHz WAV for both formats.
"""
import os
import threading
import wave

import numpy as np

STYLE_SUFFIX = {
    "midnight": "cinematic digital illustration, deep blue and magenta lighting, soft glow, high detail",
    "paper": "warm editorial illustration, paper texture, muted earthy colors, gouache",
    "neon": "cyberpunk illustration, glowing cyan and magenta neon, dark background, high contrast",
    "swiss": "minimal flat geometric illustration, bold red black and white, clean shapes, bauhaus poster",
}
NEGATIVE = "text, letters, words, watermark, logo, signature, caption, ui, blurry, lowres, deformed, extra fingers"
OUT_RATE = 44100
MUSIC_LEVELS = {"tragic": (0.10, 0.035), "serious": (0.14, 0.05)}     # (music level, level under the voice)


def read_wav(path):
    with wave.open(path) as w:
        rate, ch, n = w.getframerate(), w.getnchannels(), w.getnframes()
        data = np.frombuffer(w.readframes(n), dtype=np.int16).astype(np.float32) / 32768.0
    if ch > 1:
        data = data.reshape(-1, ch).mean(axis=1)
    return data, rate


def write_wav(path, data, rate):
    data = np.clip(np.asarray(data, dtype=np.float32), -1, 1)
    tmp = path + ".tmp.wav"
    with wave.open(tmp, "wb") as w:
        w.setnchannels(1)
        w.setsampwidth(2)
        w.setframerate(rate)
        w.writeframes((data * 32767).astype(np.int16).tobytes())
    os.replace(tmp, path)


def resample(x, src, dst):
    if src == dst or not len(x):
        return x
    n = int(round(len(x) * dst / src))
    return np.interp(np.linspace(0, len(x) - 1, n), np.arange(len(x)), x).astype(np.float32)


def duration(path):
    with wave.open(path) as w:
        return w.getnframes() / float(w.getframerate())


class Media:
    def __init__(self, image_model, image_steps=25, image_device="cuda:0", music_model="", music_device="cuda:0",
                 tts_device="cpu", log=None):
        self.image_model, self.image_steps, self.image_device = image_model, int(image_steps), image_device
        self.music_model, self.music_device, self.tts_device = music_model, music_device, tts_device
        self.log = log
        self.lock = threading.Lock()
        self.m = {}

    # ------------------------------------------------------------------ images
    def _pipe(self):
        if "sd" not in self.m:
            import torch
            from diffusers import AutoPipelineForText2Image
            kw = {"torch_dtype": torch.float16}
            if "xl" in self.image_model.lower() and "turbo" not in self.image_model.lower():
                from diffusers import AutoencoderKL
                kw["vae"] = AutoencoderKL.from_pretrained("madebyollin/sdxl-vae-fp16-fix", torch_dtype=torch.float16)
            try:
                pipe = AutoPipelineForText2Image.from_pretrained(self.image_model, variant="fp16", **kw)
            except Exception:
                pipe = AutoPipelineForText2Image.from_pretrained(self.image_model, **kw)
            pipe.set_progress_bar_config(disable=True)
            self.m["sd"] = pipe.to(self.image_device)
            if self.log:
                self.log.info("loaded %s on %s", self.image_model, self.image_device)
        return self.m["sd"]

    def image(self, prompt, style, path, seed=0):
        import torch
        with self.lock:
            pipe = self._pipe()
            full = "%s, %s" % (prompt, STYLE_SUFFIX.get(style, ""))
            gen = torch.Generator(self.image_device).manual_seed(int(seed))
            turbo = "turbo" in self.image_model.lower() or "schnell" in self.image_model.lower()
            out = pipe(prompt=full, negative_prompt=None if turbo else NEGATIVE, num_inference_steps=self.image_steps,
                       guidance_scale=0.0 if turbo else 6.0, width=1024, height=1024, generator=gen).images[0]
        out.save(path + ".tmp.png")
        os.replace(path + ".tmp.png", path)
        return path

    # ------------------------------------------------------------------ narration
    def tts(self, text, voice, path, speed=1.0):
        """Writes a 24 kHz WAV and returns its duration in seconds."""
        lang = voice[0]
        with self.lock:
            key = "tts_" + lang
            if key not in self.m:
                from kokoro import KPipeline
                try:
                    self.m[key] = KPipeline(lang_code=lang, device=self.tts_device)
                except TypeError:
                    self.m[key] = KPipeline(lang_code=lang)
            parts = []
            for r in self.m[key](text, voice=voice, speed=speed):
                audio = r.audio if hasattr(r, "audio") else r[2]
                if audio is not None:
                    parts.append(audio.detach().cpu().numpy() if hasattr(audio, "detach") else np.asarray(audio))
        if not parts:
            raise RuntimeError("no audio produced")
        data = np.concatenate(parts).astype(np.float32)
        write_wav(path, data, 24000)
        return len(data) / 24000.0

    # ------------------------------------------------------------------ music
    def music(self, prompt, path, seconds=30):
        name = self.music_model
        if name.lower().startswith("ace"):
            try:
                return self._ace(prompt, path, seconds)
            except Exception as e:
                name = os.environ.get("MUSIC_FALLBACK", "")
                if not name:
                    raise
                if self.log:
                    self.log.warning("ACE-Step failed (%s); using %s", e, name)
        with self.lock:
            if "mg" not in self.m:
                import torch
                from transformers import AutoProcessor, MusicgenForConditionalGeneration
                proc = AutoProcessor.from_pretrained(name)
                model = MusicgenForConditionalGeneration.from_pretrained(name, torch_dtype=torch.float16
                                                                         if self.music_device.startswith("cuda") else torch.float32)
                self.m["mg"] = (proc, model.to(self.music_device))
            proc, model = self.m["mg"]
            inputs = proc(text=[prompt + ", instrumental, no vocals, loopable"], padding=True, return_tensors="pt").to(self.music_device)
            out = model.generate(**inputs, max_new_tokens=int(min(30, seconds) * 50), do_sample=True, guidance_scale=3.0)
            rate = model.config.audio_encoder.sampling_rate
            data = out[0, 0].float().cpu().numpy()
        write_wav(path, data / (np.abs(data).max() + 1e-6) * 0.9, rate)
        return path

    def _ace(self, prompt, path, seconds):
        """ACE-Step (Apache 2.0, commercial use OK) in its own virtualenv; see vs_ace_worker.py."""
        import subprocess
        py = os.environ.get("ACE_PYTHON", "")
        if not py or not os.path.exists(py):
            raise RuntimeError("ACE-Step is not installed (see the notebook's install cell)")
        dev = int(self.music_device.split(":")[1]) if ":" in self.music_device else 0
        log = os.environ.get("ACE_LOG", "/kaggle/working/logs/ace_step.log")
        with self.lock, open(log, "a") as lf:
            p = subprocess.run([py, os.path.join(os.path.dirname(os.path.abspath(__file__)), "vs_ace_worker.py"), "--out", path,
                                "--prompt", prompt, "--seconds", str(int(seconds)), "--device", str(dev)],
                               stdout=lf, stderr=subprocess.STDOUT, timeout=1800)
        if p.returncode != 0 or not os.path.exists(path):
            raise RuntimeError("ACE-Step failed; see " + log)
        return path

    def unload(self, *names):
        for n in names or list(self.m):
            self.m.pop(n, None)
        try:
            import gc
            import torch
            gc.collect()
            torch.cuda.empty_cache()
        except Exception:
            pass


def tempo(src, dst, factor):
    """Speeds speech up by factor without changing its pitch (ffmpeg atempo); returns the new duration."""
    import subprocess
    subprocess.run(["ffmpeg", "-v", "error", "-y", "-i", src, "-filter:a", "atempo=%.3f" % factor, dst],
                   check=True, capture_output=True, timeout=120)
    return duration(dst)


def mix(total, narrations, music_path, out_path, tone="neutral", cuts=None):
    """narrations: [(start_seconds, wav_path)]. Music loops with crossfades and dips under the voice.
    cuts: scene-change times for soft transition whooshes (None = no sound effects)."""
    n = int(total * OUT_RATE)
    voice = np.zeros(n, np.float32)
    for start, p in narrations:
        x, r = read_wav(p)
        x = resample(x, r, OUT_RATE)
        i = int(start * OUT_RATE)
        x = x[:max(0, n - i)]
        voice[i:i + len(x)] += x
    music_gain, duck_gain = MUSIC_LEVELS.get(tone, (0.22, 0.07))
    out = voice.copy()
    if music_path and os.path.exists(music_path):
        m, r = read_wav(music_path)
        m = resample(m, r, OUT_RATE)
        xf = min(int(1.5 * OUT_RATE), len(m) // 4)
        if len(m) > 2 * xf:
            loop = m.copy()
            while len(loop) < n:                         # append with a crossfade so the loop seam is smooth
                fade = np.linspace(0, 1, xf, dtype=np.float32)
                loop[-xf:] = loop[-xf:] * (1 - fade) + m[:xf] * fade
                loop = np.concatenate([loop, m[xf:]])
            m = loop[:n]
        else:
            m = np.resize(m, n)
        env = np.abs(voice)
        win = int(0.35 * OUT_RATE)
        env = np.convolve(env, np.ones(win, np.float32) / win, mode="same")
        talking = np.clip(env / 0.02, 0, 1)
        gain = music_gain - (music_gain - duck_gain) * talking
        fade_in, fade_out = int(1.0 * OUT_RATE), int(2.5 * OUT_RATE)
        gain[:fade_in] *= np.linspace(0, 1, len(gain[:fade_in]))
        gain[-fade_out:] *= np.linspace(1, 0, len(gain[-fade_out:]))
        out = out + m * gain
    if cuts is not None:
        import vs_motion
        out = out + vs_motion.sfx_track(total, cuts, OUT_RATE, tone)[:len(out)]
    peak = np.abs(out).max()
    if peak > 0.97:
        out = out * (0.97 / peak)
    write_wav(out_path, out, OUT_RATE)
    return out_path


# ====================================================================== photos from the source
def _ahash(im):
    g = im.convert("L").resize((9, 8))
    px = list(g.getdata())
    return sum(1 << i for i in range(64) if px[(i // 8) * 9 + i % 8] > px[(i // 8) * 9 + i % 8 + 1])


def fetch_photos(images, out_dir, limit=8, referer=None, log=None):
    """Downloads candidate photos, keeps real photos (>= 480 px, sane shape, not duplicates), edits them lightly.
    Returns [{"path", "w", "h", "caption", "alt", "url"}] in the source's order of importance."""
    import io
    import urllib.request
    from PIL import Image, ImageFilter, ImageOps
    os.makedirs(out_dir, exist_ok=True)
    kept, hashes = [], []
    for c in images:
        if len(kept) >= limit:
            break
        try:
            if c.get("bytes"):
                data = c["bytes"]
            else:
                req = urllib.request.Request(c["url"], headers={"User-Agent": "Mozilla/5.0 (X11; Linux x86_64) Chrome/126 Safari/537.36",
                                                                "Referer": referer or c["url"], "Accept": "image/*,*/*;q=0.5"})
                with urllib.request.urlopen(req, timeout=25) as r:
                    data = r.read(20 << 20)
            im = Image.open(io.BytesIO(data))
            im = ImageOps.exif_transpose(im).convert("RGB")
        except Exception as e:
            if log:
                log.info("photo skipped (%s): %s", str(c.get("url"))[:120], e)
            continue
        w, h = im.size
        if min(w, h) < 480 or max(w, h) / min(w, h) > 3.2:
            continue
        hsh = _ahash(im)
        if any(bin(hsh ^ x).count("1") <= 6 for x in hashes):
            if log:
                log.info("duplicate photo skipped: %s", str(c.get("url"))[:120])
            continue                                    # same picture at another size or crop
        hashes.append(hsh)
        if max(w, h) > 2400:
            im.thumbnail((2400, 2400), Image.LANCZOS)
        im = ImageOps.autocontrast(im, cutoff=0.5, preserve_tone=True) if "preserve_tone" in ImageOps.autocontrast.__code__.co_varnames \
            else ImageOps.autocontrast(im, cutoff=0.5)
        im = im.filter(ImageFilter.UnsharpMask(radius=1.2, percent=60, threshold=3))
        path = os.path.join(out_dir, "photo_%02d.jpg" % len(kept))
        im.save(path, "JPEG", quality=92)
        kept.append({"path": path, "w": im.size[0], "h": im.size[1], "caption": c.get("caption") or "",
                     "alt": c.get("alt") or "", "url": c.get("url")})
    return kept


SCREEN_PROMPT = """This picture was found on a web page about: {topic}

Return ONLY a JSON object:
{{"description": "one short factual English sentence: who or what is shown and where; no guessing names",
  "kind": one of "news photo", "advertisement", "product", "stock graphic", "logo or text", "portrait", "other",
  "relevance": 0 to 10, how well it fits the page's topic (10 = clearly shows this story)}}

Advertisements and products include food, kitchens, appliances, phones, fashion, banks, offers and prices."""
DROP_KINDS = {"advertisement", "product", "logo or text"}


def describe_photos(photos, ollama_url, model, keep_alive="10m", log=None, topic="", keep=8):
    """Asks a vision model (Gemma 3) to describe each photo and to spot ads and off-topic pictures, which are
    dropped. Returns the photos to use; best effort (without a vision model every photo is kept)."""
    import base64
    import io
    import json as _json
    import re as _re
    import urllib.request
    from PIL import Image
    kept = []
    for i, p in enumerate(photos):
        try:
            im = Image.open(p["path"])
            im.thumbnail((640, 640))
            buf = io.BytesIO()
            im.save(buf, "JPEG", quality=85)
            body = _json.dumps({"model": model, "stream": False, "keep_alive": keep_alive, "format": "json",
                                "options": {"temperature": 0.1},
                                "messages": [{"role": "user", "images": [base64.b64encode(buf.getvalue()).decode()],
                                              "content": SCREEN_PROMPT.format(topic=topic[:300] or "(unknown)")}]}).encode()
            req = urllib.request.Request(ollama_url.rstrip("/") + "/api/chat", data=body, headers={"Content-Type": "application/json"})
            with urllib.request.urlopen(req, timeout=120) as r:
                content = _json.loads(r.read())["message"]["content"]
            m = _re.search(r"\{.*\}", content, _re.S)
            try:
                v = _json.loads(m.group(0)) if m else {}
            except ValueError:
                v = {}
            if not isinstance(v, dict) or not v:
                v = {"description": content}            # a plain-text answer: keep it as the description
        except Exception as e:
            if log:
                log.info("photo check skipped: %s", e)
            return (kept + photos[i:])[:keep]          # the model cannot see images: keep the rest unchecked
        kind = str(v.get("kind") or "").lower().strip()
        try:
            rel = float(v.get("relevance", 5))
        except (TypeError, ValueError):
            rel = 5.0
        p["description"] = str(v.get("description") or "").strip()[:240]
        p["kind"], p["relevance"] = kind, rel
        if kind in DROP_KINDS or rel < 3:
            if log:
                log.info("photo dropped (%s, relevance %s): %s", kind or "?", rel, str(p.get("url"))[:120])
            continue
        kept.append(p)
        if len(kept) >= keep:
            break
    return sorted(kept, key=lambda p: p.get("kind") == "stock graphic")   # real photos lead (the headline uses the first)
''')
print("wrote", os.path.join(APP_DIR, 'vs_media.py'))

In [ ]:
# Voices by language: Indic Parler-TTS, Svara-TTS, Piper, Kokoro, online voices
# Writes vs_tts.py into APP_DIR. Safe to re-run.
import os
APP_DIR = '/kaggle/working/video_app'
os.makedirs(APP_DIR, exist_ok=True)
with open(os.path.join(APP_DIR, 'vs_tts.py'), "w", encoding="utf-8") as f:
    f.write(r'''"""Narration voices for Video Studio, by language.

- Kokoro-82M (Apache 2.0): English and a few European languages, in the server process (CPU is fine).
- Indic Parler-TTS (Apache 2.0): Nepali, Hindi and other Indian languages, in vs_parler_worker.py (its own
  virtualenv and process, on the media GPU), with a named speaker so the voice stays the same in every scene.
- Piper (MIT; voice from the OpenSLR Nepali corpus): a light CPU Nepali voice, used when Parler is unavailable.
- Svara-TTS v1 (Apache 2.0, Kenpath): male and female voices for 19 Indic languages including Nepali; a Llama-3.2-3B
  speech model whose SNAC codes are decoded to 24 kHz audio, run here with plain transformers on the media GPU.
- Microsoft Edge neural voices (online, opt-in with ONLINE_VOICES=1): very natural Nepali / Hindi / English voices
  through the edge-tts client. Not open source, and text is sent to Microsoft.
"auto" picks the best installed voice for the video's language.
"""
import json
import os
import re
import shutil
import subprocess
import sys
import time
import urllib.request

import studio_http as K

PARLER_PYTHON = os.environ.get("PARLER_PYTHON", "")
PARLER_PORT = int(os.environ.get("PARLER_PORT", "7862"))
PIPER_DIR = os.environ.get("PIPER_DIR", "/kaggle/working/piper")
SVARA_MODEL = os.environ.get("SVARA_MODEL", "")                 # e.g. kenpath/svara-tts-v1 ("" = off)
ONLINE_VOICES = os.environ.get("ONLINE_VOICES", "0") == "1"
HERE = os.path.dirname(os.path.abspath(__file__))

VOICES = {
    "ne_amrita": {"engine": "parler", "lang": "ne", "speaker": "Amrita", "label": "नेपाली · Amrita (female)"},
    "ne_piper": {"engine": "piper", "lang": "ne", "model": "ne_NP-google-medium", "label": "नेपाली · Piper (light)"},
    "ne_svara_f": {"engine": "svara", "lang": "ne", "speaker": "Nepali (Female)", "label": "नेपाली · Svara (female)"},
    "ne_svara_m": {"engine": "svara", "lang": "ne", "speaker": "Nepali (Male)", "label": "नेपाली · Svara (male)"},
    "ne_hemkala": {"engine": "edge", "lang": "ne", "speaker": "ne-NP-HemkalaNeural", "label": "नेपाली · Hemkala (female, online)"},
    "ne_sagar": {"engine": "edge", "lang": "ne", "speaker": "ne-NP-SagarNeural", "label": "नेपाली · Sagar (male, online)"},
    "hi_divya": {"engine": "parler", "lang": "hi", "speaker": "Divya", "label": "हिन्दी · Divya (female)"},
    "hi_svara_f": {"engine": "svara", "lang": "hi", "speaker": "Hindi (Female)", "label": "हिन्दी · Svara (female)"},
    "hi_svara_m": {"engine": "svara", "lang": "hi", "speaker": "Hindi (Male)", "label": "हिन्दी · Svara (male)"},
    "hi_swara": {"engine": "edge", "lang": "hi", "speaker": "hi-IN-SwaraNeural", "label": "हिन्दी · Swara (female, online)"},
    "hi_madhur": {"engine": "edge", "lang": "hi", "speaker": "hi-IN-MadhurNeural", "label": "हिन्दी · Madhur (male, online)"},
    "en_aria": {"engine": "edge", "lang": "en", "speaker": "en-US-AriaNeural", "label": "English (US) · Aria (female, online)"},
    "en_guy": {"engine": "edge", "lang": "en", "speaker": "en-US-GuyNeural", "label": "English (US) · Guy (male, online)"},
    "hi_rohit": {"engine": "parler", "lang": "hi", "speaker": "Rohit", "label": "हिन्दी · Rohit (male)"},
    "af_heart": {"engine": "kokoro", "lang": "en", "label": "English (US) · female, warm"},
    "af_bella": {"engine": "kokoro", "lang": "en", "label": "English (US) · female, bright"},
    "am_michael": {"engine": "kokoro", "lang": "en", "label": "English (US) · male"},
    "am_fenrir": {"engine": "kokoro", "lang": "en", "label": "English (US) · male, deep"},
    "bf_emma": {"engine": "kokoro", "lang": "en", "label": "English (UK) · female"},
    "bm_george": {"engine": "kokoro", "lang": "en", "label": "English (UK) · male"},
    "ef_dora": {"engine": "kokoro", "lang": "es", "label": "Español · female"},
    "ff_siwis": {"engine": "kokoro", "lang": "fr", "label": "Français · female"},
    "if_sara": {"engine": "kokoro", "lang": "it", "label": "Italiano · female"},
    "pf_dora": {"engine": "kokoro", "lang": "pt", "label": "Português (BR) · female"},
}
AUTO = {"en": ["af_heart", "en_aria"], "ne": ["ne_amrita", "ne_svara_f", "ne_hemkala", "ne_piper"],
        "hi": ["hi_divya", "hi_svara_f", "hi_swara"], "es": ["ef_dora"], "fr": ["ff_siwis"],
        "it": ["if_sara"], "pt": ["pf_dora"]}
PARLER_LANGS = {"as", "bn", "brx", "doi", "gu", "kn", "kok", "mai", "ml", "mni", "mr", "or", "sa", "sat", "sd", "ta", "te", "ur"}
STYLE = {"tragic": "in a calm, gentle and serious tone", "serious": "in a calm, serious and measured tone",
         "upbeat": "in a bright, friendly and energetic tone", "inspiring": "in a warm, confident tone"}


def parler_ok():
    return bool(PARLER_PYTHON) and os.path.exists(PARLER_PYTHON)


def piper_ok(model="ne_NP-google-medium"):
    import importlib.util
    return (importlib.util.find_spec("piper") is not None or bool(shutil.which("piper"))) and \
        os.path.exists(os.path.join(PIPER_DIR, model + ".onnx"))


def svara_ok():
    import importlib.util
    return bool(SVARA_MODEL) and importlib.util.find_spec("snac") is not None


def edge_ok():
    return ONLINE_VOICES and bool(shutil.which("edge-tts"))


def kokoro_ok():
    import importlib.util
    return importlib.util.find_spec("kokoro") is not None


def available(voice):
    v = VOICES.get(voice)
    if not v:
        return False
    return {"parler": parler_ok, "piper": lambda: piper_ok(v.get("model")), "kokoro": kokoro_ok, "svara": svara_ok,
            "edge": edge_ok}[v["engine"]]()


def pick(voice, lang):
    """Resolves "auto" (or an unavailable voice) to a usable voice for the language; None means captions only."""
    if voice and voice != "auto" and voice in VOICES and available(voice):
        return voice
    for v in AUTO.get(lang, []):
        if available(v):
            return v
    if lang in PARLER_LANGS and parler_ok():
        return "indic:" + lang
    return None


def voice_options():
    out = {"auto": "Auto (match the video's language)"}
    for k, v in VOICES.items():
        if available(k):
            out[k] = v["label"]
    return out


# ---------------------------------------------------------------------- Parler worker
def _parler_health():
    try:
        with urllib.request.urlopen("http://127.0.0.1:%d/health" % PARLER_PORT, timeout=3) as r:
            return json.loads(r.read())
    except Exception:
        return None


def ensure_parler(device, log=None, wait=900):
    h = _parler_health()
    if h and h.get("state") == "ready":
        return
    if not h or h.get("state") == "error":
        if not parler_ok():
            raise RuntimeError("Indic Parler-TTS is not installed (see the notebook's install cell)")
        K.spawn("vs-parler", [PARLER_PYTHON, os.path.join(HERE, "vs_parler_worker.py"), "--port", str(PARLER_PORT)],
                os.environ.get("PARLER_LOG", "/kaggle/working/logs/parler.log"),
                env=dict(os.environ, PARLER_DEVICE=device, PYTHONUNBUFFERED="1"), cwd=HERE)
        if log:
            log.info("starting the Parler TTS worker on %s", device)
    t = time.time()
    while time.time() - t < wait:
        h = _parler_health()
        if h and h.get("state") == "ready":
            return
        if h and h.get("state") == "error":
            raise RuntimeError("Parler TTS failed to load: %s" % h.get("error"))
        if not K.process_alive("vs-parler"):
            raise RuntimeError("the Parler TTS worker stopped; see /kaggle/working/logs/parler.log")
        time.sleep(3)
    raise RuntimeError("Parler TTS took too long to load")


def _parler(text, speaker, tone, path, device, log, lang=None):
    ensure_parler(device, log)
    who = ("%s speaks" % speaker) if speaker else ("A clear female speaker speaks %s" % ("in " + lang if lang else ""))
    desc = "%s %s, at a moderate pace, with a very clear, close recording and no background noise." % (
        who, STYLE.get(tone, "in a clear, natural, expressive tone"))
    req = urllib.request.Request("http://127.0.0.1:%d/tts" % PARLER_PORT, headers={"Content-Type": "application/json"},
                                 data=json.dumps({"text": text, "description": desc, "path": path, "seed": 11}).encode())
    with urllib.request.urlopen(req, timeout=900) as r:
        return json.loads(r.read())["duration"]


def _piper(text, model, path):
    onnx = os.path.join(PIPER_DIR, model + ".onnx")
    cmds = [[sys.executable, "-m", "piper", "-m", onnx, "-f", path]]
    if shutil.which("piper"):
        cmds.append(["piper", "-m", onnx, "-f", path])
    err = ""
    for cmd in cmds:
        p = subprocess.run(cmd, input=re.sub(r"\s+", " ", text), capture_output=True, text=True, timeout=300)
        if p.returncode == 0 and os.path.exists(path):
            import vs_media
            return vs_media.duration(path)
        err = (p.stderr or p.stdout)[-400:]
    raise RuntimeError("piper failed: " + err)


# ---------------------------------------------------------------------- Svara (transformers, in process)
_SVARA = {}
SOS, EOS_SPEECH, SOH, EOH, SOAI, EOT, AUDIO, BOS = 128257, 128258, 128259, 128260, 128261, 128009, 156939, 128000
CODE_BASE = 128266


def _svara_load(device, log):
    if "m" not in _SVARA:
        import torch
        from snac import SNAC
        from transformers import AutoModelForCausalLM, AutoTokenizer
        tok = AutoTokenizer.from_pretrained(SVARA_MODEL)
        model = AutoModelForCausalLM.from_pretrained(SVARA_MODEL, torch_dtype=torch.float16).to(device).eval()
        snac = SNAC.from_pretrained("hubertsiuzdak/snac_24khz").eval().to(device)
        _SVARA.update(m=model, tok=tok, snac=snac)
        if log:
            log.info("loaded %s on %s", SVARA_MODEL, device)
    return _SVARA["m"], _SVARA["tok"], _SVARA["snac"]


def svara_unload():
    _SVARA.clear()


def _svara_codes(ids):
    """Generated token ids -> the three SNAC code streams (7 tokens per frame: c0, c1, c2, c2, c1, c2, c2)."""
    codes = [t - CODE_BASE for t in ids if t >= CODE_BASE]
    codes = [c - (k % 7) * 4096 for k, c in enumerate(codes)]
    n = len(codes) // 7
    if not n or any(c < 0 or c > 4095 for c in codes[:n * 7]):
        raise RuntimeError("Svara produced invalid audio codes")
    f = [codes[7 * i:7 * i + 7] for i in range(n)]
    return ([x[0] for x in f], [y for x in f for y in (x[1], x[4])], [y for x in f for y in (x[2], x[3], x[5], x[6])])


def _svara(text, speaker, path, device, log):
    import numpy as np
    import torch
    import vs_media
    model, tok, snac = _svara_load(device, log)
    parts = []
    for sent in [x for x in re.split(r"(?<=[.!?।॥])\s+", text.strip()) if x.strip()]:
        ids = tok("%s: %s" % (speaker, sent), add_special_tokens=False).input_ids
        prompt = [BOS, SOH, AUDIO] + ids + [EOH, EOT, SOAI, SOS]
        x = torch.tensor([prompt], device=device)
        with torch.no_grad():
            out = model.generate(x, attention_mask=torch.ones_like(x), max_new_tokens=min(4000, 120 + 26 * len(sent)),
                                 do_sample=True, temperature=0.75, top_p=0.9, repetition_penalty=1.1,
                                 eos_token_id=[EOS_SPEECH], pad_token_id=EOS_SPEECH)
        c0, c1, c2 = _svara_codes(out[0, len(prompt):].tolist())
        with torch.no_grad():
            audio = snac.decode([torch.tensor([c], device=device) for c in (c0, c1, c2)])
        parts += [audio[0, 0].float().cpu().numpy(), np.zeros(int(0.14 * 24000), np.float32)]
    if not parts:
        raise RuntimeError("no text to speak")
    data = np.concatenate(parts)
    vs_media.write_wav(path, data / (np.abs(data).max() + 1e-6) * 0.9, 24000)
    return len(data) / 24000.0


# ---------------------------------------------------------------------- Edge (online, opt-in)
def _edge(text, speaker, path, tone):
    import vs_media
    mp3 = path[:-4] + ".mp3"
    rate = "-6%" if tone in ("tragic", "serious") else "+0%"
    p = subprocess.run(["edge-tts", "--voice", speaker, "--rate=" + rate, "--text", re.sub(r"\s+", " ", text),
                        "--write-media", mp3], capture_output=True, text=True, timeout=180)
    if p.returncode != 0 or not os.path.exists(mp3):
        raise RuntimeError("online voice failed: " + (p.stderr or p.stdout)[-300:])
    subprocess.run(["ffmpeg", "-v", "error", "-y", "-i", mp3, "-ac", "1", "-ar", "24000", path], check=True, timeout=120)
    os.remove(mp3)
    return vs_media.duration(path)


def speak(text, voice, path, media, tone="neutral", device="cuda:0", log=None):
    """Writes narration to path and returns its duration in seconds."""
    if voice.startswith("indic:"):
        return _parler(text, None, tone, path, device, log, voice.split(":", 1)[1])
    v = VOICES[voice]
    if v["engine"] == "parler":
        try:
            return _parler(text, v["speaker"], tone, path, device, log)
        except Exception as e:
            if v["lang"] == "ne" and piper_ok():
                if log:
                    log.warning("Parler failed (%s); using Piper for Nepali", e)
                return _piper(text, "ne_NP-google-medium", path)
            raise
    if v["engine"] == "piper":
        return _piper(text, v["model"], path)
    if v["engine"] in ("svara", "edge"):
        try:
            return _svara(text, v["speaker"], path, device, log) if v["engine"] == "svara" else _edge(text, v["speaker"], path, tone)
        except Exception as e:
            fb = next((x for x in AUTO.get(v["lang"], []) if VOICES[x]["engine"] not in ("svara", "edge") and available(x)), None)
            if not fb:
                raise
            if log:
                log.warning("%s failed (%s); using %s", voice, e, fb)
            return speak(text, fb, path, media, tone, device, log)
    return media.tts(text, voice, path, speed=0.95 if tone in ("tragic", "serious") else 1.0)
''')
print("wrote", os.path.join(APP_DIR, 'vs_tts.py'))

In [ ]:
# Motion: depth parallax, subject cut-outs, location maps, transition sounds, AI clips
# Writes vs_motion.py into APP_DIR. Safe to re-run.
import os
APP_DIR = '/kaggle/working/video_app'
os.makedirs(APP_DIR, exist_ok=True)
with open(os.path.join(APP_DIR, 'vs_motion.py'), "w", encoding="utf-8") as f:
    f.write(r'''"""Motion for still photos, location maps, sound effects and (optional) AI image-to-video for Video Studio.

Everything here is best effort: when a model or service is missing the video still renders, just with less motion.

- depth(): Depth Anything V2 Small (Apache 2.0) -> a depth map per photo. The scene page then moves near and far
  parts of the real photo at different speeds (2.5D parallax, a WebGL shader in vs_scenes.py). Nothing is invented.
- cutout(): BiRefNet (MIT) separates the main subject so it can lift off the background in headline scenes.
- place_map(): geocodes the story's place (OpenStreetMap Nominatim) and snapshots a country view and a local view
  with MapLibre GL + OpenFreeMap tiles in headless Chromium; the map scene zooms between them with a pin.
- whoosh() / hit(): transition sounds synthesised in code (no sample libraries, no licences).
- animate(): image-to-video (LTX-Video or Wan 2.2 TI2V via diffusers) for non-news scenes only, off by default.
"""
import json
import os
import threading
import urllib.parse
import urllib.request

import numpy as np

DEPTH_MODEL = "depth-anything/Depth-Anything-V2-Small-hf"
CUTOUT_MODEL = "ZhengPeng7/BiRefNet_lite"
UA = "agentic-notebooks-video-studio/1.0 (https://github.com/echoghimire/agentic-notebooks)"
MAPLIBRE = "https://unpkg.com/maplibre-gl@4.7.1/dist/maplibre-gl"
MAP_STYLES = {"dark": "https://tiles.openfreemap.org/styles/dark", "light": "https://tiles.openfreemap.org/styles/positron",
              "bright": "https://tiles.openfreemap.org/styles/liberty"}
I2V_MODELS = {"ltx": "Lightricks/LTX-Video", "wan": "Wan-AI/Wan2.2-TI2V-5B-Diffusers"}


class Motion:
    def __init__(self, device="cuda:0", log=None):
        self.device, self.log = device, log
        self.lock = threading.Lock()
        self.m = {}

    def _info(self, *a):
        if self.log:
            self.log.info(*a)

    # ------------------------------------------------------------------ depth parallax
    def depth(self, photo_path, out_path):
        """Writes a smoothed 8-bit depth map (bright = near) next to the photo; returns out_path."""
        from PIL import Image, ImageFilter
        with self.lock:
            if "depth" not in self.m:
                import torch
                from transformers import pipeline
                dev = 0 if self.device.startswith("cuda") else -1
                if self.device.startswith("cuda:"):
                    dev = int(self.device.split(":")[1])
                self.m["depth"] = pipeline("depth-estimation", model=DEPTH_MODEL, device=dev,
                                           torch_dtype=torch.float16 if dev >= 0 else torch.float32)
                self._info("loaded %s", DEPTH_MODEL)
            im = Image.open(photo_path).convert("RGB")
            im.thumbnail((1024, 1024))
            d = self.m["depth"](im)["depth"].convert("L")
        d = d.resize(im.size).filter(ImageFilter.GaussianBlur(max(2, im.size[0] // 300)))   # soft edges: no tearing
        a = np.asarray(d, np.float32)
        lo, hi = np.percentile(a, 2), np.percentile(a, 98)
        a = np.clip((a - lo) / max(1.0, hi - lo), 0, 1) * 255
        Image.fromarray(a.astype(np.uint8)).save(out_path)
        return out_path

    # ------------------------------------------------------------------ subject cut-out
    def cutout(self, photo_path, out_path):
        """Writes an RGBA PNG of the main subject; returns (out_path, covered fraction) or (None, frac) when the
        photo has no clear single subject (a crowd, a landscape...)."""
        import torch
        from PIL import Image
        with self.lock:
            if "cut" not in self.m:
                from transformers import AutoModelForImageSegmentation
                model = AutoModelForImageSegmentation.from_pretrained(CUTOUT_MODEL, trust_remote_code=True)
                model = model.to(self.device).eval()
                if self.device.startswith("cuda"):
                    model = model.half()
                self.m["cut"] = model
                self._info("loaded %s", CUTOUT_MODEL)
            model = self.m["cut"]
            im = Image.open(photo_path).convert("RGB")
            x = np.asarray(im.resize((1024, 1024), Image.BILINEAR), np.float32) / 255.0
            x = (x - [0.485, 0.456, 0.406]) / [0.229, 0.224, 0.225]
            t = torch.from_numpy(x.transpose(2, 0, 1)).unsqueeze(0).to(self.device)
            t = t.half() if self.device.startswith("cuda") else t.float()
            with torch.no_grad():
                pred = model(t)[-1].sigmoid().float().cpu()[0, 0].numpy()
        mask = Image.fromarray((pred * 255).astype(np.uint8)).resize(im.size, Image.BILINEAR)
        m = np.asarray(mask, np.float32) / 255
        frac = float((m > 0.5).mean())
        ys, xs = np.where(m > 0.5)
        touches = 0 if not len(xs) else sum([xs.min() < 3, ys.min() < 3, xs.max() > im.size[0] - 4, ys.max() > im.size[1] - 4])
        if not 0.06 <= frac <= 0.55 or touches >= 3:      # no clear subject: leave the photo whole
            return None, frac
        rgba = im.copy()
        rgba.putalpha(mask)
        rgba.save(out_path)
        return out_path, frac

    def unload(self):
        self.m.clear()
        try:
            import gc
            import torch
            gc.collect()
            torch.cuda.empty_cache()
        except Exception:
            pass

    # ------------------------------------------------------------------ image-to-video (opt-in, non-news)
    def animate(self, image_path, prompt, out_dir, model="ltx", seconds=3.0, log=None):
        """Generates a short clip from a still and writes its frames as JPEGs into out_dir (24 fps).
        Returns the number of frames. Heavy: minutes per clip on a T4."""
        import torch
        from PIL import Image
        repo = I2V_MODELS.get(model, model)
        with self.lock:
            key = "i2v:" + repo
            if key not in self.m:
                self.m = {k: v for k, v in self.m.items() if not k.startswith("i2v:")}
                if "wan" in repo.lower():
                    from diffusers import AutoencoderKLWan, WanImageToVideoPipeline
                    vae = AutoencoderKLWan.from_pretrained(repo, subfolder="vae", torch_dtype=torch.float32)
                    pipe = WanImageToVideoPipeline.from_pretrained(repo, vae=vae, torch_dtype=torch.float16)
                else:
                    from diffusers import LTXImageToVideoPipeline
                    pipe = LTXImageToVideoPipeline.from_pretrained(repo, torch_dtype=torch.float16)
                gpu = int(self.device.split(":")[1]) if ":" in self.device else 0
                pipe.enable_model_cpu_offload(gpu_id=gpu)
                if hasattr(pipe, "vae") and hasattr(pipe.vae, "enable_tiling"):
                    pipe.vae.enable_tiling()
                pipe.set_progress_bar_config(disable=True)
                self.m[key] = pipe
                self._info("loaded %s", repo)
            pipe = self.m[key]
            im = Image.open(image_path).convert("RGB")
            wan = "wan" in repo.lower()
            w, h = ((832, 480) if wan else (768, 512))
            if im.size[0] < im.size[1]:
                w, h = h, w
            im = im.resize((w, h), Image.LANCZOS)
            fps = 24
            frames = int(seconds * fps) // 8 * 8 + 1         # both models want 8k+1 frames
            neg = "worst quality, inconsistent motion, blurry, jittery, distorted, text, watermark"
            gen = torch.Generator("cpu").manual_seed(7)
            if wan:
                out = pipe(image=im, prompt=prompt, negative_prompt=neg, width=w, height=h, num_frames=frames,
                           num_inference_steps=30, guidance_scale=5.0, generator=gen).frames[0]
            else:
                out = pipe(image=im, prompt=prompt, negative_prompt=neg, width=w, height=h, num_frames=frames,
                           num_inference_steps=30, guidance_scale=3.0, generator=gen).frames[0]
        os.makedirs(out_dir, exist_ok=True)
        for i, f in enumerate(out):
            (f if hasattr(f, "save") else Image.fromarray((np.asarray(f) * 255).clip(0, 255).astype(np.uint8))).save(
                os.path.join(out_dir, "f%03d.jpg" % i), quality=90)
        return len(out)


# ---------------------------------------------------------------------- location map
def geocode(place, cache_dir=None):
    """(lat, lon, display name) for a place name via OpenStreetMap Nominatim (one request, cached)."""
    key = urllib.parse.quote(place.strip().lower())
    path = os.path.join(cache_dir, "geo_%s.json" % key[:80]) if cache_dir else None
    if path and os.path.exists(path):
        return tuple(json.load(open(path)))
    url = "https://nominatim.openstreetmap.org/search?" + urllib.parse.urlencode(
        {"q": place, "format": "json", "limit": 1, "accept-language": "en"})
    req = urllib.request.Request(url, headers={"User-Agent": UA})
    with urllib.request.urlopen(req, timeout=20) as r:
        res = json.loads(r.read())
    if not res:
        return None
    out = (float(res[0]["lat"]), float(res[0]["lon"]), res[0].get("display_name", place))
    if path:
        os.makedirs(cache_dir, exist_ok=True)
        json.dump(out, open(path, "w"))
    return out


MAP_PAGE = """<!doctype html><html><head><meta charset="utf-8">
<link rel="stylesheet" href="%(lib)s.css"><script src="%(lib)s.js"></script>
<style>html,body,#m{margin:0;width:100%%;height:100%%;background:#111}.maplibregl-ctrl{display:none}</style></head>
<body><div id="m"></div><script>
window.__idle = false;
const map = new maplibregl.Map({container: "m", style: "%(style)s", center: [%(lon)f, %(lat)f], zoom: %(zoom)f,
  interactive: false, attributionControl: false, preserveDrawingBuffer: true, fadeDuration: 0});
map.on("idle", () => { window.__idle = true; });
</script></body></html>"""


MAP_ZOOMS = (("wide", 5.0), ("mid", 6.6), ("close", 8.2))     # each step is 2**1.6 = 3x closer (matches vs_scenes)


def place_map(lat, lon, out_dir, theme="dark", size=(1920, 1920)):
    """Snapshots three views centred on the place, each 3x closer than the last; the map scene zooms through
    them with cross-fades. Returns {"wide", "mid", "close"} paths."""
    from playwright.sync_api import sync_playwright
    os.makedirs(out_dir, exist_ok=True)
    out = {}
    with sync_playwright() as pw:
        b = pw.chromium.launch(executable_path=os.environ.get("CHROMIUM_PATH") or None,
                               args=["--no-sandbox", "--use-angle=swiftshader", "--enable-unsafe-swiftshader",
                                     "--ignore-gpu-blocklist"])
        page = b.new_page(viewport={"width": size[0], "height": size[1]})
        for name, zoom in MAP_ZOOMS:
            html = MAP_PAGE % {"lib": MAPLIBRE, "style": MAP_STYLES.get(theme, MAP_STYLES["dark"]), "lon": lon, "lat": lat,
                               "zoom": zoom}
            f = os.path.join(out_dir, "map_%s.html" % name)
            open(f, "w", encoding="utf-8").write(html)
            page.goto("file://" + f, wait_until="load", timeout=60000)
            page.wait_for_function("window.__idle === true", timeout=90000)
            page.wait_for_timeout(400)
            p = os.path.join(out_dir, "map_%s.jpg" % name)
            page.screenshot(path=p, type="jpeg", quality=90)
            out[name] = p
        b.close()
    return out


# ---------------------------------------------------------------------- transition sounds
def _noise(n, seed):
    return np.random.default_rng(seed).standard_normal(n).astype(np.float32)


def whoosh(rate=44100, dur=0.7, seed=1):
    """A filtered-noise sweep that rises then falls: a soft transition swoosh."""
    n = int(rate * dur)
    x = _noise(n, seed)
    t = np.linspace(0, 1, n, dtype=np.float32)
    cut = 0.02 + 0.25 * np.sin(np.pi * t) ** 2               # one-pole low-pass whose cutoff sweeps up and down
    y = np.zeros(n, np.float32)
    acc = 0.0
    for i in range(n):
        acc += cut[i] * (x[i] - acc)
        y[i] = acc
    env = np.clip(np.sin(np.pi * t), 0, 1) ** 1.5
    y = y * env
    return y / (np.abs(y).max() + 1e-6)


def hit(rate=44100, dur=0.9, seed=2):
    """A low, soft impact (sine drop plus a little noise) for headline reveals."""
    n = int(rate * dur)
    t = np.arange(n, dtype=np.float32) / rate
    f = 90 * np.exp(-t * 5) + 45
    y = np.sin(2 * np.pi * np.cumsum(f) / rate) * np.exp(-t * 4.5)
    y += 0.15 * _noise(n, seed) * np.exp(-t * 30)
    return (y / (np.abs(y).max() + 1e-6)).astype(np.float32)


def sfx_track(total, cuts, rate=44100, tone="neutral"):
    """Whooshes centred on each scene change (seconds) and a soft hit at the start, as one track."""
    level = {"tragic": 0.10, "serious": 0.14}.get(tone, 0.22)
    out = np.zeros(int(total * rate), np.float32)
    w = whoosh(rate)
    for k, c in enumerate(cuts):
        i = int((c - 0.35) * rate)
        seg = w if k % 2 == 0 else w[::-1]                 # alternate so repeated cuts do not sound identical
        if 0 <= i < len(out):
            out[i:i + len(seg)] += seg[:len(out) - i] * level
    hh = hit(rate)
    out[:len(hh)] += hh[:len(out)] * level * 0.8
    return out


# ---------------------------------------------------------------------- helpers
def frames_in(d):
    return sorted(os.path.join(d, f) for f in os.listdir(d) if f.endswith(".jpg")) if os.path.isdir(d) else []
''')
print("wrote", os.path.join(APP_DIR, 'vs_motion.py'))

In [ ]:
# Indic Parler-TTS worker (runs in its own virtualenv)
# Writes vs_parler_worker.py into APP_DIR. Safe to re-run.
import os
APP_DIR = '/kaggle/working/video_app'
os.makedirs(APP_DIR, exist_ok=True)
with open(os.path.join(APP_DIR, 'vs_parler_worker.py'), "w", encoding="utf-8") as f:
    f.write(r'''"""Indic Parler-TTS worker (Nepali, Hindi and 19 more Indian languages; Apache 2.0).

parler-tts pins its own transformers version, so this runs in a separate virtualenv that shares Kaggle's
PyTorch (created by the notebook's install cell) and is started by vs_tts.py:
    <venv>/bin/python vs_parler_worker.py --port 7862
Listens on 127.0.0.1 only. GET /health -> {"state": "loading"|"ready"|"error"};
POST /tts {"text", "description", "path", "seed"} -> {"duration"} (writes a 16-bit WAV to path).
Long text is spoken sentence by sentence (Parler is best below ~25 s per call) with short pauses between.
"""
import argparse
import json
import os
import re
import threading
import time
import traceback
import wave
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

MODEL = os.environ.get("PARLER_MODEL", "ai4bharat/indic-parler-tts")
DEVICE = os.environ.get("PARLER_DEVICE", "cuda:0")
STATE = {"state": "loading", "error": None, "model": MODEL}
M = {}
LOCK = threading.Lock()


def load():
    try:
        import torch
        from parler_tts import ParlerTTSForConditionalGeneration
        from transformers import AutoTokenizer
        dev = DEVICE if torch.cuda.is_available() else "cpu"
        model = ParlerTTSForConditionalGeneration.from_pretrained(
            MODEL, torch_dtype=torch.float16 if dev.startswith("cuda") else torch.float32).to(dev)
        M.update(model=model, tok=AutoTokenizer.from_pretrained(MODEL),
                 dtok=AutoTokenizer.from_pretrained(model.config.text_encoder._name_or_path), dev=dev,
                 rate=model.config.sampling_rate)
        STATE.update(state="ready", device=dev)
        print("parler ready on", dev, flush=True)
    except Exception as e:
        traceback.print_exc()
        STATE.update(state="error", error="%s: %s" % (type(e).__name__, str(e)[:400]))


def split(text, limit=180):
    parts = [p.strip() for p in re.split(r"(?<=[.!?।॥])\s+", text) if p.strip()]
    out = []
    for p in parts:
        while len(p) > limit:
            cut = p.rfind(" ", 0, limit)
            cut = cut if cut > limit // 2 else limit
            out.append(p[:cut].strip())
            p = p[cut:].strip()
        if p:
            out.append(p)
    return out or [text]


def tts(b):
    import numpy as np
    import torch
    text, desc, path = str(b["text"]), str(b["description"]), b["path"]
    model, tok, dtok, dev = M["model"], M["tok"], M["dtok"], M["dev"]
    d = dtok(desc, return_tensors="pt").to(dev)
    pieces = []
    with LOCK:
        for i, sent in enumerate(split(text)):
            torch.manual_seed(int(b.get("seed", 7)) + i)          # same voice and pacing every time
            p = tok(sent, return_tensors="pt").to(dev)
            with torch.inference_mode():
                gen = model.generate(input_ids=d.input_ids, attention_mask=d.attention_mask,
                                     prompt_input_ids=p.input_ids, prompt_attention_mask=p.attention_mask)
            a = gen.to(torch.float32).cpu().numpy().squeeze()
            pieces += [a, np.zeros(int(0.18 * M["rate"]), np.float32)]
    audio = np.concatenate(pieces[:-1]) if pieces else np.zeros(1, np.float32)
    audio = audio / (np.abs(audio).max() + 1e-6) * 0.9
    tmp = path + ".tmp.wav"
    with wave.open(tmp, "wb") as w:
        w.setnchannels(1)
        w.setsampwidth(2)
        w.setframerate(M["rate"])
        w.writeframes((np.clip(audio, -1, 1) * 32767).astype(np.int16).tobytes())
    os.replace(tmp, path)
    return {"duration": len(audio) / float(M["rate"])}


class H(BaseHTTPRequestHandler):
    protocol_version = "HTTP/1.1"

    def log_message(self, *a):
        pass

    def send(self, code, obj):
        data = json.dumps(obj).encode()
        self.send_response(code)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(data)))
        self.end_headers()
        self.wfile.write(data)

    def do_GET(self):
        self.send(200, STATE)

    def do_POST(self):
        body = json.loads(self.rfile.read(int(self.headers.get("Content-Length") or 0)) or b"{}")
        if STATE["state"] != "ready":
            return self.send(409, STATE)
        try:
            t = time.time()
            out = tts(body)
            out["seconds"] = round(time.time() - t, 2)
            self.send(200, out)
        except Exception as e:
            traceback.print_exc()
            self.send(500, {"error": "%s: %s" % (type(e).__name__, str(e)[:400])})


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--port", type=int, default=7862)
    srv = ThreadingHTTPServer(("127.0.0.1", ap.parse_args().port), H)
    threading.Thread(target=load, daemon=True).start()
    srv.serve_forever()
''')
print("wrote", os.path.join(APP_DIR, 'vs_parler_worker.py'))

In [ ]:
# ACE-Step music worker (runs in its own virtualenv)
# Writes vs_ace_worker.py into APP_DIR. Safe to re-run.
import os
APP_DIR = '/kaggle/working/video_app'
os.makedirs(APP_DIR, exist_ok=True)
with open(os.path.join(APP_DIR, 'vs_ace_worker.py'), "w", encoding="utf-8") as f:
    f.write(r'''"""One-shot ACE-Step music generation (Apache 2.0), run in its own virtualenv because ACE-Step pins its own
transformers. The process exits when done, so the GPU memory is freed for rendering.

    python vs_ace_worker.py --out music.wav --prompt "soft ambient pad, slow, piano" --seconds 30 --device 1
"""
import argparse
import os
import subprocess

ap = argparse.ArgumentParser()
ap.add_argument("--out", required=True)
ap.add_argument("--prompt", required=True)
ap.add_argument("--seconds", type=float, default=30)
ap.add_argument("--device", type=int, default=0)
ap.add_argument("--seed", type=int, default=11)
a = ap.parse_args()

from acestep.pipeline_ace_step import ACEStepPipeline  # noqa: E402

# T4 has no fast bfloat16: float32 with CPU offload keeps it within 16 GB
pipe = ACEStepPipeline(checkpoint_dir=os.environ.get("ACE_CHECKPOINTS") or None, device_id=a.device, dtype="float32",
                       torch_compile=False, cpu_offload=True, overlapped_decode=True)
raw = a.out + ".raw.wav"
pipe(format="wav", audio_duration=a.seconds, prompt=a.prompt + ", instrumental, no vocals", lyrics="[instrumental]",
     infer_step=60, guidance_scale=15.0, scheduler_type="euler", cfg_type="apg", omega_scale=10.0,
     manual_seeds=[a.seed], save_path=raw)
subprocess.run(["ffmpeg", "-v", "error", "-y", "-i", raw, "-ac", "1", "-ar", "44100", "-sample_fmt", "s16", a.out], check=True)
os.remove(raw)
print("ok", a.out)
''')
print("wrote", os.path.join(APP_DIR, 'vs_ace_worker.py'))

In [ ]:
# Renderer: headless Chromium frames -> ffmpeg
# Writes vs_render.py into APP_DIR. Safe to re-run.
import os
APP_DIR = '/kaggle/working/video_app'
os.makedirs(APP_DIR, exist_ok=True)
with open(os.path.join(APP_DIR, 'vs_render.py'), "w", encoding="utf-8") as f:
    f.write(r'''"""Renders one format of a job to MP4. Run as its own process (the server starts one per format, in parallel):
    python vs_render.py <job_dir> <landscape|reel>

Reads <job_dir>/plan.json (scenes with durations, images, captions; written by vs_server.py), writes one
HTML file per scene, seeks every frame in headless Chromium, pipes JPEG screenshots into ffmpeg, joins the
scenes and muxes <job_dir>/audio.wav. Progress goes to <job_dir>/<fmt>/progress.json.
"""
import json
import os
import shutil
import subprocess
import sys
import time

import vs_scenes as V

# Software WebGL (SwiftShader) for the depth-parallax layer, and file:// images readable by WebGL
CHROME_ARGS = ["--no-sandbox", "--disable-dev-shm-usage", "--font-render-hinting=none", "--allow-file-access-from-files",
               "--use-angle=swiftshader", "--enable-unsafe-swiftshader", "--ignore-gpu-blocklist"]


def progress(out_dir, **kw):
    tmp = os.path.join(out_dir, "progress.json.tmp")
    with open(tmp, "w") as f:
        json.dump(dict(kw, updated=time.time()), f)
    os.replace(tmp, os.path.join(out_dir, "progress.json"))


def ffmpeg(*args):
    p = subprocess.run(["ffmpeg", "-hide_banner", "-loglevel", "error", "-nostdin", "-y", *args],
                       capture_output=True, text=True)
    if p.returncode != 0:
        raise RuntimeError("ffmpeg failed: " + p.stderr[-1500:])


def photo_for(sc, w, h):
    """Crop the photo when its shape is close to the frame's; otherwise letterbox it over a blurred fill."""
    ph = sc.get("photo")
    if not ph or not ph.get("path"):
        return None
    pa = ph.get("w", 1) / float(ph.get("h", 1) or 1)
    fa = w / float(h)
    fit = "cover" if 0.72 <= pa / fa <= 1.38 else "contain"
    return dict(ph, fit=fit)


def main(job_dir, fmt):
    plan = json.load(open(os.path.join(job_dir, "plan.json"), encoding="utf-8"))
    out_dir = os.path.join(job_dir, fmt)
    shutil.rmtree(out_dir, ignore_errors=True)
    os.makedirs(out_dir)
    fps = int(plan.get("fps", 30))
    w, h = V.SIZES[fmt]
    if plan.get("quality") == "720p":
        w, h = w * 2 // 3, h * 2 // 3
    scenes = plan["scenes"]
    total_frames = sum(max(1, round(s["dur"] * fps)) for s in scenes)
    total_dur = sum(s["dur"] for s in scenes)
    done, t_acc = 0, 0.0
    progress(out_dir, state="starting", frames=0, total_frames=total_frames)
    from playwright.sync_api import sync_playwright
    t0 = time.time()
    with sync_playwright() as pw:
        browser = pw.chromium.launch(executable_path=os.environ.get("CHROMIUM_PATH") or None,
                                     args=CHROME_ARGS)
        page = browser.new_page(viewport={"width": w, "height": h}, device_scale_factor=1)
        parts = []
        for i, sc in enumerate(scenes):
            caps = V.caption_chunks(sc["narration"], sc["narr_start"], sc["narr_dur"],
                                    5 if fmt == "reel" else 9) if plan.get("captions") else None
            doc = V.scene_html(sc, i, len(scenes), plan["story"], plan["style"], fmt, sc["dur"], photo_for(sc, w, h),
                               caps, 100 * t_acc / total_dur, 100 * (t_acc + sc["dur"]) / total_dur)
            t_acc += sc["dur"]
            path = os.path.join(out_dir, "scene_%02d.html" % i)
            with open(path, "w", encoding="utf-8") as f:
                f.write(doc)
            page.goto("file://" + os.path.abspath(path))
            page.wait_for_function("window.__ready === true", timeout=60000)
            n = max(1, round(sc["dur"] * fps))
            mp4 = os.path.join(out_dir, "scene_%02d.mp4" % i)
            enc = subprocess.Popen(["ffmpeg", "-hide_banner", "-loglevel", "error", "-y", "-f", "image2pipe",
                                    "-framerate", str(fps), "-c:v", "mjpeg", "-i", "-", "-c:v", "libx264",
                                    "-preset", "veryfast", "-crf", "19", "-pix_fmt", "yuv420p", "-r", str(fps), mp4],
                                   stdin=subprocess.PIPE, stderr=subprocess.PIPE)
            try:
                for k in range(n):
                    page.evaluate("t => window.__seek(t)", k / fps)
                    enc.stdin.write(page.screenshot(type="jpeg", quality=92))
                    done += 1
                    if done % 15 == 0:
                        el = time.time() - t0
                        progress(out_dir, state="rendering", scene=i + 1, scenes=len(scenes), frames=done,
                                 total_frames=total_frames, eta=round(el / done * (total_frames - done)))
                    if i == 0 and k == min(n - 1, int(1.8 * fps)):
                        page.screenshot(path=os.path.join(out_dir, "poster.jpg"), type="jpeg", quality=88)
            finally:
                enc.stdin.close()
                err = enc.stderr.read().decode("utf-8", "replace")
                if enc.wait() != 0:
                    raise RuntimeError("ffmpeg could not encode scene %d: %s" % (i + 1, err[-800:]))
            parts.append(mp4)
        browser.close()
    progress(out_dir, state="joining", frames=done, total_frames=total_frames)
    lst = os.path.join(out_dir, "parts.txt")
    with open(lst, "w") as f:
        f.writelines("file '%s'\n" % os.path.abspath(p) for p in parts)
    silent = os.path.join(out_dir, "silent.mp4")
    ffmpeg("-f", "concat", "-safe", "0", "-i", lst, "-c", "copy", silent)
    final = os.path.join(job_dir, "%s.mp4" % fmt)
    audio = os.path.join(job_dir, "audio.wav")
    if os.path.exists(audio):
        ffmpeg("-i", silent, "-i", audio, "-map", "0:v", "-map", "1:a", "-c:v", "copy", "-c:a", "aac", "-b:a", "192k",
               "-shortest", "-movflags", "+faststart", final + ".tmp.mp4")
    else:
        ffmpeg("-i", silent, "-c", "copy", "-movflags", "+faststart", final + ".tmp.mp4")
    os.replace(final + ".tmp.mp4", final)
    for p in parts + [silent, lst]:
        os.remove(p)
    if os.path.exists(os.path.join(out_dir, "poster.jpg")):
        shutil.copy(os.path.join(out_dir, "poster.jpg"), os.path.join(job_dir, "%s.jpg" % fmt))
    progress(out_dir, state="done", frames=done, total_frames=total_frames, seconds=round(time.time() - t0),
             size=os.path.getsize(final))


if __name__ == "__main__":
    try:
        main(sys.argv[1], sys.argv[2])
    except Exception as e:
        import traceback
        traceback.print_exc()
        d = os.path.join(sys.argv[1], sys.argv[2])
        os.makedirs(d, exist_ok=True)
        progress(d, state="error", error="%s: %s" % (type(e).__name__, str(e)[:800]))
        sys.exit(1)
''')
print("wrote", os.path.join(APP_DIR, 'vs_render.py'))

In [ ]:
# Studio server on port 7860: jobs, API and MCP
# Writes vs_server.py into APP_DIR. Safe to re-run.
import os
APP_DIR = '/kaggle/working/video_app'
os.makedirs(APP_DIR, exist_ok=True)
with open(os.path.join(APP_DIR, 'vs_server.py'), "w", encoding="utf-8") as f:
    f.write(r'''"""Video Studio: any link or idea -> narrated video with music, as landscape (16:9) and reel (9:16) MP4s.

Runs as its own process (started by the notebook with studio_http.spawn):
    APP_PASSWORD=... python vs_server.py --port 7860

Pipeline per job (one job at a time, files in WORK_DIR/jobs/<id>/):
    source (vs_source: text, photos, language) -> photos downloaded, edited, described (vs_media)
    -> storyboard in the video's language (local LLM via Ollama, vs_story) -> [optional review/edit]
    -> visuals: the source's photos first, SDXL only for non-news topics -> narration (vs_tts: Kokoro, Indic
    Parler-TTS, Piper) -> music matched to the tone (MusicGen) -> mix -> render landscape + reel in parallel
Photos, images and narration are cached by content, so editing one scene and re-rendering redoes only that scene.
"""
import argparse
import hashlib
import json
import os
import shutil
import subprocess
import sys
import threading
import time
import traceback
import urllib.request
import uuid

import studio_http as K
import vs_media as M
import vs_motion as MO
import vs_scenes as V
import vs_source as SRC
import vs_story as ST
import vs_tts as TTS

HERE = os.path.dirname(os.path.abspath(__file__))
WORK = os.environ.get("WORK_DIR", "/kaggle/working/video_studio")
JOBS = os.path.join(WORK, "jobs")
OLLAMA_URL = os.environ.get("OLLAMA_URL", "http://127.0.0.1:11434")
LLM_MODEL = os.environ.get("LLM_MODEL", "gemma3:12b")
LLM_KEEP_ALIVE = os.environ.get("LLM_KEEP_ALIVE", "10m")
DESCRIBE_PHOTOS = os.environ.get("DESCRIBE_PHOTOS", "1") == "1"
IMAGE_MODEL = os.environ.get("IMAGE_MODEL", "stabilityai/stable-diffusion-xl-base-1.0")
IMAGE_STEPS = int(os.environ.get("IMAGE_STEPS", "25"))
MUSIC_MODEL = os.environ.get("MUSIC_MODEL", "facebook/musicgen-small")
TTS_ENABLED = os.environ.get("TTS", "1") != "0"
MEDIA_DEVICE = os.environ.get("MEDIA_DEVICE", "cuda:0")
TTS_DEVICE = os.environ.get("TTS_DEVICE", "cpu")
UNLOAD_AFTER = os.environ.get("UNLOAD_AFTER", "0") == "1"
DEPTH_ON = os.environ.get("DEPTH_PARALLAX", "1") == "1"            # 2.5D parallax of photos (Depth Anything V2 Small)
CUTOUT_ON = os.environ.get("SUBJECT_CUTOUT", "1") == "1"           # subject lift-off in headline scenes (BiRefNet)
MAPS_ON = os.environ.get("LOCATION_MAPS", "1") == "1"              # map fly-in for news with a place (OpenStreetMap)
I2V_MODEL = os.environ.get("I2V_MODEL", "")                        # "ltx" / "wan" / a repo id; "" = off
I2V_MAX_CLIPS = int(os.environ.get("I2V_MAX_CLIPS", "3"))
MOTION_CHOICES = ("auto", "parallax", "ai", "none")
DEFAULT_QUALITY = os.environ.get("RENDER_QUALITY", "1080p")
FPS = int(os.environ.get("RENDER_FPS", "30"))
FORMATS = ("landscape", "reel")
STYLE_CHOICES = ("auto",) + tuple(V.STYLES)
LANG_CHOICES = ("auto", "ne", "en", "hi", "es", "fr", "it", "pt")
MAX_WAIT = 85
log = K.file_logger(os.environ.get("APP_LOG", "/kaggle/working/logs/video_studio.log"), "video")

LOCK = threading.RLock()
WAKE = threading.Condition(LOCK)
QUEUE = []
MEDIA = M.Media(IMAGE_MODEL, IMAGE_STEPS, MEDIA_DEVICE, MUSIC_MODEL, MEDIA_DEVICE, TTS_DEVICE, log)
MOTION = MO.Motion(MEDIA_DEVICE, log)


# ====================================================================== jobs on disk
def jdir(jid):
    jid = K.safe_name(jid, 40)
    d = os.path.join(JOBS, jid)
    if not jid or not os.path.isfile(os.path.join(d, "job.json")):
        raise K.HTTPError(404, "unknown job %r" % jid)
    return d


def load(jid):
    return K.read_json(os.path.join(jdir(jid), "job.json"))


def save(job, **kw):
    with LOCK:
        path = os.path.join(JOBS, job["id"], "job.json")
        cur = K.read_json(path, job) or job
        cur.update(kw, updated=time.time())
        K.write_json(path, cur)
        job.clear()
        job.update(cur)
    return job


def all_jobs():
    out = []
    if os.path.isdir(JOBS):
        for name in os.listdir(JOBS):
            j = K.read_json(os.path.join(JOBS, name, "job.json"))
            if j:
                out.append(j)
    return sorted(out, key=lambda j: -j["created"])


def files_of(jid):
    d = os.path.join(JOBS, jid)
    return {f: "/api/jobs/%s/files/%s" % (jid, f) for f in ("landscape.mp4", "reel.mp4", "landscape.jpg", "reel.jpg")
            if os.path.exists(os.path.join(d, f))}


def public(job, full=False):
    keys = ("id", "title", "state", "step", "progress", "created", "updated", "error", "warnings", "options",
            "duration", "render", "seconds", "language", "style", "voice")
    out = {k: job.get(k) for k in keys if k in job}
    out["files"] = files_of(job["id"])
    if full:
        d = os.path.join(JOBS, job["id"])
        sb = K.read_json(os.path.join(d, "storyboard.json"))
        if sb:
            out["storyboard"] = sb
        photos = K.read_json(os.path.join(d, "photos.json"), []) or []
        out["photos"] = [{"index": i, "url": "/api/jobs/%s/files/photos/%s" % (job["id"], os.path.basename(p["path"])),
                          "caption": p.get("caption"), "description": p.get("description"), "kind": p.get("kind") or "link",
                          "video": "/api/jobs/%s/files/photos/%s" % (job["id"], os.path.basename(p["clip"])) if p.get("clip") else None}
                         for i, p in enumerate(photos)]
        out["images"] = {str(i): "/api/jobs/%s/files/%s/%s" % (job["id"], os.path.basename(os.path.dirname(p)), os.path.basename(p))
                         for i, p in (job.get("scene_images") or {}).items() if p}
    return out


def parse_options(b, base=None):
    o = dict(base or {"style": "auto", "length": 60, "voice": "auto", "language": "auto", "music": bool(MUSIC_MODEL),
                      "captions": True, "formats": list(FORMATS), "quality": DEFAULT_QUALITY, "review": False,
                      "motion": "auto", "map": True, "sfx": True})
    if b.get("style") not in (None, ""):
        if b["style"] not in STYLE_CHOICES:
            raise ValueError("style must be one of " + ", ".join(STYLE_CHOICES))
        o["style"] = b["style"]
    if b.get("length") not in (None, ""):
        L = int(b["length"])
        if L not in ST.LENGTHS:
            raise ValueError("length must be 15, 30, 60 or 90 seconds")
        o["length"] = L
    if b.get("voice"):
        if b["voice"] not in TTS.VOICES and b["voice"] not in ("auto", "none"):
            raise ValueError("voice must be auto, none or one of %s" % ", ".join(TTS.VOICES))
        o["voice"] = b["voice"]
    if b.get("language"):
        if b["language"] not in LANG_CHOICES:
            raise ValueError("language must be one of " + ", ".join(LANG_CHOICES))
        o["language"] = b["language"]
    if b.get("motion"):
        if b["motion"] not in MOTION_CHOICES:
            raise ValueError("motion must be one of " + ", ".join(MOTION_CHOICES))
        o["motion"] = b["motion"]
    for k in ("music", "captions", "review", "map", "sfx"):
        if b.get(k) is not None and b.get(k) != "":
            o[k] = b[k] if isinstance(b[k], bool) else str(b[k]).lower() in ("1", "true", "yes", "on")
    if b.get("formats") is not None and b.get("formats") != "":
        f = b["formats"] if isinstance(b["formats"], list) else [x.strip() for x in str(b["formats"]).split(",")]
        if not f or any(x not in FORMATS for x in f):
            raise ValueError("formats must be a list with landscape and/or reel")
        o["formats"] = [x for x in FORMATS if x in f]
    if b.get("quality"):
        if b["quality"] not in ("1080p", "720p"):
            raise ValueError("quality must be 1080p or 720p")
        o["quality"] = b["quality"]
    if o["music"] and not MUSIC_MODEL:
        o["music"] = False
    return o


def create(source, opts):
    source = str(source or "").strip()
    if not source:
        raise ValueError("paste a link (news article, GitHub, YouTube, PDF...) or describe the video")
    if len(source) > 4000:
        raise ValueError("the description is longer than 4000 characters")
    jid = time.strftime("%Y%m%d-%H%M%S-") + uuid.uuid4().hex[:5]
    d = os.path.join(JOBS, jid)
    for sub in ("images", "voice", "photos"):
        os.makedirs(os.path.join(d, sub))
    job = {"id": jid, "title": source.split()[0][:100], "input": source, "options": opts, "state": "queued",
           "step": "waiting", "progress": 0.0, "created": time.time(), "warnings": [], "approved": not opts["review"]}
    K.write_json(os.path.join(d, "job.json"), job)
    return enqueue(job)


def enqueue(job, **kw):
    with WAKE:
        save(job, state="queued", step="waiting", error=None, **kw)
        if job["id"] not in QUEUE:
            QUEUE.append(job["id"])
        WAKE.notify_all()
    return job


# ====================================================================== pipeline
def h(*parts):
    return hashlib.sha1("\x1f".join(str(p) for p in parts).encode()).hexdigest()[:16]


def scene_duration(sc, narr_dur, length):
    base = {"headline": 3.4, "stats": 4.2, "outro": 3.0}.get(sc["layout"], 3.2)
    if length <= 15:
        base = min(base, 2.6)
    if sc["layout"] == "code":
        base = max(base, 2.6 + len(sc.get("code") or "") / 28.0)
    if sc["layout"] == "bullets":
        base = max(base, 1.4 + 0.45 * len(sc.get("bullets") or []))
    return round(max(base, narr_dur + (0.7 if length <= 15 else 1.0)), 2)


def auto_style(sb, src):
    if sb.get("category") == "news" or sb.get("tone") in ("tragic", "serious"):
        return "broadcast"
    if src["kind"] == "article" and sb.get("category") not in ("tech", "education", "promo", "story"):
        return "broadcast"                              # an article the model did not label: treat it as news
    return {"github": "midnight", "pdf": "paper"}.get(src["kind"]) or {
        "tech": "midnight", "education": "paper", "story": "documentary", "promo": "neon"}.get(sb.get("category"), "documentary")


def step(job, name, frac):
    save(job, state="running", step=name, progress=round(frac, 3))
    log.info("%s: %s", job["id"], name)


def assign_photos(scenes, n_photos):
    """Scenes without a photo get unused ones (no photo twice in a row); headline and outro reuse the lead photo."""
    used = {sc["photo"] for sc in scenes if sc.get("photo", -1) >= 0}           # -2 = the user chose no picture
    free = [i for i in range(n_photos) if i not in used]
    prev = None
    for k, sc in enumerate(scenes):
        if sc.get("photo", -1) == -1 and n_photos and sc["layout"] in ("photo", "bullets", "quote", "headline", "outro"):
            if sc["layout"] in ("headline", "outro"):
                sc["photo"] = 0
            elif free:
                sc["photo"] = free.pop(0)
            elif sc["layout"] == "photo" and n_photos > 1:
                sc["photo"] = (prev + 1) % n_photos if prev is not None else 0
        prev = sc.get("photo") if sc.get("photo", -1) >= 0 else prev
    return scenes


def map_for(place, style, d):
    """Map snapshots for a place name (cached per job); None when the place cannot be found."""
    geo = MO.geocode(place, os.path.join(WORK, "geo"))
    if not geo:
        return None
    out = os.path.join(d, "map_" + h(place, style))
    paths = {k: os.path.join(out, "map_%s.jpg" % k) for k, _ in MO.MAP_ZOOMS}
    if not all(os.path.exists(p) for p in paths.values()):
        paths = MO.place_map(geo[0], geo[1], out, "light" if style in V.LIGHT else "dark")
    return paths


def add_motion(job, scenes, visuals, motion, news, warnings):
    """Adds depth maps, subject cut-outs or AI clips to the scenes' pictures (cached next to each picture)."""
    noted = set()

    def note(msg):
        if msg not in noted:
            noted.add(msg)
            warnings.append(msg)
    use_ai = motion == "ai" and bool(I2V_MODEL) and not news
    if motion == "ai" and news:
        note("AI motion is never used on news (it would invent movement); real photos got 2.5D parallax instead")
    elif motion == "ai" and not I2V_MODEL:
        note("AI motion needs I2V_MODEL in the notebook settings; used 2.5D parallax")
    order = sorted(visuals)
    clips, broken = 0, set()                            # an effect whose model failed once is not retried
    for k, i in enumerate(order):
        v, lay = visuals[i], scenes[i]["layout"]
        base = v["path"]
        if v.get("frames"):
            continue                                    # already moving (the user's video clip)
        step(job, "adding motion %d/%d" % (k + 1, len(order)), 0.2 + 0.2 * k / max(1, len(order)))
        try:
            kind = "ai"
            if use_ai and "ai" not in broken and clips < I2V_MAX_CLIPS and lay in ("headline", "photo", "bullets"):
                fdir = base + ".i2v_" + h(I2V_MODEL)
                if not MO.frames_in(fdir):
                    prompt = (scenes[i].get("image_prompt") or scenes[i].get("heading") or "") + \
                        ", subtle natural motion, slow cinematic camera move, stable, high quality"
                    MOTION.animate(base, prompt, fdir, I2V_MODEL)
                frames = MO.frames_in(fdir)
                if frames:
                    v.update(frames=frames, fps=24)
                    clips += 1
                    continue
            kind = "cut"
            if CUTOUT_ON and "cut" not in broken and motion in ("auto", "ai") and lay == "headline":
                cut, none = base + ".cut.png", base + ".nocut"
                if not os.path.exists(cut) and not os.path.exists(none):
                    if not MOTION.cutout(base, cut)[0]:
                        open(none, "w").close()            # no clear subject: remember, use parallax
                if os.path.exists(cut):
                    v["cut"] = cut
                    continue
            kind = "depth"
            if DEPTH_ON and "depth" not in broken:
                dp = base + ".depth.png"
                if not os.path.exists(dp):
                    MOTION.depth(base, dp)
                v["depth"] = dp
        except Exception as e:
            broken.add(kind)
            log.warning("motion (%s) for scene %d failed: %s\n%s", kind, i + 1, e, traceback.format_exc())
            note("some motion effects were skipped (%s)" % str(e)[:160])
    MOTION.unload()


def run_job(job):
    d = os.path.join(JOBS, job["id"])
    o = job["options"]
    t0 = time.time()
    warnings = []
    # 1. source + photos
    src = K.read_json(os.path.join(d, "source.json"))
    photos = K.read_json(os.path.join(d, "photos.json"))
    if not src:
        step(job, "reading the link", 0.02)
        src = SRC.fetch_source(job["input"])
        step(job, "collecting photos", 0.05)
        photos = M.fetch_photos(src.get("images") or [], os.path.join(d, "photos"), 14, referer=src.get("url"), log=log)
        for im in src.get("images") or []:
            im.pop("bytes", None)
        if photos and DESCRIBE_PHOTOS and LLM_MODEL:
            step(job, "looking at the photos", 0.07)
            photos = M.describe_photos(photos, OLLAMA_URL, LLM_MODEL, LLM_KEEP_ALIVE, log,
                                       topic=src["title"] + ". " + (src.get("description") or ""), keep=8)
        photos = photos[:8]
        K.write_json(os.path.join(d, "source.json"), src)
        K.write_json(os.path.join(d, "photos.json"), photos)
        save(job, title=src["title"][:100])
    photos = photos or []
    lang = o.get("language") if o.get("language") not in (None, "auto") else (src.get("language") or "en")
    # 2. storyboard
    sb = K.read_json(os.path.join(d, "storyboard.json"))
    if not sb:
        step(job, "writing the script in %s (%s)" % (ST.lang_info(lang)[0], LLM_MODEL), 0.1)
        try:
            sb, warn = ST.write_storyboard(src, o["length"], lang, OLLAMA_URL, LLM_MODEL, LLM_KEEP_ALIVE, photos, log)
        except Exception as e:
            sb, warn = ST.fallback(src, o["length"], lang, len(photos)), "script model unavailable (%s); used the article's own sentences" % e
        if warn:
            warnings.append(warn)
        K.write_json(os.path.join(d, "storyboard.json"), sb)
        save(job, title=sb["title"][:100])
    lang = sb.get("language") or lang
    style = o["style"] if o["style"] != "auto" else auto_style(sb, src)
    save(job, language=lang, style=style)
    if not job.get("approved"):
        save(job, state="review", step="review the script, then render", progress=0.15, warnings=warnings)
        return
    scenes = assign_photos([dict(sc) for sc in sb["scenes"]], len(photos))
    news = sb.get("category") == "news" or sb.get("tone") in ("tragic", "serious")
    # 2b. where it happened: a map fly-in after the headline (news with a real place, 30 s and longer)
    if o.get("map", True) and MAPS_ON and news and sb.get("place") and o["length"] > 15 and len(scenes) > 1:
        try:
            step(job, "drawing the map of %s" % sb["place"], 0.12)
            mp = map_for(sb["place"], style, d)
            if mp:
                scenes.insert(1, {"layout": "map", "heading": sb.get("place_local") or "", "kicker": "", "bullets": [],
                                  "narration": "", "photo": -2, "code": "", "quote": "", "stats": [], "inserted": True,
                                  "map": dict(mp, label=sb.get("place_local") or sb["place"].split(",")[0])})
        except Exception as e:
            log.warning("map skipped: %s", e)
            warnings.append("map skipped (%s)" % str(e)[:120])
    n = len(scenes)
    credit = (src.get("facts") or {}).get("site") or ""
    # 3. visuals: source photos first; generated images only for non-news topics
    visuals = {}
    for i, sc in enumerate(scenes):
        p = sc.get("photo", -1)
        if 0 <= p < len(photos):
            ph = photos[p]
            visuals[i] = {"path": ph["path"], "w": ph["w"], "h": ph["h"],
                          "credit": "" if ph.get("kind") in ("upload", "video") else credit}
            if ph.get("frames_dir"):                    # the user's own video clip
                visuals[i].update(frames=MO.frames_in(ph["frames_dir"]), fps=ph.get("fps", 24))
    if IMAGE_MODEL and not news:
        for i, sc in enumerate(scenes):
            if i in visuals or sc["layout"] not in ("headline", "bullets", "photo", "outro") or not sc.get("image_prompt"):
                continue
            path = os.path.join(d, "images", h(sc["image_prompt"], style, IMAGE_MODEL) + ".png")
            if not os.path.exists(path):
                step(job, "drawing image %d/%d" % (i + 1, n), 0.15 + 0.3 * i / n)
                try:
                    MEDIA.image(sc["image_prompt"], style, path, seed=int(h(sc["image_prompt"])[:6], 16))
                except Exception as e:
                    log.error("image failed\n%s", traceback.format_exc())
                    warnings.append("image for scene %d failed: %s" % (i + 1, str(e)[:200]))
                    continue
            visuals[i] = {"path": path, "w": 1024, "h": 1024, "credit": "AI"}
    if 0 in visuals and n - 1 not in visuals and scenes[n - 1].get("photo", -1) != ST.NO_PHOTO:
        visuals[n - 1] = dict(visuals[0])
    # 3b. motion: real photos move in 2.5D (depth parallax) or lift their subject; AI clips only off the news
    motion = o.get("motion", "auto")
    if motion != "none":
        add_motion(job, scenes, visuals, motion, news, warnings)
    # 4. narration
    narr, voice = {}, None
    if TTS_ENABLED and o["voice"] != "none":
        voice = TTS.pick(o["voice"], lang)
        if not voice:
            warnings.append("no voice is installed for %s; the video has captions only" % ST.lang_info(lang)[0])
        elif o["voice"] not in ("auto", voice):
            warnings.append("voice %s is not available; used %s" % (o["voice"], voice))
    if voice:
        for i, sc in enumerate(scenes):
            text = (sc.get("narration") or "").strip()
            if not text:
                continue
            path = os.path.join(d, "voice", h(text, voice, sb.get("tone")) + ".wav")
            if not os.path.exists(path):
                step(job, "recording narration %d/%d" % (i + 1, n), 0.45 + 0.17 * i / n)
                try:
                    TTS.speak(text, voice, path, MEDIA, sb.get("tone"), MEDIA_DEVICE, log)
                except Exception as e:
                    log.error("tts failed\n%s", traceback.format_exc())
                    warnings.append("narration failed (%s); the video has captions only" % str(e)[:200])
                    narr = {}
                    break
            narr[i] = (path, M.duration(path))
    if voice and TTS.VOICES.get(voice, {}).get("engine") == "svara":
        TTS.svara_unload()                              # free the GPU for music and rendering
        MEDIA.unload("-")
    spoken, room = sum(x[1] for x in narr.values()), 0.8 * o["length"]
    if narr and spoken > room * 1.04:                   # too long for the chosen length: speak a little faster
        f = min(1.25, spoken / room)
        log.info("%s: narration %.1fs for %ss; speeding up %.2fx", job["id"], spoken, o["length"], f)
        try:
            for i, (path, _) in list(narr.items()):
                fast = path[:-4] + "_x%.2f.wav" % f
                narr[i] = (fast, M.duration(fast) if os.path.exists(fast) else M.tempo(path, fast, f))
        except Exception as e:
            log.warning("speed-up failed: %s", e)
    save(job, voice=voice)
    # 5. timing
    wps = ST.WPS.get(lang, 2.3)
    nds = [narr[i][1] if i in narr else len((sc.get("narration") or "").split()) / wps for i, sc in enumerate(scenes)]
    durs = [scene_duration(sc, nds[i], o["length"]) for i, sc in enumerate(scenes)]
    if sum(durs) < 0.8 * o["length"]:                   # short narration: let the pictures breathe, up to 1.6x
        f = min(1.6, 0.85 * o["length"] / sum(durs))
        durs = [round(x * f, 2) for x in durs]
    plan_scenes, t = [], 0.0
    for i, sc in enumerate(scenes):
        plan_scenes.append(dict(sc, dur=durs[i], narr_start=0.5, narr_dur=nds[i], photo=visuals.get(i), start=t))
        t += durs[i]
    total = round(t, 2)
    # 6. music
    music = None
    if o["music"] and MUSIC_MODEL:
        music = os.path.join(d, "music_%s.wav" % h(sb.get("music_prompt"), MUSIC_MODEL))
        if not os.path.exists(music):
            step(job, "composing music", 0.64)
            try:
                MEDIA.music(sb.get("music_prompt") or "light background music", music, 30)
            except Exception as e:
                log.error("music failed\n%s", traceback.format_exc())
                warnings.append("music failed: %s" % str(e)[:200])
                music = None
    if UNLOAD_AFTER:
        MEDIA.unload()
        K.stop_process("vs-parler")
    # 7. audio mix
    step(job, "mixing audio", 0.68)
    audio = os.path.join(d, "audio.wav")
    cuts = [ps["start"] for ps in plan_scenes[1:]] if o.get("sfx", True) else None
    if narr or music or cuts:
        M.mix(total, [(ps["start"] + ps["narr_start"], narr[i][0]) for i, ps in enumerate(plan_scenes) if i in narr],
              music, audio, sb.get("tone", "neutral"), cuts)
    elif os.path.exists(audio):
        os.remove(audio)
    # 8. render
    story = dict(sb, language=lang)
    K.write_json(os.path.join(d, "plan.json"), {"story": story, "style": style, "captions": o["captions"], "fps": FPS,
                                                "quality": o["quality"], "scenes": plan_scenes})
    for f in FORMATS:
        for ext in (".mp4", ".jpg"):
            if os.path.exists(os.path.join(d, f + ext)):
                os.remove(os.path.join(d, f + ext))
    procs = {f: subprocess.Popen([sys.executable, os.path.join(HERE, "vs_render.py"), d, f], cwd=HERE,
                                 stdout=open(os.path.join(d, "render_%s.log" % f), "w"), stderr=subprocess.STDOUT)
             for f in o["formats"]}
    while any(p.poll() is None for p in procs.values()):
        prog = {f: K.read_json(os.path.join(d, f, "progress.json"), {}) or {} for f in procs}
        frac = sum((p.get("frames", 0) / max(1, p.get("total_frames", 1))) for p in prog.values()) / len(procs)
        eta = max([p.get("eta", 0) or 0 for p in prog.values()] + [0])
        save(job, state="running", step="rendering %s (about %d:%02d left)" % (" + ".join(procs), eta // 60, eta % 60),
             progress=round(0.7 + 0.3 * frac, 3), render=prog)
        time.sleep(2)
    errors = []
    for f, p in procs.items():
        pr = K.read_json(os.path.join(d, f, "progress.json"), {}) or {}
        if p.returncode != 0 or pr.get("state") != "done":
            errors.append("%s: %s" % (f, pr.get("error") or K.tail(os.path.join(d, "render_%s.log" % f), 5)))
    if errors:
        raise RuntimeError("rendering failed; " + " | ".join(errors))
    save(job, state="done", step="done", progress=1.0, duration=total, warnings=warnings, seconds=round(time.time() - t0),
         scene_images={str(k): visuals[i]["path"] for k, i in enumerate(i for i, sc in enumerate(scenes) if not sc.get("inserted"))
                       if i in visuals},
         render={f: K.read_json(os.path.join(d, f, "progress.json"), {}) for f in procs})
    log.info("%s done in %ds", job["id"], time.time() - t0)


def worker():
    while True:
        with WAKE:
            while not QUEUE:
                WAKE.wait(30)
            jid = QUEUE.pop(0)
        try:
            job = load(jid)
        except K.HTTPError:
            continue
        try:
            run_job(job)
        except Exception as e:
            log.error("job %s failed\n%s", jid, traceback.format_exc())
            save(job, state="error", step="failed", error="%s: %s" % (type(e).__name__, str(e)[:800]))


def wait(jid, seconds):
    t = time.time()
    while True:
        job = load(jid)
        if job["state"] not in ("queued", "running") or time.time() - t >= max(0, min(MAX_WAIT, float(seconds or 0))):
            return job
        time.sleep(1.5)


ASSET_LIMIT = 200 << 20


def add_asset(jid, data_iter=None, url=None, name=""):
    """Adds the user's own picture or video clip to a job's photos; returns its index. Videos keep up to 10 s,
    cut into 24 fps frames that the scene plays exactly; pictures are checked and lightly edited like link photos."""
    d = jdir(jid)
    job = load(jid)
    if job["state"] in ("queued", "running"):
        raise ValueError("the job is busy (%s); wait for it to finish" % job["state"])
    pdir = os.path.join(d, "photos")
    os.makedirs(pdir, exist_ok=True)
    tmp = os.path.join(pdir, "upload_%s.tmp" % uuid.uuid4().hex[:8])
    with open(tmp, "wb") as f:
        if url:
            req = urllib.request.Request(url, headers={"User-Agent": SRC.UA})
            with urllib.request.urlopen(req, timeout=60) as r:
                while True:
                    chunk = r.read(1 << 20)
                    if not chunk:
                        break
                    f.write(chunk)
                    if f.tell() > ASSET_LIMIT:
                        raise ValueError("that file is larger than 200 MB")
        else:
            for chunk in data_iter:
                f.write(chunk)
    photos = K.read_json(os.path.join(d, "photos.json"), []) or []
    k = len(photos)
    try:
        from PIL import Image, ImageOps
        im = ImageOps.exif_transpose(Image.open(tmp)).convert("RGB")
        im.thumbnail((2400, 2400))
        path = os.path.join(pdir, "upload_%02d.jpg" % k)
        im.save(path, "JPEG", quality=92)
        entry = {"path": path, "w": im.size[0], "h": im.size[1], "caption": "", "alt": name, "url": url or "", "kind": "upload"}
        os.remove(tmp)
    except Exception:
        probe = subprocess.run(["ffprobe", "-v", "error", "-select_streams", "v:0", "-show_entries", "stream=width,height",
                                "-of", "csv=p=0", tmp], capture_output=True, text=True)
        if probe.returncode != 0 or "," not in probe.stdout:
            os.remove(tmp)
            raise ValueError("that file is not a picture or a video ffmpeg can read")
        clip = os.path.join(pdir, "clip_%02d.mp4" % k)
        fdir = os.path.join(pdir, "clip_%02d_frames" % k)
        os.makedirs(fdir, exist_ok=True)
        sz = "scale='if(gt(iw,ih),min(1600,iw),-2)':'if(gt(iw,ih),-2,min(1600,ih))'"
        subprocess.run(["ffmpeg", "-v", "error", "-y", "-i", tmp, "-t", "10", "-an", "-vf", sz, "-c:v", "libx264", "-preset",
                        "veryfast", "-crf", "20", "-pix_fmt", "yuv420p", clip], check=True, timeout=600)
        subprocess.run(["ffmpeg", "-v", "error", "-y", "-i", clip, "-vf", "fps=24", "-q:v", "3",
                        os.path.join(fdir, "f%03d.jpg")], check=True, timeout=600)
        os.remove(tmp)
        frames = MO.frames_in(fdir)
        if not frames:
            raise ValueError("could not read frames from that video")
        from PIL import Image
        poster = os.path.join(pdir, "clip_%02d.jpg" % k)
        shutil.copy(frames[min(len(frames) - 1, 12)], poster)
        w, hh = Image.open(poster).size
        entry = {"path": poster, "w": w, "h": hh, "caption": "", "alt": name, "url": url or "", "kind": "video",
                 "clip": clip, "frames_dir": fdir, "fps": 24}
    photos.append(entry)
    K.write_json(os.path.join(d, "photos.json"), photos)
    return {"index": k, "kind": entry["kind"], "photos": public(load(jid), full=True)["photos"]}


def set_storyboard(jid, sb, render=False):
    d = jdir(jid)
    job = load(jid)
    if job["state"] in ("queued", "running"):
        raise ValueError("the job is busy (%s); wait for it to finish" % job["state"])
    src = K.read_json(os.path.join(d, "source.json")) or {"kind": "prompt", "title": job["title"], "text": "", "facts": {}}
    photos = K.read_json(os.path.join(d, "photos.json"), []) or []
    if isinstance(sb, str):
        sb = json.loads(sb)
    lang = job.get("language") or src.get("language") or "en"
    clean = ST.validate_user_storyboard(sb, src, job["options"]["length"], lang, len(photos))
    K.write_json(os.path.join(d, "storyboard.json"), clean)
    save(job, title=clean["title"][:100])
    if render:
        enqueue(job, approved=True)
    return clean


def rerender(jid, b):
    job = load(jid)
    if job["state"] in ("queued", "running"):
        raise ValueError("the job is already %s" % job["state"])
    b = b or {}
    opts = parse_options(b, job["options"])
    if b.get("rewrite") or (b.get("language") and b["language"] != job["options"].get("language")):
        p = os.path.join(jdir(jid), "storyboard.json")
        if os.path.exists(p):
            os.remove(p)
    return enqueue(job, options=opts, approved=not (b.get("rewrite") and opts["review"]))


def delete(jid):
    job = load(jid)
    if job["state"] == "running":
        raise ValueError("the job is running; wait for it to finish")
    with LOCK:
        if jid in QUEUE:
            QUEUE.remove(jid)
    shutil.rmtree(jdir(jid))
    return {"deleted": jid}


def info():
    jobs = all_jobs()
    return {"styles": list(STYLE_CHOICES), "voices": TTS.voice_options(), "lengths": list(ST.LENGTHS),
            "languages": {k: (ST.lang_info(k)[1] if k != "auto" else "Same as the link") for k in LANG_CHOICES},
            "formats": list(FORMATS), "qualities": ["1080p", "720p"], "llm": LLM_MODEL, "image_model": IMAGE_MODEL or None,
            "music_model": MUSIC_MODEL or None, "tts": TTS_ENABLED, "gpus": K.gpu_info(),
            "motion": {"choices": list(MOTION_CHOICES), "parallax": DEPTH_ON, "cutout": CUTOUT_ON, "maps": MAPS_ON,
                       "ai": I2V_MODEL or None},
            "queue": {s: sum(1 for j in jobs if j["state"] == s) for s in ("queued", "running", "review", "done", "error")}}


# ====================================================================== app
def build_app():
    app = K.App("Video Studio", password=os.environ.get("APP_PASSWORD"), log=log, max_body=8 << 20, instructions=(
        "Makes narrated videos with music from any link (news article in Nepali or English, GitHub repo, YouTube, PDF) "
        "or an idea, as landscape (16:9) and reel (9:16) MP4s. It uses the link's own photos, speaks the link's "
        "language (Nepali and Hindi via Indic Parler-TTS) and picks a style from the content (broadcast for news). "
        "make_video queues a job; poll get_job until state is 'done' (a few minutes), then download files.landscape.mp4 / "
        "files.reel.mp4 with the same password. With review=true it stops at state 'review': read get_storyboard, edit "
        "it with update_storyboard(render=true) or call render."))
    app.page("/", os.path.join(HERE, "vs_ui.html"))
    app.static("/static/", HERE)

    @app.route("GET", "/api/info")
    def _info(req):
        return info()

    @app.route("GET", "/api/jobs")
    def _jobs(req):
        return {"jobs": [public(j) for j in all_jobs()]}

    @app.route("POST", "/api/jobs")
    def _create(req):
        b = req.json()
        return public(create(b.get("source"), parse_options(b)))

    @app.route("GET", r"/api/jobs/(?P<j>[\w-]+)")
    def _job(req):
        return public(wait(req.params["j"], req.arg("wait", 0, float)), full=True)

    @app.route("POST", r"/api/jobs/(?P<j>[\w-]+)/storyboard")
    def _sb(req):
        b = req.json()
        return {"storyboard": set_storyboard(req.params["j"], b.get("storyboard"), bool(b.get("render")))}

    @app.route("POST", r"/api/jobs/(?P<j>[\w-]+)/render")
    def _render(req):
        return public(rerender(req.params["j"], req.json()))

    @app.route("POST", r"/api/jobs/(?P<j>[\w-]+)/assets")
    def _asset(req):
        """The raw picture or video file as the body (any image type, or mp4 / mov / webm up to 200 MB)."""
        return add_asset(req.params["j"], req.iter_body(limit=ASSET_LIMIT), name=req.arg("name", "") or "")

    @app.route("POST", r"/api/jobs/(?P<j>[\w-]+)/delete")
    def _delete(req):
        return delete(req.params["j"])

    @app.route("GET", r"/api/jobs/(?P<j>[\w-]+)/files/(?P<f>(?:images/|photos/)?[\w.-]+)")
    def _file(req):
        d = jdir(req.params["j"])
        f = req.params["f"]
        job = load(req.params["j"])
        name = "%s-%s" % (K.safe_name(job.get("title"), 50) or job["id"], f) if f.endswith(".mp4") else None
        return K.FileResponse(os.path.join(d, f), name=name if req.arg("download") else None)

    # ---------------------------------------------------------------- MCP tools
    opt_schema = {
        "motion": {"type": "string", "enum": list(MOTION_CHOICES), "default": "auto",
                   "description": "auto: 2.5D depth parallax on photos and subject lift-off on headlines; ai: AI "
                                  "image-to-video clips (non-news only, slow; needs I2V_MODEL); parallax; none"},
        "map": {"type": "boolean", "default": True, "description": "map fly-in to the story's place (news, 30 s+)"},
        "sfx": {"type": "boolean", "default": True, "description": "soft transition sounds"},
        "style": {"type": "string", "enum": list(STYLE_CHOICES), "default": "auto",
                  "description": "auto picks broadcast for news, midnight for tech, documentary for stories"},
        "length": {"type": "integer", "enum": list(ST.LENGTHS), "default": 60, "description": "seconds"},
        "language": {"type": "string", "enum": list(LANG_CHOICES), "default": "auto", "description": "auto = the link's language"},
        "voice": {"type": "string", "enum": ["auto", "none"] + list(TTS.VOICES), "default": "auto"},
        "music": {"type": "boolean", "default": True}, "captions": {"type": "boolean", "default": True},
        "formats": {"type": "array", "items": {"type": "string", "enum": list(FORMATS)}, "default": list(FORMATS)},
        "quality": {"type": "string", "enum": ["1080p", "720p"], "default": DEFAULT_QUALITY}}

    @app.tool("make_video", "Make a narrated video with music from a link (news article, GitHub repo, YouTube, PDF) or a "
              "description. Text after a link is passed to the scriptwriter as instructions. Returns a job; poll get_job.",
              dict({"source": {"type": "string"}, "review": {"type": "boolean", "default": False,
                                                             "description": "stop after the script so it can be edited"},
                    "wait_seconds": {"type": "number", "default": 0}}, **opt_schema), ["source"])
    def t_make(source, wait_seconds=0, **opts):
        job = create(source, parse_options(opts))
        return public(wait(job["id"], wait_seconds) if wait_seconds else job)

    @app.tool("get_job", "Job state (queued, running, review, done, error), progress and download links. "
              "wait_seconds (max 85) waits for it to finish.", {
                  "job_id": {"type": "string"}, "wait_seconds": {"type": "number", "default": 0}}, ["job_id"])
    def t_get(job_id, wait_seconds=0):
        return public(wait(job_id, wait_seconds))

    @app.tool("get_storyboard", "The job's script: title, tagline, language, category, tone, music prompt and scenes "
              "(layout, heading, kicker, bullets, narration, photo index, image_prompt, code, quote, stats), plus the "
              "photos found on the link.", {"job_id": {"type": "string"}}, ["job_id"])
    def t_sb(job_id):
        sb = K.read_json(os.path.join(jdir(job_id), "storyboard.json"))
        if not sb:
            raise ValueError("the script is not written yet (state: %s)" % load(job_id)["state"])
        return dict(sb, photos=public(load(job_id), full=True)["photos"])

    @app.tool("update_storyboard", "Replace the job's script (same shape as get_storyboard) and optionally render it. "
              "Unchanged scenes reuse their photos and narration. Each scene's photo is an index into the job's photos "
              "(get_storyboard lists them, add_asset adds more), -1 to let the studio pick, or -2 for no picture.", {
                  "job_id": {"type": "string"}, "storyboard": {"type": "object"},
                  "render": {"type": "boolean", "default": True}}, ["job_id", "storyboard"])
    def t_update(job_id, storyboard, render=True):
        storyboard = {k: v for k, v in storyboard.items() if k != "photos"}
        return {"storyboard": set_storyboard(job_id, storyboard, render), "state": load(job_id)["state"]}

    @app.tool("add_asset", "Add a picture or a short video clip (mp4/mov/webm, first 10 s used) from a URL to a job's "
              "photos, then use its index as a scene's photo in update_storyboard. Uploads from disk: POST the raw file to "
              "/api/jobs/<id>/assets.", {"job_id": {"type": "string"}, "url": {"type": "string"}}, ["job_id", "url"])
    def t_asset(job_id, url):
        return add_asset(job_id, url=url)

    @app.tool("render", "Render (or re-render) a job, optionally with new options. rewrite=true (or a new language) "
              "writes a new script first.", dict({"job_id": {"type": "string"}, "rewrite": {"type": "boolean", "default": False}},
                                                 **opt_schema), ["job_id"])
    def t_render(job_id, **b):
        return public(rerender(job_id, b))

    @app.tool("list_jobs", "Recent jobs, newest first.", {"limit": {"type": "integer", "default": 20}})
    def t_list(limit=20):
        return [public(j) for j in all_jobs()[:max(1, int(limit))]]

    @app.tool("delete_job", "Delete a job and its files.", {"job_id": {"type": "string"}}, ["job_id"])
    def t_delete(job_id):
        return delete(job_id)

    @app.tool("list_options", "Styles, languages, installed voices, lengths, formats and the models in use.")
    def t_options():
        return info()

    return app


def resume():
    for j in all_jobs():
        if j["state"] in ("running", "queued"):
            QUEUE.append(j["id"])
            save(j, state="queued", step="waiting (restarted)")


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--port", type=int, default=7860)
    os.makedirs(JOBS, exist_ok=True)
    resume()
    threading.Thread(target=worker, daemon=True).start()
    build_app().serve_forever(ap.parse_args().port)
''')
print("wrote", os.path.join(APP_DIR, 'vs_server.py'))

In [ ]:
# Studio page
# Writes vs_ui.html into APP_DIR. Safe to re-run.
import os
APP_DIR = '/kaggle/working/video_app'
os.makedirs(APP_DIR, exist_ok=True)
with open(os.path.join(APP_DIR, 'vs_ui.html'), "w", encoding="utf-8") as f:
    f.write(r'''<!doctype html>
<html lang="en"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>Video Studio</title>
<link rel="stylesheet" href="/static/kit.css">
<style>
body{font-family:system-ui,-apple-system,"Segoe UI",Roboto,"Noto Sans","Noto Sans Devanagari",sans-serif}
.topbar{position:sticky;top:0;z-index:30;display:flex;align-items:center;gap:14px;min-height:64px;padding:10px 20px;
 background:color-mix(in srgb,var(--panel) 92%,transparent);backdrop-filter:blur(10px);border-bottom:1px solid var(--line)}
.ham{width:44px;height:44px;flex:none;border-radius:10px;border:1px solid var(--line);background:var(--panel2);display:flex;flex-direction:column;align-items:center;justify-content:center;gap:4px;cursor:pointer;padding:0}
.ham i{display:block;width:18px;height:2px;background:var(--text);border-radius:2px}
.brand{display:flex;align-items:center;gap:12px;font-weight:800;font-size:21px;letter-spacing:-.01em;white-space:nowrap}
.brand .dot{width:34px;height:34px;border-radius:10px;flex:none;background:linear-gradient(135deg,var(--accent),#ff5f8a);display:flex;align-items:center;justify-content:center;color:#fff;font-size:14px}
.topbar .right{margin-left:auto;display:flex;gap:8px;align-items:center}
.pill{font-size:13px;color:var(--muted);border:1px solid var(--line);border-radius:999px;padding:4px 10px;white-space:nowrap}
/* drawer */
.scrim{position:fixed;inset:0;background:rgba(0,0,0,.4);z-index:40;opacity:0;pointer-events:none;transition:opacity .2s}
.drawer{position:fixed;top:0;left:0;bottom:0;width:min(340px,88vw);z-index:41;background:var(--panel);border-right:1px solid var(--line);
 transform:translateX(-102%);transition:transform .22s ease;display:flex;flex-direction:column}
body.open .scrim{opacity:1;pointer-events:auto}body.open .drawer{transform:none}
.drawer .head{display:flex;align-items:center;gap:10px;padding:14px 16px;border-bottom:1px solid var(--line)}
.drawer .list{flex:1;overflow:auto;padding:8px}
.drawer .foot{border-top:1px solid var(--line);padding:10px 12px;display:flex;flex-direction:column;gap:6px}
.drawer .foot a,.drawer .foot button{justify-content:flex-start;width:100%}
.item{display:flex;gap:10px;align-items:center;padding:8px;border-radius:10px;cursor:pointer}
.item:hover{background:var(--panel2)}.item.on{background:color-mix(in srgb,var(--accent) 14%,transparent)}
.item img,.item .ph{width:72px;height:41px;border-radius:6px;object-fit:cover;background:var(--panel2);flex:none}
.item .t{font-size:13px;font-weight:600;overflow:hidden;text-overflow:ellipsis;white-space:nowrap}
.item .m{font-size:11.5px;color:var(--muted);display:flex;gap:6px;align-items:center}
/* create */
main{max-width:1100px}
.hero{background:var(--panel);border:1px solid var(--line);border-radius:18px;padding:18px;box-shadow:0 10px 30px rgba(0,0,0,.06)}
.hero textarea{border:0;background:transparent;font-size:17px;min-height:64px;padding:6px 4px;resize:none;outline:none!important}
.hero .row2{display:flex;flex-wrap:wrap;gap:10px;align-items:center;margin-top:10px;padding-top:12px;border-top:1px solid var(--line)}
.seg{display:inline-flex;border:1px solid var(--line);border-radius:10px;overflow:hidden}
.seg button{border:0;border-radius:0;background:transparent;padding:7px 12px;font-size:13px}
.seg button.on{background:var(--accent);color:#fff}
.fmt{display:inline-flex;gap:6px}
.fmt button{gap:6px;padding:6px 10px;font-size:13px}.fmt button.on{border-color:var(--accent);color:var(--accent);background:color-mix(in srgb,var(--accent) 10%,transparent)}
.fmt .ic{display:inline-block;border:2px solid currentColor;border-radius:3px}
.go{margin-left:auto;padding:10px 20px;font-size:15px;border-radius:12px}
details.more{margin-top:12px}details.more summary{cursor:pointer;color:var(--muted);font-size:13px}
.opts{display:grid;grid-template-columns:repeat(auto-fill,minmax(170px,1fr));gap:12px;margin-top:12px}
.styles{display:flex;gap:8px;flex-wrap:wrap;margin-top:12px}
.sty{border:1px solid var(--line);border-radius:10px;padding:8px 10px;cursor:pointer;font-size:13px;display:flex;gap:8px;align-items:center;background:var(--panel2)}
.sty.on{border-color:var(--accent);box-shadow:0 0 0 2px color-mix(in srgb,var(--accent) 30%,transparent)}
.sty i{width:18px;height:18px;border-radius:5px;display:inline-block}
.examples{display:flex;gap:8px;flex-wrap:wrap;margin-top:10px}.examples button{font-size:12px;padding:4px 10px;border-radius:999px}
/* job */
.jobhead{display:flex;align-items:center;gap:10px;flex-wrap:wrap;margin-bottom:6px}
.jobhead h2{margin:0;font-size:22px;line-height:1.3}
.prog{margin:14px 0 6px;padding:18px 20px;border-radius:16px;background:var(--panel2);border:1px solid var(--line)}
.prog .top{display:flex;align-items:baseline;gap:14px;flex-wrap:wrap}
.prog .pct{font-size:40px;font-weight:800;line-height:1;font-variant-numeric:tabular-nums;color:var(--accent)}
.prog .what{font-size:16px;font-weight:600}
.prog .eta{margin-left:auto;font-size:13px;color:var(--muted);font-variant-numeric:tabular-nums}
.prog .track{height:16px;border-radius:99px;background:color-mix(in srgb,var(--text) 10%,transparent);overflow:hidden;margin-top:14px}
.prog .fill{height:100%;min-width:16px;border-radius:99px;background:linear-gradient(90deg,var(--accent),#ff5f8a);transition:width .8s ease;position:relative;overflow:hidden}
.prog .fill::after{content:"";position:absolute;inset:0;background:linear-gradient(90deg,transparent,rgba(255,255,255,.4),transparent);animation:shim 1.5s linear infinite}
@keyframes shim{from{transform:translateX(-100%)}to{transform:translateX(100%)}}
.steps{display:grid;grid-template-columns:repeat(7,1fr);gap:6px;margin-top:14px}
.steps span{font-size:12.5px;text-align:center;padding:7px 4px;border-radius:10px;border:1px solid var(--line);color:var(--muted);white-space:nowrap;overflow:hidden;text-overflow:ellipsis}
.steps span.done{color:var(--ok);border-color:color-mix(in srgb,var(--ok) 50%,transparent);background:color-mix(in srgb,var(--ok) 8%,transparent)}
.steps span.now{color:#fff;background:var(--accent);border-color:var(--accent);font-weight:700}
details.sb>summary{cursor:pointer;font-weight:700;font-size:15px;margin:14px 0;padding:12px 14px;border:1px solid var(--line);border-radius:12px;background:var(--panel2);list-style-position:inside}
.players{display:flex;gap:20px;flex-wrap:wrap;align-items:flex-start;margin:12px 0}
.player{display:flex;flex-direction:column;gap:8px}
.player video{display:block;background:#000}
.player.land video{width:min(620px,92vw);border-radius:12px}
.phone{padding:10px;border-radius:30px;background:#111;box-shadow:0 10px 30px rgba(0,0,0,.25)}
.phone video{height:min(520px,70vh);border-radius:20px;aspect-ratio:9/16}
.strip{display:flex;gap:8px;overflow-x:auto;padding-bottom:6px}
.strip img{height:84px;border-radius:8px;flex:none}
.scene{border:1px solid var(--line);border-radius:12px;padding:10px;margin-bottom:10px;display:grid;grid-template-columns:120px 1fr;gap:12px}
.scene img,.scene .noimg{width:120px;height:120px;object-fit:cover;border-radius:8px;background:var(--panel2)}
.scene .noimg{display:flex;align-items:center;justify-content:center;color:var(--muted);font-size:12px}
.scene textarea{min-height:54px}
.scene>*{min-width:0}
.pick{display:flex;gap:6px;overflow-x:auto;padding:4px 0 2px}
.pick .opt{flex:none;width:76px;height:54px;border-radius:8px;border:2px solid transparent;background:var(--panel2) center/cover no-repeat;
 cursor:pointer;position:relative;display:flex;align-items:center;justify-content:center;font-size:11.5px;color:var(--muted);text-align:center;line-height:1.2}
.pick .opt.on{border-color:var(--accent);box-shadow:0 0 0 2px color-mix(in srgb,var(--accent) 35%,transparent)}
.pick .opt .tag{position:absolute;left:3px;bottom:3px;font-size:10px;background:rgba(0,0,0,.65);color:#fff;border-radius:4px;padding:0 4px}
.pick .opt.up{border:2px dashed var(--line)}.pick .opt.up:hover{border-color:var(--accent);color:var(--accent)}
.empty-hero{color:var(--muted);text-align:center;padding:30px 10px}
@media (max-width:640px){.steps{grid-template-columns:repeat(4,1fr)}.prog .pct{font-size:32px}.brand{font-size:18px}.go{width:100%;margin-left:0}.scene{grid-template-columns:1fr}.scene img,.scene .noimg{width:100%;height:160px}}
</style></head><body>
<header class="topbar">
  <button class="ham" id="ham" aria-label="Menu"><i></i><i></i><i></i></button>
  <div class="brand"><span class="dot">▶</span>Video Studio</div>
  <div class="right"><span class="pill" id="status">…</span></div>
</header>
<div class="scrim" id="scrim"></div>
<aside class="drawer" id="drawer">
  <div class="head"><b>Your videos</b><button class="primary" style="margin-left:auto" id="newbtn">+ New</button></div>
  <div class="list" id="jobs"><div class="empty">No videos yet.</div></div>
  <div class="foot">
    <button id="agentsbtn">🤖 Use from Claude / agents</button>
    <a class="btn" href="/logout">↩ Sign out</a>
    <div class="small muted" id="models"></div>
  </div>
</aside>
<main>
  <section id="create" class="hero">
    <textarea id="source" rows="2" placeholder="Paste a link: a news article (नेपाली or English), GitHub repo, YouTube video or PDF. Or describe your video."></textarea>
    <div class="examples small muted" id="hint">Tip: add instructions after the link, e.g. “… focus on the rescue”.</div>
    <div class="row2">
      <div class="seg" id="len"><button data-v="15">15s</button><button data-v="30">30s</button><button data-v="60" class="on">60s</button><button data-v="90">90s</button></div>
      <div class="fmt"><button id="fl" class="on"><span class="ic" style="width:18px;height:11px"></span>16:9</button><button id="fr" class="on"><span class="ic" style="width:10px;height:16px"></span>Reel 9:16</button></div>
      <button class="primary go" id="go">Make video</button>
    </div>
    <details class="more"><summary>More options</summary>
      <div class="styles" id="styles"></div>
      <div class="opts">
        <label>Language<select id="language"></select></label>
        <label>Voice<select id="voice"></select></label>
        <label>Quality<select id="quality"><option>1080p</option><option>720p</option></select></label>
        <label>Motion<select id="motion">
          <option value="auto">Auto · 3D photos + subject pop</option><option value="parallax">3D photos only</option>
          <option value="ai">AI video clips (non-news, slow)</option><option value="none">None (still photos)</option></select></label>
        <label class="inline"><input type="checkbox" id="map" checked> Map of the place (news)</label>
        <label class="inline"><input type="checkbox" id="sfx" checked> Transition sounds</label>
        <label class="inline"><input type="checkbox" id="music" checked> Background music</label>
        <label class="inline"><input type="checkbox" id="captions" checked> Captions</label>
        <label class="inline"><input type="checkbox" id="review"> Let me edit the script first</label>
      </div>
    </details>
  </section>
  <section id="detail" style="margin-top:16px"></section>
  <section id="agents" class="card" style="margin-top:16px" hidden>
    <h2>Use from Claude Code or other agents</h2>
    <pre class="log" id="mcp"></pre>
    <p class="small muted">Tools: make_video, get_job, get_storyboard, update_storyboard, render, list_jobs, delete_job, list_options.</p>
  </section>
</main>
<script src="/static/kit.js"></script>
<script>
const SW = {auto: "linear-gradient(135deg,#7c9cff,#e1261c,#e9b872)", broadcast: "#e1261c", documentary: "#e9b872", midnight: "#7c9cff", paper: "#d2552d", neon: "#00f0ff", swiss: "#111"};
const STEPS = [["link", /reading|collecting/], ["photos", /photos|looking/], ["script", /script|review/], ["visuals", /drawing/], ["voice", /narration/], ["music", /music|mixing/], ["render", /render/]];
let cur = null, INFO = null, editing = false, style = "auto", length = 60, viewKey = "", sbOpen = false;
$("#mcp").textContent = mcpHelp("video-studio");

const open = v => document.body.classList.toggle("open", v);
$("#ham").onclick = () => open(!document.body.classList.contains("open"));
$("#scrim").onclick = () => open(false);
$("#newbtn").onclick = () => { cur = null; viewKey = ""; open(false); $("#detail").replaceChildren(); $("#source").focus(); window.scrollTo({top: 0, behavior: "smooth"}); };
$("#agentsbtn").onclick = () => { $("#agents").hidden = false; open(false); $("#agents").scrollIntoView({behavior: "smooth"}); };
$$("#len button").forEach(b => b.onclick = () => { length = +b.dataset.v; $$("#len button").forEach(x => x.classList.toggle("on", x === b)); });
["fl", "fr"].forEach(id => $("#" + id).onclick = e => e.currentTarget.classList.toggle("on"));
$("#source").addEventListener("keydown", e => { if (e.key === "Enter" && (e.ctrlKey || e.metaKey)) $("#go").click(); });

function opts() {
  return {style, length, language: $("#language").value, voice: $("#voice").value, quality: $("#quality").value,
          music: $("#music").checked, captions: $("#captions").checked, review: $("#review").checked,
          motion: $("#motion").value, map: $("#map").checked, sfx: $("#sfx").checked,
          formats: [$("#fl").classList.contains("on") && "landscape", $("#fr").classList.contains("on") && "reel"].filter(Boolean)};
}
$("#go").onclick = async () => {
  const o = opts(); if (!o.formats.length) return toast("Pick 16:9, reel or both", "bad");
  if (!$("#source").value.trim()) return toast("Paste a link or describe the video", "bad");
  $("#go").disabled = true;
  try { const j = await guard(() => api("/api/jobs", {json: {source: $("#source").value, ...o}}), "Started"); cur = j.id; editing = false; sbOpen = false; viewKey = ""; $("#source").value = ""; loadJobs(); show(); }
  finally { $("#go").disabled = false; }
};

async function loadInfo() {
  INFO = await api("/api/info");
  $("#models").textContent = [INFO.llm, INFO.image_model && INFO.image_model.split("/").pop(), INFO.music_model && INFO.music_model.split("/").pop()].filter(Boolean).join(" · ");
  $("#status").textContent = INFO.queue.running ? "⏳ " + INFO.queue.running + " rendering" + (INFO.queue.queued ? " · " + INFO.queue.queued + " waiting" : "") : "Ready";
  if (!$("#styles").children.length) {
    INFO.styles.forEach(s => $("#styles").append(el("div", {class: "sty" + (s === style ? " on" : ""), "data-s": s, onclick: e => {
      style = s; $$(".sty").forEach(x => x.classList.toggle("on", x.dataset.s === s)); }}, el("i", {style: {background: SW[s] || "#888"}}), s === "auto" ? "Auto" : s)));
    Object.entries(INFO.languages).forEach(([k, v]) => $("#language").append(el("option", {value: k}, v)));
    Object.entries(INFO.voices).forEach(([k, v]) => $("#voice").append(el("option", {value: k}, v)));
    $("#voice").append(el("option", {value: "none"}, "No narration (captions only)"));
    if (!INFO.music_model) { $("#music").checked = false; $("#music").disabled = true; }
    if (INFO.motion && !INFO.motion.ai) { const o = $("#motion option[value=ai]"); o.disabled = true; o.textContent += " · set I2V_MODEL"; }
  }
}

async function loadJobs() {
  const {jobs} = await api("/api/jobs");
  $("#jobs").replaceChildren(...jobs.map(j => el("div", {class: "item" + (j.id === cur ? " on" : ""), onclick: () => { if (cur !== j.id) { cur = j.id; editing = false; sbOpen = false; viewKey = ""; } open(false); loadJobs(); show(); }},
    j.files["landscape.jpg"] || j.files["reel.jpg"] ? el("img", {src: j.files["landscape.jpg"] || j.files["reel.jpg"], alt: ""}) : el("div", {class: "ph"}),
    el("div", {style: {minWidth: 0, flex: 1}}, el("div", {class: "t"}, j.title), el("div", {class: "m"}, badge(j.state), fmtAgo(j.created)),
      j.state === "running" ? progressBar(j.progress) : null))));
  if (!jobs.length) $("#jobs").append(el("div", {class: "empty"}, "No videos yet."));
  const j = jobs.find(x => x.id === cur);
  if (j && (["running", "queued"].includes(j.state) || keyOf(j) !== viewKey)) show();
}

function stepIndex(j) {
  return j.state === "done" ? STEPS.length : Math.max(0, STEPS.findIndex(([, re]) => re.test(j.step || "")));
}
function stepper(j) {
  return el("div", {class: "steps", id: "pp-steps"}, ...STEPS.map(([name]) => el("span", {}, name)));
}
function progressPanel(j) {
  return el("div", {class: "prog", id: "pp"},
    el("div", {class: "top"}, el("span", {class: "pct", id: "pp-pct"}, "0%"), el("span", {class: "what", id: "pp-step"}, ""),
      el("span", {class: "eta", id: "pp-eta"}, "")),
    el("div", {class: "track"}, el("div", {class: "fill", id: "pp-fill", style: {width: "0%"}})), stepper(j));
}
function updateProgress(j) {
  if (!$("#pp")) return;
  const pct = Math.round(Math.max(0, Math.min(1, j.progress || 0)) * 100), idx = stepIndex(j);
  $("#pp-pct").textContent = j.state === "queued" ? "…" : pct + "%";
  $("#pp-step").textContent = j.state === "queued" ? "Waiting for the video before it" : (j.step || "").replace(/^./, c => c.toUpperCase());
  $("#pp-fill").style.width = Math.max(2, pct) + "%";
  const s = Math.max(0, Math.round(Date.now() / 1000 - j.created));
  $("#pp-eta").textContent = "started " + Math.floor(s / 60) + ":" + String(s % 60).padStart(2, "0") + " ago";
  $$("#pp-steps span").forEach((e, i) => { e.className = i < idx ? "done" : i === idx ? "now" : ""; e.textContent = (i < idx ? "✓ " : "") + STEPS[i][0]; });
}
function keyOf(j) {
  return [j.id, j.state, Object.keys(j.files || {}).sort().join(","), (j.photos || []).length, (j.warnings || []).length].join("|");
}

let PHOTOS = [], JOBID = null;
function picker(i, cur) {
  const box = el("div", {class: "pick", "data-pick": i});
  const choose = v => { $(`[data-i="${i}"][data-k="photo"]`).value = v; editing = true; drawPick(box, i, v); };
  box.choose = choose;
  drawPick(box, i, cur);
  return box;
}
function drawPick(box, i, cur) {
  const opt = (v, label, bg, tag) => el("div", {class: "opt" + (+cur === v ? " on" : ""), title: label, onclick: () => box.choose(v),
    style: bg ? {backgroundImage: `url("${bg}")`} : {}}, bg ? null : label, tag ? el("span", {class: "tag"}, tag) : null);
  const file = el("input", {type: "file", accept: "image/*,video/*", hidden: true, onchange: e => upload(e.target.files[0], box)});
  box.replaceChildren(opt(-1, "Auto"), opt(-2, "No picture"),
    ...PHOTOS.map(p => opt(p.index, p.description || p.caption || "photo " + p.index, p.url, p.kind === "video" ? "▶ clip" : p.kind === "upload" ? "yours" : null)),
    el("div", {class: "opt up", title: "Upload a picture or a video clip (first 10 s)", onclick: () => file.click()}, "＋ Upload"), file);
  const thumb = $(`[data-thumb="${i}"]`), sel = PHOTOS.find(p => p.index === +cur);
  if (thumb && sel) thumb.replaceChildren(el("img", {src: sel.url, alt: ""}));
}
async function upload(f, box) {
  if (!f) return;
  toast("Uploading " + f.name + "…");
  try {
    const r = await fetch("/api/jobs/" + JOBID + "/assets?name=" + encodeURIComponent(f.name), {method: "POST", body: f});
    const j = await r.json(); if (!r.ok) throw new Error(j.error || r.statusText);
    PHOTOS = j.photos;
    $$("[data-pick]").forEach(b => drawPick(b, +b.dataset.pick, $(`[data-i="${b.dataset.pick}"][data-k="photo"]`).value));
    box.choose(j.index); toast(j.kind === "video" ? "Clip added to this scene" : "Picture added to this scene", "ok");
  } catch (e) { toast("Upload failed: " + e.message, "bad"); }
}
function sceneEditor(sb, images) {
  return sb.scenes.map((s, i) => {
    const img = images[String(i)];
    const f = (k, v, ph, big) => el(big ? "textarea" : "input", {"data-i": i, "data-k": k, value: v || "", placeholder: ph, style: {width: "100%"}}, big ? v || "" : null);
    return el("div", {class: "scene"}, el("div", {"data-thumb": i}, img ? el("img", {src: img, alt: ""}) : el("div", {class: "noimg"}, s.layout)),
      el("div", {class: "col"}, el("div", {class: "row small muted"}, "Scene " + (i + 1) + " · " + s.layout),
        el("input", {type: "hidden", "data-i": i, "data-k": "photo", value: s.photo ?? -1}),
        el("div", {class: "small muted"}, "Picture or clip"), picker(i, s.photo ?? -1),
        f("heading", s.heading, "On-screen headline"), f("kicker", s.kicker, "Small label (place • date)"),
        ["bullets", "photo"].includes(s.layout) ? f("bullets", (s.bullets || []).join("\n"), "Points, one per line", true) : null,
        s.layout === "code" ? f("code", s.code, "Code", true) : null,
        s.layout === "quote" ? f("quote", s.quote, "Quote", true) : null,
        f("narration", s.narration, "What the voice says", true)));
  });
}

function collect(sb) {
  const out = JSON.parse(JSON.stringify(sb));
  $$("[data-k]", $("#detail")).forEach(e => { const s = out.scenes[+e.dataset.i], k = e.dataset.k;
    s[k] = k === "bullets" ? e.value.split("\n").map(x => x.trim()).filter(Boolean) : k === "photo" ? +e.value : e.value; });
  out.title = $("#sbtitle").value; out.tagline = $("#sbtag").value;
  return out;
}

async function show() {
  if (!cur) return;
  const j = await api("/api/jobs/" + cur);
  if (j.id !== cur) return;
  const key = keyOf(j);
  if (key === viewKey || (editing && j.id === viewKey.split("|")[0])) { updateProgress(j); return; }   // update in place
  viewKey = key;
  const parts = [];
  parts.push(el("div", {class: "jobhead"}, el("h2", {}, j.title), badge(j.state),
    j.duration ? el("span", {class: "small muted"}, [fmtDur(j.duration), j.language, j.style, j.voice, j.options.quality].filter(Boolean).join(" · ")) : null,
    el("span", {style: {marginLeft: "auto"}, class: "row"},
      ["done", "error", "review"].includes(j.state) ? el("button", {onclick: async () => { await guard(() => api("/api/jobs/" + j.id + "/render", {json: {rewrite: true, review: true}}), "Writing a new script…"); show(); }}, "↻ New script") : null,
      el("button", {class: "danger", onclick: async () => { if (!confirm("Delete this video?")) return; await guard(() => api("/api/jobs/" + j.id + "/delete", {json: {}})); cur = null; $("#detail").replaceChildren(); loadJobs(); }}, "Delete"))));
  if (j.state === "queued" || j.state === "running") parts.push(progressPanel(j));
  if (j.error) parts.push(el("pre", {class: "log"}, j.error));
  (j.warnings || []).forEach(w => parts.push(el("div", {class: "small muted"}, "Note: " + w)));
  if (j.files["landscape.mp4"] || j.files["reel.mp4"]) {
    parts.push(el("div", {class: "players"},
      j.files["reel.mp4"] ? el("div", {class: "player"}, el("div", {class: "phone"}, el("video", {src: j.files["reel.mp4"], controls: true, preload: "metadata", poster: j.files["reel.jpg"] || null, playsinline: true})),
        el("a", {class: "btn primary", href: j.files["reel.mp4"] + "?download=1"}, "⬇ Reel 9:16")) : null,
      j.files["landscape.mp4"] ? el("div", {class: "player land"}, el("video", {src: j.files["landscape.mp4"], controls: true, preload: "metadata", poster: j.files["landscape.jpg"] || null}),
        el("a", {class: "btn primary", href: j.files["landscape.mp4"] + "?download=1"}, "⬇ Landscape 16:9")) : null));
  }
  if ((j.photos || []).length) parts.push(el("h3", {}, "Photos from the link"), el("div", {class: "strip"}, ...j.photos.map(p => el("img", {src: p.url, title: p.description || p.caption || "", alt: ""}))));
  PHOTOS = j.photos || []; JOBID = j.id;
  if (j.storyboard) {
    const sb = j.storyboard;
    parts.push(el("details", {class: "sb", open: j.state === "review" || sbOpen ? true : null, ontoggle: e => { sbOpen = e.currentTarget.open; }},
      el("summary", {}, j.state === "review" ? "Check the script, then render" : "✎ Edit the script"),
      el("div", {class: "row"}, el("label", {style: {flex: 1}}, "Title", el("input", {id: "sbtitle", value: sb.title})),
        el("label", {style: {flex: 2}}, "Tagline", el("input", {id: "sbtag", value: sb.tagline || ""}))),
      el("div", {style: {marginTop: "10px"}}, ...sceneEditor(sb, j.images || {})),
      el("div", {class: "row"},
        el("button", {class: "primary", disabled: ["queued", "running"].includes(j.state), onclick: async () => {
          editing = false; viewKey = ""; await guard(() => api("/api/jobs/" + j.id + "/storyboard", {json: {storyboard: collect(sb), render: true}}), "Rendering…"); show(); }},
          j.state === "review" ? "Render video" : "Save & re-render"),
        el("span", {class: "small muted"}, "Unchanged scenes keep their photos and narration."))));
  }
  const d = $("#detail");
  d.className = "card";
  d.replaceChildren(...parts);
  updateProgress(j);
  $$("input,textarea", d).forEach(e => e.addEventListener("input", () => { editing = true; }));
}

loadInfo().then(loadJobs);
every(5000, loadInfo);
every(3000, loadJobs);
</script></body></html>
''')
print("wrote", os.path.join(APP_DIR, 'vs_ui.html'))

In [ ]:
# 3. Load the helpers
import os, sys
APP_DIR = '/kaggle/working/video_app'
_missing = [f for f in ("studio_http.py", "kit.css", "kit.js") if not os.path.exists(os.path.join(APP_DIR, f))]
if _missing:
    raise RuntimeError("Run the 'Writes ... into APP_DIR' cells above first (missing: %s)" % ", ".join(_missing))
if APP_DIR not in sys.path:
    sys.path.insert(0, APP_DIR)
import studio_http as K   # Notebook helpers: kaggle_secret, spawn, wait_http, tail, start_tunnel, keep_alive

In [ ]:
# 4. Install everything (quiet; 8-12 minutes; re-running skips finished steps)
import shutil, subprocess
def pip_install(*args, keep_core=True):
    """pip install, quietly. keep_core pins Kaggle's own torch / numpy builds so nothing replaces them."""
    cons = "/tmp/keep_core.txt"
    lines = []
    if keep_core:
        import importlib.metadata as md
        for pkg in ("torch", "torchvision", "torchaudio", "numpy"):
            try:
                lines.append("%s==%s" % (pkg, md.version(pkg)))
            except md.PackageNotFoundError:
                pass
    open(cons, "w").write("\n".join(lines) + "\n")
    cmd = [sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", "-c", cons, *args]
    p = subprocess.run(cmd, capture_output=True, text=True)
    if p.returncode != 0:
        print(p.stdout[-2000:], p.stderr[-3000:])
        raise RuntimeError("pip install failed: " + " ".join(args))

def sh(cmd):
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if p.returncode != 0:
        print(p.stdout[-1500:], p.stderr[-2500:])
        raise RuntimeError("failed: " + cmd[:100])

os.makedirs(LOG_DIR, exist_ok=True)
print("System packages (ffmpeg, espeak-ng, Noto fonts incl. Devanagari)...")
sh("apt-get -qq update && DEBIAN_FRONTEND=noninteractive apt-get -qq install -y ffmpeg espeak-ng zstd "
   "fonts-noto-core fonts-noto-mono fonts-dejavu-core > /dev/null")
print("Python packages...")
pip_install("playwright", "yt-dlp", "pymupdf", "soundfile", "diffusers", "accelerate")
if NARRATION:
    try:
        # Kokoro and misaki declare Python < 3.13 but run on 3.13 (Kaggle's current Python): install their
        # dependencies from wheels, then the two packages themselves without the version check
        pip_install("--only-binary=:all:", "huggingface-hub", "loguru", "transformers", "addict", "regex",
                    "espeakng-loader", "num2words", "phonemizer-fork", "spacy", "spacy-curated-transformers")
        pip_install("--no-deps", "--ignore-requires-python", "kokoro>=0.9.4", "misaki[en]>=0.9.4")
    except RuntimeError:
        print("  Kokoro did not install (English voices are skipped); the error is above.")
# Extras: the app works without them (built-in article reader; Indic Parler or captions for Nepali)
extras = [("trafilatura", "better article extraction"), ("piper-tts", "the light Nepali voice")]
if SUBJECT_CUTOUT:
    extras.append(("timm kornia einops", "subject cut-outs"))
if SVARA_VOICES:
    extras.append(("snac", "the Svara voices"))
if ONLINE_VOICES:
    extras.append(("edge-tts", "the online voices"))
for extra, why in extras:
    try:
        pip_install(*extra.split())
    except RuntimeError:
        print("  %s did not install (%s is skipped); the error is above." % (extra, why))
print("Headless Chromium...")
sh(sys.executable + " -m playwright install --with-deps chromium > /dev/null")
if not shutil.which("ollama"):
    print("Ollama...")
    sh("curl -fsSL https://ollama.com/install.sh | sh")
K.ensure_cloudflared()

from huggingface_hub import hf_hub_download, snapshot_download
print("Nepali voices...")
os.makedirs(PIPER_DIR, exist_ok=True)
try:
    for ext in (".onnx", ".onnx.json"):
        p = hf_hub_download("rhasspy/piper-voices", "ne/ne_NP/google/medium/ne_NP-google-medium" + ext)
        shutil.copy(p, os.path.join(PIPER_DIR, "ne_NP-google-medium" + ext))
except Exception as e:
    print("  Piper Nepali voice unavailable:", e)
if INDIC_VOICES and not os.path.exists(PARLER_ENV + "/bin/python"):
    try:
        # parler-tts pins its own transformers: a virtualenv that reuses Kaggle's PyTorch keeps it apart
        sh("%s -m venv --system-site-packages %s" % (sys.executable, PARLER_ENV))
        sh("%s/bin/pip install -q --disable-pip-version-check git+https://github.com/huggingface/parler-tts.git" % PARLER_ENV)
    except RuntimeError:
        print("  Indic Parler-TTS did not install; Nepali uses the Piper voice.")
        shutil.rmtree(PARLER_ENV, ignore_errors=True)
if INDIC_VOICES and os.path.exists(PARLER_ENV + "/bin/python"):
    snapshot_download("ai4bharat/indic-parler-tts")
    snapshot_download("google/flan-t5-large", allow_patterns=["*.json", "*.model", "tokenizer*"])
if SVARA_VOICES:
    try:
        snapshot_download("kenpath/svara-tts-v1", allow_patterns=["*.json", "*.safetensors", "tokenizer*"])
        snapshot_download("hubertsiuzdak/snac_24khz")
    except Exception as e:
        print("  Svara voices unavailable:", e)
        SVARA_VOICES = False

print("Motion models...")
for flag, repo in ((DEPTH_PARALLAX, "depth-anything/Depth-Anything-V2-Small-hf"), (SUBJECT_CUTOUT, "ZhengPeng7/BiRefNet_lite")):
    if flag:
        try:
            snapshot_download(repo)
        except Exception as e:
            print("  %s unavailable: %s" % (repo, e))
if I2V_MODEL:
    print("Image-to-video model %s (large; first time only)..." % I2V_MODEL)
    snapshot_download({"ltx": "Lightricks/LTX-Video", "wan": "Wan-AI/Wan2.2-TI2V-5B-Diffusers"}.get(I2V_MODEL, I2V_MODEL),
                      allow_patterns=["model_index.json", "scheduler/*", "text_encoder/*", "tokenizer/*", "transformer/*",
                                      "vae/*", "image_encoder/*", "image_processor/*"])   # the diffusers folders only
if MUSIC_MODEL.lower().startswith("ace") and not os.path.exists(ACE_ENV + "/bin/python"):
    try:
        # ACE-Step pins old transformers and spaCy: its own virtualenv reusing Kaggle's PyTorch, packages from wheels
        sh("%s -m venv --system-site-packages %s" % (sys.executable, ACE_ENV))
        sh("%s/bin/pip install -q --disable-pip-version-check --no-deps git+https://github.com/ace-step/ACE-Step.git" % ACE_ENV)
        sh("%s/bin/pip install -q --disable-pip-version-check -c /tmp/keep_core.txt --only-binary=:all: transformers==4.50.0 "
           "'diffusers>=0.33' loguru pypinyin py3langid hangul-romanize num2words cutlet 'fugashi[unidic-lite]' "
           "pytorch_lightning soundfile librosa spacy" % ACE_ENV)
        snapshot_download("ACE-Step/ACE-Step-v1-3.5B", local_dir=ACE_CKPT)
    except Exception as e:
        print("  ACE-Step did not install (%s); using MusicGen for music." % e)
        shutil.rmtree(ACE_ENV, ignore_errors=True)
        MUSIC_MODEL = "facebook/musicgen-small"

print("Image, music and English voice models (first time only)...")
if IMAGE_MODEL:
    snapshot_download(IMAGE_MODEL, allow_patterns=["*.json", "*.txt", "*fp16.safetensors", "tokenizer*/*"])
    if "xl" in IMAGE_MODEL.lower():
        snapshot_download("madebyollin/sdxl-vae-fp16-fix", allow_patterns=["*.json", "*.safetensors"])
if MUSIC_MODEL and not MUSIC_MODEL.lower().startswith("ace"):
    snapshot_download(MUSIC_MODEL, ignore_patterns=["*.bin", "*.msgpack", "*.h5"])
if NARRATION:
    snapshot_download("hexgrad/Kokoro-82M", allow_patterns=["*.json", "*.pth", "voices/*.pt"])
gpus = K.gpu_info()
print("Install complete. GPUs:", ", ".join("%d: %s" % (g["index"], g["name"]) for g in gpus) or "none - pick 'GPU T4 x2'")

In [ ]:
# 5. Start Ollama (the scriptwriter) on GPU 0 and pull the model. Log: /kaggle/working/logs/ollama.log
import subprocess
ollama_env = dict(os.environ, OLLAMA_HOST="127.0.0.1:11434", CUDA_VISIBLE_DEVICES="0", OLLAMA_CONTEXT_LENGTH="16384")
K.spawn("ollama", ["ollama", "serve"], LOG_DIR + "/ollama.log", env=ollama_env)
if not K.wait_http("http://127.0.0.1:11434/api/tags", timeout=60, name="ollama"):
    print(K.tail(LOG_DIR + "/ollama.log"))
    raise RuntimeError("Ollama did not start; log above")
print("Pulling %s (first time ~8 GB)..." % LLM_MODEL)
p = subprocess.run(["ollama", "pull", LLM_MODEL], env=ollama_env, capture_output=True, text=True)
if p.returncode != 0:
    print(p.stderr[-1500:])
    raise RuntimeError("ollama pull failed")
print("Scriptwriter ready.")

In [ ]:
# 6. Start the studio on port 7860 (background process). Log: /kaggle/working/logs/video_studio.log
PASSWORD = K.ui_password("VIDEO_UI_PASSWORD")
n_gpu = len(K.gpu_info())
parler_py = PARLER_ENV + "/bin/python" if INDIC_VOICES and os.path.exists(PARLER_ENV + "/bin/python") else ""
env = dict(os.environ, APP_PASSWORD=PASSWORD, WORK_DIR=WORK_DIR, APP_LOG=LOG_DIR + "/video_studio.log",
           PARLER_LOG=LOG_DIR + "/parler.log", PYTHONUNBUFFERED="1", OLLAMA_URL="http://127.0.0.1:11434",
           LLM_MODEL=LLM_MODEL, IMAGE_MODEL=IMAGE_MODEL, IMAGE_STEPS=str(IMAGE_STEPS), MUSIC_MODEL=MUSIC_MODEL,
           TTS="1" if NARRATION else "0", TTS_DEVICE="cpu", RENDER_QUALITY=RENDER_QUALITY,
           PARLER_PYTHON=parler_py, PIPER_DIR=PIPER_DIR,
           SVARA_MODEL="kenpath/svara-tts-v1" if SVARA_VOICES else "", ONLINE_VOICES="1" if ONLINE_VOICES else "0",
           DEPTH_PARALLAX="1" if DEPTH_PARALLAX else "0", SUBJECT_CUTOUT="1" if SUBJECT_CUTOUT else "0",
           LOCATION_MAPS="1" if LOCATION_MAPS else "0", I2V_MODEL=I2V_MODEL,
           ACE_PYTHON=ACE_ENV + "/bin/python" if os.path.exists(ACE_ENV + "/bin/python") else "",
           ACE_LOG=LOG_DIR + "/ace_step.log", ACE_CHECKPOINTS=ACE_CKPT,
           MEDIA_DEVICE="cuda:1" if n_gpu > 1 else "cuda:0",
           LLM_KEEP_ALIVE="10m" if n_gpu > 1 else "0",      # one GPU: free the LLM's memory before drawing
           UNLOAD_AFTER="0" if n_gpu > 1 else "1", TOKENIZERS_PARALLELISM="false")
env.pop("GITHUB_TOKEN", None)
gh = K.kaggle_secret("GITHUB_TOKEN")
if gh:
    env["GITHUB_TOKEN"] = gh
K.spawn("video-studio", [sys.executable, os.path.join(APP_DIR, "vs_server.py"), "--port", str(PORT)],
        LOG_DIR + "/video_studio.log", env=env, cwd=APP_DIR)
if not K.wait_http("http://127.0.0.1:%d/health" % PORT, timeout=60, name="video-studio"):
    print(K.tail(LOG_DIR + "/video_studio.log", 40))
    raise RuntimeError("The studio did not start; log above")
for host in ("127.0.0.1", "[::1]"):
    print(host, "->", "up" if K.wait_http("http://%s:%d/health" % (host, PORT), timeout=3) else "not reachable")
print("Voices:", ", ".join(v for v in K.api_get(PORT, "/api/info", PASSWORD)["voices"]))
print("Models load on the first video (1-3 minutes extra).")

In [ ]:
# 7. Publish through your Cloudflare Tunnel (token from the VIDEO_TUNNEL_TOKEN secret, never printed)
tunnel = K.start_tunnel("VIDEO_TUNNEL_TOKEN", PORT, LOG_DIR + "/cloudflared.log")

In [ ]:
# 8. Keep-alive monitor. Leave it running; interrupting it only stops the status lines.
def queue_line():
    q = K.api_get(PORT, "/api/info", PASSWORD)["queue"]
    return "videos: %d rendering, %d waiting, %d done" % (q["running"], q["queued"], q["done"])

K.keep_alive(PORT, ["video-studio", "ollama", "cloudflared"], extra=queue_line)